# Reference-Induced Leakage in Subject-Referenced Classification of Repeated Measurements: Theory, a Pre-Training Index, and Evidence from Three Datasets
### Complete reproduction notebook

**Authors:** Syaiful Anam, Ahmad Zarkasi, Ahmad Hakiim Jamaluddin

Running all cells regenerates **every number, table and figure** of the article and of Online Resource 1 from the public
dataset of Veil et al. (2025), and ends with an automatic **consistency check** against the printed values.

| Article / Online Resource 1 | Notebook section |
|---|---|
| Table 1 (hierarchical structure, evaluable samples) | 4, 17 |
| Sect. 3.1: Cole fits, ICC and design effect (Table S1), Mahalanobis distances (Fig. 2b), drift | 6, 11, 12, 16.6 |
| Sect. 3.2: factorial design (Table 2), optimism decomposition and 2x2 interaction | 7, 9, 16.6 |
| Sect. 3.3: selected frequencies and ICC of ln\|Z\| (Fig. 3) | 5, 13, 16.3 |
| Sect. 3.4: representations and comparators (Table 3), per-patient outcomes (Table S2), patient-level BA | 7, 10, 16.6 |
| Sect. 3.5: permutation nulls (Fig. 4, Table S3), selection-aware test and Clopper-Pearson bound | 16.1, 16.4, 16.6 |
| Sect. 3.6: fully nested selection (Algorithm S1, Tables S4, S5), robustness (Table 4), size-matched referencing, P2 labellings, logistic regression (Table S6) | 8, 16.2, 16.3 |
| Sect. 3.7: simulation (Fig. 5, Tables S7-S9, Note S1) | 16.5 |
| Tables 1-4 of the article and Tables S1-S9 of Online Resource 1 (rebuilt and checked cell by cell) | 17 |
| Figures 1-5 | 18 |
| Consistency check of all values | 19 |

**Runtime** (two CPU cores, e.g. Colab): Sections 1-15 take about 15-25 minutes. The stand-alone analyses of Section 16 take
about 13 hours if recomputed, of which about 10 hours are the selection-aware permutation test. Every script checkpoints
its progress; result files already present in `outputs_ijs/revision` (e.g. copied from the `results/revision` folder of the
code repository) are loaded instead of recomputed, so a run with cached results takes about 25 minutes.

## 1. Environment

In [1]:
!pip -q install xgboost openpyxl 2>/dev/null
import platform, importlib
print("python", platform.python_version())
for m in ("numpy", "pandas", "scipy", "sklearn", "xgboost", "matplotlib"):
    print(f"{m:11s}", importlib.import_module(m).__version__)

python 3.13.16
numpy       2.5.3
pandas      3.0.5
scipy       1.18.1
sklearn     1.9.1
xgboost     3.4.1
matplotlib  3.11.2


## 2. Analysis modules
`eis_pipeline.py`: harmonization, representations, XGBoost, metrics. `eis_groups.py`: spectrum-wise and sample-wise referencing, nested and leaky selection, sample-level metrics, cluster bootstrap. `cole_analysis.py`: Cole-model fitting. `ijs_figures.py`: figures.

In [2]:
%%writefile eis_pipeline.py
"""
eis_pipeline.py
================================================================================
Distribution-Aware Patient-Referenced Electrochemical Impedance Sensing with
Absolute-Mahalanobis Fusion for Bladder Tumor Classification.

Reference implementation of the complete computational workflow described in
the manuscript (Sections "Materials and Methods").

Source dataset
--------------
Veil, Krauss, Amend, Fend & Sawodny (2025), Scientific Reports 15, 825.
doi:10.1038/s41598-024-84844-9
Repository: https://github.com/cayobro/electrical-impedance-data-bladder

Everything in this module is deterministic given SEED0.
================================================================================
"""

from __future__ import annotations

import json
import os
import pickle
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. GLOBAL CONFIGURATION  (manuscript Section "Implementation and Reproducibility")
# ==============================================================================

SEED0 = 20260921          # global reproducibility seed s_0
N_REPEATS = 30            # repeated classifier fits / reference subsamples / perturbations
N_FREQ_GRID = 50          # candidate frequencies
F_MIN, F_MAX = 1e3, 1e6   # 1 kHz - 1 MHz
TOP_M = 5                 # retained frequencies per outer fold
FUSION_W = 0.5            # equal-probability fusion
THRESHOLD = 0.5           # fixed decision threshold

REFERENCE_BUDGETS = [1, 3, 5, 10, "All"]
SIGMA_LEVELS = [0.0, 0.01, 0.025, 0.05, 0.10]

# Prespecified XGBoost parameters (manuscript Table 2)
XGB_PARAMS = dict(
    n_estimators=180,
    max_depth=3,
    learning_rate=0.04,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.0,
    reg_alpha=0.10,
    reg_lambda=2.00,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=0,
    verbosity=0,
)

# ------------------------------------------------------------------------------
# Generic decade-spaced baseline ("Generic decade-spaced five" in the paper).
# Snapped to the nearest candidate-grid frequencies (1.000, 9.541, 104.811,
# 494.171, 1000.000 kHz). The configuration published by Veil et al. (2025)
# is evaluated separately in extra_analyses.py (VEIL_HZ), interpolated exactly.
# Values in Hz.
# ------------------------------------------------------------------------------
FIXED_FIVE_HZ = [1.0e3, 1.0e4, 1.0e5, 5.0e5, 1.0e6]


def candidate_grid(n=N_FREQ_GRID, fmin=F_MIN, fmax=F_MAX):
    """Eq. (1): 50 logarithmically spaced frequencies between 1 kHz and 1 MHz."""
    return np.logspace(np.log10(fmin), np.log10(fmax), n)


# ==============================================================================
# 2. CLASSIFIER BACKEND
# ==============================================================================

def make_classifier(seed):
    """Return a fresh XGBoost classifier with the prespecified configuration."""
    from xgboost import XGBClassifier

    return XGBClassifier(random_state=int(seed), **XGB_PARAMS)


# ==============================================================================
# 3. DATA LOADING AND SPECTRAL HARMONIZATION
# ==============================================================================

def _class_label(key: str) -> int:
    """Map a dataset class key to 0 = healthy, 1 = tumor."""
    k = key.lower()
    if "health" in k or k in {"h", "normal", "benign"}:
        return 0
    return 1


def load_raw_dataset(pickle_path: str) -> dict:
    """Load the published pickle (patient-wise dictionary of measurements)."""
    with open(pickle_path, "rb") as fh:
        return pickle.load(fh)


def _measurement_arrays(measurement):
    """Extract (frequency, complex impedance) from either dict or object records."""
    if isinstance(measurement, dict):
        f = np.asarray(measurement["frequency"], dtype=float)
        z = np.asarray(measurement["z"])
    else:
        f = np.asarray(getattr(measurement, "frequency"), dtype=float)
        z = np.asarray(getattr(measurement, "z"))
    return f.ravel(), np.asarray(z, dtype=complex).ravel()


def harmonize(raw: dict, grid=None):
    """
    Spectral harmonization (manuscript Section 2.2).

    Real and negative-imaginary components are interpolated independently onto
    the common grid, linearly in the log10-frequency coordinate.

    Returns
    -------
    R    : (N, 50) real part
    NX   : (N, 50) negative imaginary part
    pid  : (N,)    patient label, e.g. 'P1'
    y    : (N,)    0 = healthy, 1 = tumor
    meta : DataFrame with one row per patient (original resolution, counts)
    """
    if grid is None:
        grid = candidate_grid()
    log_grid = np.log10(grid)

    R_rows, NX_rows, pid_rows, y_rows = [], [], [], []
    meta_rows = []

    patient_keys = sorted(raw.keys(), key=lambda s: (len(s), s))
    for p_index, pkey in enumerate(patient_keys, start=1):
        pname = f"P{p_index}"
        n_points = set()
        counts = {0: 0, 1: 0}

        for ckey, measurements in raw[pkey].items():
            # The public repository also stores a free-text "info" entry per
            # patient; only list-valued entries are measurement records.
            if not isinstance(measurements, (list, tuple)):
                continue
            label = _class_label(ckey)
            for meas in measurements:
                f, z = _measurement_arrays(meas)
                order = np.argsort(f)
                f, z = f[order], z[order]
                n_points.add(int(f.size))

                logf = np.log10(f)
                r = np.interp(log_grid, logf, np.real(z))
                nx = np.interp(log_grid, logf, -np.imag(z))

                R_rows.append(r)
                NX_rows.append(nx)
                pid_rows.append(pname)
                y_rows.append(label)
                counts[label] += 1

        meta_rows.append(
            dict(
                Patient=pname,
                SourceKey=pkey,
                Healthy=counts[0],
                Tumor=counts[1],
                Total=counts[0] + counts[1],
                FrequencyPoints=sorted(n_points)[0] if len(n_points) == 1
                else "/".join(str(v) for v in sorted(n_points)),
            )
        )

    R = np.asarray(R_rows, dtype=float)
    NX = np.asarray(NX_rows, dtype=float)
    pid = np.asarray(pid_rows)
    y = np.asarray(y_rows, dtype=int)
    meta = pd.DataFrame(meta_rows)
    return R, NX, pid, y, meta


def build_vectors(R, NX, freq_idx):
    """
    Eq. (5): x = [R(f1), -X(f1), ..., R(fm), -X(fm)] for the retained frequencies.
    """
    freq_idx = np.atleast_1d(np.asarray(freq_idx, dtype=int))
    blocks = []
    for j in freq_idx:
        blocks.append(R[:, j])
        blocks.append(NX[:, j])
    return np.column_stack(blocks)


def nearest_grid_index(grid, target_hz):
    """Index of the candidate-grid frequency closest to a target frequency."""
    return int(np.argmin(np.abs(np.log10(grid) - np.log10(target_hz))))


# ==============================================================================
# 4. PATIENT-REFERENCED REPRESENTATIONS
# ==============================================================================

def _mahalanobis(x, mu, precision):
    d = x - mu
    val = float(d @ precision @ d)
    return float(np.sqrt(max(val, 0.0)))


def build_representations(
    X,
    pid,
    y,
    patients=None,
    budget=None,
    rng=None,
):
    """
    Construct both patient-referenced representations (manuscript Section 2.3).

    For every sample the reference pool consists of healthy spectra of the SAME
    patient. For a healthy query the query itself is removed (leave-self-out).

    Parameters
    ----------
    budget : None or int
        None / "All" -> all eligible healthy references.
        int K        -> the first K references of a per-patient random ordering.
    rng : np.random.Generator
        Used only when a finite budget is requested.

    Returns
    -------
    A : (N, d)  element-wise absolute deviation, Eq. (7)
    M : (N, 1)  Ledoit-Wolf shrinkage Mahalanobis distance, Eq. (11)
    """
    n, d = X.shape
    A = np.zeros((n, d), dtype=float)
    M = np.zeros((n, 1), dtype=float)

    if patients is None:
        patients = np.unique(pid)
    if isinstance(budget, str) and budget.lower() == "all":
        budget = None

    for p in patients:
        p_mask = pid == p
        p_rows = np.flatnonzero(p_mask)
        healthy_rows = np.flatnonzero(p_mask & (y == 0))
        if healthy_rows.size == 0:
            continue

        # Patient-specific random ordering used by the reference-budget analysis
        if budget is not None:
            order = rng.permutation(healthy_rows)
        else:
            order = healthy_rows

        cache = {}
        for i in p_rows:
            pool = order[order != i] if y[i] == 0 else order
            if budget is not None:
                pool = pool[: int(budget)]

            key = pool.tobytes()
            if key not in cache:
                ref = X[pool]
                med = np.median(ref, axis=0)
                if ref.shape[0] >= 2:
                    lw = LedoitWolf(assume_centered=False).fit(ref)
                    cache[key] = (med, lw.location_, lw.precision_)
                else:
                    # Covariance is not identifiable from a single reference:
                    # the Mahalanobis branch is set to zero (manuscript Sec. 2.9)
                    cache[key] = (med, None, None)

            med, mu, prec = cache[key]
            A[i] = np.abs(X[i] - med)
            M[i, 0] = 0.0 if mu is None else _mahalanobis(X[i], mu, prec)

    return A, M


# ==============================================================================
# 5. PATIENT-CLASS SAMPLE WEIGHTS  (Eqs. 21-22)
# ==============================================================================

def patient_class_weights(pid, y):
    w = np.zeros(len(y), dtype=float)
    for p in np.unique(pid):
        for c in (0, 1):
            m = (pid == p) & (y == c)
            k = int(m.sum())
            if k:
                w[m] = 1.0 / k
    s = w.sum()
    return w * (len(y) / s) if s > 0 else np.ones(len(y))


# ==============================================================================
# 6. METRICS  (manuscript Section 2.11)
# ==============================================================================

def patient_metrics(y_true, prob, threshold=THRESHOLD):
    y_pred = (prob >= threshold).astype(int)
    out = dict(BA=balanced_accuracy_score(y_true, y_pred),
               Accuracy=accuracy_score(y_true, y_pred))

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    out["Sensitivity"] = tp / (tp + fn) if (tp + fn) else np.nan
    out["Specificity"] = tn / (tn + fp) if (tn + fp) else np.nan
    out["Precision"] = precision_score(y_true, y_pred, zero_division=0)
    out["F1"] = f1_score(y_true, y_pred, zero_division=0)
    out["MCC"] = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 or len(np.unique(y_true)) > 1 else 0.0
    out["AUC"] = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan
    return out


def fit_predict(A_tr, M_tr, y_tr, w_tr, A_te, M_te, seed):
    """Fit both branches and return (p_abs, p_mah, p_fusion) for the test rows."""
    clf_a = make_classifier(seed)
    clf_a.fit(A_tr, y_tr, sample_weight=w_tr)
    p_abs = clf_a.predict_proba(A_te)[:, 1]

    clf_m = make_classifier(seed + 1)
    clf_m.fit(M_tr, y_tr, sample_weight=w_tr)
    p_mah = clf_m.predict_proba(M_te)[:, 1]

    p_fus = FUSION_W * p_abs + (1.0 - FUSION_W) * p_mah
    return p_abs, p_mah, p_fus


# ==============================================================================
# 7. NESTED PATIENT-WISE FREQUENCY RANKING  (manuscript Section 2.4)
# ==============================================================================

def rank_frequencies_for_fold(R, NX, pid, y, train_patients, seed, verbose=False):
    """
    Rank all candidate frequencies using ONLY the outer-training patients.

    Inner validation: each training patient is held out in turn, the remaining
    four are used for fitting. The ranking score is Eq. (18).
    """
    rows = []
    n_cand = R.shape[1]
    tr_mask = np.isin(pid, train_patients)

    for j in range(n_cand):
        Xj = build_vectors(R, NX, [j])[tr_mask]
        pid_j, y_j = pid[tr_mask], y[tr_mask]

        A, M = build_representations(Xj, pid_j, y_j)

        ba_list, auc_list = [], []
        for inner_p in train_patients:
            te = pid_j == inner_p
            tr = ~te
            if len(np.unique(y_j[te])) < 2 or len(np.unique(y_j[tr])) < 2:
                continue
            w = patient_class_weights(pid_j[tr], y_j[tr])
            _, _, p_fus = fit_predict(
                A[tr], M[tr], y_j[tr], w, A[te], M[te], seed
            )
            m = patient_metrics(y_j[te], p_fus)
            ba_list.append(m["BA"])
            auc_list.append(m["AUC"])

        ba = np.asarray(ba_list, dtype=float)
        mean_ba = float(np.mean(ba))
        worst_ba = float(np.min(ba))
        sd_ba = float(np.std(ba, ddof=1)) if ba.size > 1 else 0.0
        mean_auc = float(np.nanmean(auc_list))

        score = float(np.sqrt((1 - mean_ba) ** 2 + (1 - worst_ba) ** 2 + sd_ba ** 2))
        rows.append(dict(index=j, MeanBA=mean_ba, WorstBA=worst_ba,
                         SD=sd_ba, MeanAUC=mean_auc, Score=score))
        if verbose:
            print(f"    f[{j:02d}] score={score:.4f} meanBA={mean_ba:.4f}")

    df = pd.DataFrame(rows)
    # Ties: higher mean BA, higher worst BA, lower SD, higher AUC
    df = df.sort_values(
        by=["Score", "MeanBA", "WorstBA", "SD", "MeanAUC"],
        ascending=[True, False, False, True, False],
    ).reset_index(drop=True)
    return df


def nested_frequency_selection(R, NX, pid, y, patients, seed=SEED0, verbose=True):
    """Run the ranking once per outer LOPO fold and retain the top-five."""
    selected, rankings = {}, {}
    for p in patients:
        train_patients = [q for q in patients if q != p]
        if verbose:
            print(f"  outer fold: held-out {p}")
        df = rank_frequencies_for_fold(R, NX, pid, y, train_patients, seed)
        rankings[p] = df
        selected[p] = sorted(df["index"].to_numpy()[:TOP_M].tolist())
    return selected, rankings


# ==============================================================================
# 8. PERTURBATION MODELS  (manuscript Section 2.10)
# ==============================================================================

MODE_OFFSET = {"gaussian": 0, "common": 1}   # deterministic, unlike hash()


def perturb(X, rows, sigma, mode, rng):
    """Apply a multiplicative perturbation to the selected rows of X."""
    Xp = X.copy()
    if sigma <= 0 or rows.size == 0:
        return Xp
    if mode == "gaussian":            # Eq. (30): independent, component-wise
        eps = rng.normal(0.0, sigma, size=(rows.size, X.shape[1]))
        Xp[rows] = X[rows] * (1.0 + eps)
    elif mode == "common":            # Eq. (31): one gain per spectrum
        g = rng.normal(0.0, sigma, size=(rows.size, 1))
        Xp[rows] = X[rows] * (1.0 + g)
    else:
        raise ValueError(f"unknown perturbation mode: {mode}")
    return Xp


# ==============================================================================
# 9. OUTER LOPO EVALUATION
# ==============================================================================

def evaluate(
    R,
    NX,
    pid,
    y,
    patients,
    selected,
    n_repeats=N_REPEATS,
    budget=None,
    sigma=0.0,
    mode="gaussian",
    seed0=SEED0,
    progress=None,
):
    """
    Patient-referenced leave-one-patient-out evaluation.

    Returns a long DataFrame with one row per (repeat, patient, configuration).
    Configurations: 'Absolute', 'Mahalanobis', 'Fusion'.
    """
    records = []
    train_cache = {}          # clean training-patient representations per fold

    for p in patients:
        freq_idx = selected[p] if isinstance(selected, dict) else selected
        X_clean = build_vectors(R, NX, freq_idx)
        train_patients = [q for q in patients if q != p]
        tr = np.isin(pid, train_patients)
        te = pid == p
        te_rows = np.flatnonzero(te)
        w = patient_class_weights(pid[tr], y[tr])

        # Training patients are never perturbed and always use the full
        # reference, so their representations are identical in every repeat.
        key = (p, tuple(np.atleast_1d(freq_idx).tolist()))
        if key not in train_cache:
            A_tr_all, M_tr_all = build_representations(
                X_clean, pid, y, patients=train_patients
            )
            train_cache[key] = (A_tr_all[tr], M_tr_all[tr])
        A_tr, M_tr = train_cache[key]

        for rep in range(n_repeats):
            rng = np.random.default_rng(
                seed0
                + 1000 * rep
                + 100000 * int(round(sigma * 1e4))
                + 7919 * MODE_OFFSET.get(mode, 0)
                + 13 * list(patients).index(p)
            )
            X = perturb(X_clean, te_rows, sigma, mode, rng) if sigma > 0 else X_clean

            A_te_all, M_te_all = build_representations(
                X, pid, y, patients=[p], budget=budget, rng=rng
            )

            p_abs, p_mah, p_fus = fit_predict(
                A_tr, M_tr, y[tr], w, A_te_all[te], M_te_all[te], seed=rep,
            )

            for name, prob in (("Absolute", p_abs),
                               ("Mahalanobis", p_mah),
                               ("Fusion", p_fus)):
                m = patient_metrics(y[te], prob)
                m.update(Repeat=rep, Patient=p, Configuration=name)
                records.append(m)

        if progress is not None:
            progress(list(patients).index(p) + 1, len(patients))

    return pd.DataFrame.from_records(records)


def summarize(df, configuration="Fusion"):
    """Patient-first aggregation (manuscript Section 2.11)."""
    sub = df[df["Configuration"] == configuration]
    per_patient = sub.groupby("Patient").mean(numeric_only=True).drop(columns=["Repeat"])
    per_repeat_mean_ba = sub.groupby("Repeat")["BA"].mean()

    return dict(
        per_patient=per_patient,
        MeanBA=float(per_patient["BA"].mean()),
        WorstBA=float(per_patient["BA"].min()),
        InterPatientSD=float(per_patient["BA"].std(ddof=1)),
        RepeatSD=float(per_repeat_mean_ba.std(ddof=1)),
        AUC=float(per_patient["AUC"].mean()),
        Sensitivity=float(per_patient["Sensitivity"].mean()),
        Specificity=float(per_patient["Specificity"].mean()),
        Accuracy=float(per_patient["Accuracy"].mean()),
        Precision=float(per_patient["Precision"].mean()),
        F1=float(per_patient["F1"].mean()),
        MCC=float(per_patient["MCC"].mean()),
    )


# ==============================================================================
# 10. AURC  (Eqs. 34-35)
# ==============================================================================

_TRAPZ = getattr(np, "trapezoid", None) or np.trapz  # NumPy 1.x / 2.x compatible


def aurc(sigmas, values):
    s = np.asarray(sigmas, dtype=float)
    v = np.asarray(values, dtype=float)
    return float(_TRAPZ(v, s) / (s[-1] - s[0]))


# ==============================================================================
# 11. STATISTICS  (manuscript Section 2.12)
# ==============================================================================

def wilcoxon_paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if np.allclose(a, b):
        return dict(W=0.0, p=1.0)
    res = stats.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    return dict(W=float(res.statistic), p=float(res.pvalue))


def friedman(*groups):
    res = stats.friedmanchisquare(*groups)
    return dict(chi2=float(res.statistic), p=float(res.pvalue))


def holm(pvalues, labels=None):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    m = p.size
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        running = max(running, val)
        adjusted[idx] = min(running, 1.0)
    if labels is None:
        return adjusted
    return pd.DataFrame(dict(Comparison=labels, p_raw=p, p_holm=adjusted))


# ==============================================================================
# 12. EXPORT HELPERS
# ==============================================================================

@dataclass
class Exporter:
    outdir: str = "outputs"
    tables: dict = field(default_factory=dict)

    def __post_init__(self):
        for sub in ("figures", "tables", "latex"):
            os.makedirs(os.path.join(self.outdir, sub), exist_ok=True)

    def table(self, name, df, caption="", label="", float_format="%.4f"):
        self.tables[name] = df
        df.to_csv(os.path.join(self.outdir, "tables", f"{name}.csv"), index=False)
        tex = df.to_latex(index=False, escape=True, float_format=float_format,
                          caption=caption or name, label=label or f"tab:{name}")
        with open(os.path.join(self.outdir, "latex", f"{name}.tex"), "w") as fh:
            fh.write(tex)
        return df

    def figure(self, fig, filename, dpi=400):
        path = os.path.join(self.outdir, "figures", filename)
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        return path

    def workbook(self, filename="all_tables.xlsx"):
        path = os.path.join(self.outdir, filename)
        with pd.ExcelWriter(path, engine="openpyxl") as xl:
            for name, df in self.tables.items():
                df.to_excel(xl, sheet_name=name[:31], index=False)
        return path

    def json(self, obj, filename="results_summary.json"):
        path = os.path.join(self.outdir, filename)
        with open(path, "w") as fh:
            json.dump(obj, fh, indent=2, default=float)
        return path

Writing eis_pipeline.py


In [3]:
%%writefile eis_groups.py
"""
eis_groups.py -- tissue-sample-aware patient-referenced evaluation.

The public dataset contains repeated spectra of a small number of tissue samples
per patient. Two referencing designs are implemented with the same code:

* spectrum-wise (naive):  every spectrum is its own group, so the healthy
  reference of a healthy query excludes only the query spectrum itself
  (the leave-self-out rule of the original analysis);
* sample-wise (corrected): the reference excludes ALL spectra of the query's
  tissue sample (leave-one-tissue-sample-out).

The same referencing design is used for frequency ranking, classifier training
and testing. Performance is aggregated at the tissue-sample level.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score

import eis_pipeline as ep


# ----------------------------------------------------------------------------- data
def sample_ids(raw):
    """Tissue-sample identifier for every spectrum, in ep.harmonize order."""
    out = []
    for p_index, pkey in enumerate(sorted(raw.keys(), key=lambda s: (len(s), s)), start=1):
        for ckey, meas in raw[pkey].items():
            if not isinstance(meas, (list, tuple)):
                continue
            for m in meas:
                t = m["tissue"] if isinstance(m, dict) else getattr(m, "tissue")
                out.append(f"P{p_index}:{t}")
    return np.asarray(out)


def spectrum_groups(n):
    return np.asarray([f"s{i}" for i in range(n)])


# ------------------------------------------------------------------ representations
def build_rep_g(X, pid, y, groups, patients):
    """Absolute-deviation and Ledoit-Wolf Mahalanobis representations relative to the
    same-patient healthy spectra that do NOT share the query's group.
    Returns A, M and a mask of rows that have at least one reference spectrum."""
    n, d = X.shape
    A = np.full((n, d), np.nan)
    M = np.full((n, 1), np.nan)
    for p in patients:
        rows = np.flatnonzero(pid == p)
        healthy = np.flatnonzero((pid == p) & (y == 0))
        cache = {}
        for i in rows:
            pool = healthy[groups[healthy] != groups[i]]
            if pool.size == 0:
                continue
            key = pool.tobytes()
            if key not in cache:
                ref = X[pool]
                med = np.median(ref, axis=0)
                if ref.shape[0] >= 2:
                    lw = LedoitWolf().fit(ref)
                    cache[key] = (med, lw.location_, lw.precision_)
                else:
                    cache[key] = (med, None, None)
            med, mu, prec = cache[key]
            A[i] = np.abs(X[i] - med)
            M[i, 0] = 0.0 if mu is None else float(np.sqrt(max((X[i] - mu) @ prec @ (X[i] - mu), 0.0)))
    valid = ~np.isnan(A).any(1)
    return A, M, valid


# ---------------------------------------------------------------------- weighting
def class_balanced_weights(pid, y):
    """Each class receives half of the total weight; within a class every patient
    contributes equally."""
    w = np.zeros(len(y))
    for c in (0, 1):
        pats = np.unique(pid[y == c])
        for p in pats:
            m = (pid == p) & (y == c)
            w[m] = 1.0 / (len(pats) * m.sum())
    return w * len(y) / w.sum()


def weight_fn(kind):
    return ep.patient_class_weights if kind == "patient_class" else class_balanced_weights


# --------------------------------------------------------------- frequency ranking
def rank_frequencies_g(R, NX, pid, y, groups, train_patients, seed=ep.SEED0):
    """Rank candidate frequencies with inner leave-one-patient-out over train_patients.
    Inner patients without both classes evaluable are not scored (they still train)."""
    rows = []
    for j in range(R.shape[1]):
        X = ep.build_vectors(R, NX, [j])
        A, M, ok = build_rep_g(X, pid, y, groups, train_patients)
        bas, aucs = [], []
        for q in train_patients:
            te = (pid == q) & ok
            tr = np.isin(pid, train_patients) & (pid != q) & ok
            if len(np.unique(y[te])) < 2 or len(np.unique(y[tr])) < 2:
                continue
            w = ep.patient_class_weights(pid[tr], y[tr])
            _, _, pf = ep.fit_predict(A[tr], M[tr], y[tr], w, A[te], M[te], seed)
            mm = ep.patient_metrics(y[te], pf)
            bas.append(mm["BA"]); aucs.append(mm["AUC"])
        ba = np.asarray(bas)
        mean_ba, worst = float(ba.mean()), float(ba.min())
        sd = float(ba.std(ddof=1)) if ba.size > 1 else 0.0
        rows.append(dict(index=j, MeanBA=mean_ba, WorstBA=worst, SD=sd, MeanAUC=float(np.nanmean(aucs)),
                         Score=float(np.sqrt((1 - mean_ba) ** 2 + (1 - worst) ** 2 + sd ** 2))))
    return pd.DataFrame(rows).sort_values(["Score", "MeanBA", "WorstBA", "SD", "MeanAUC"],
                                          ascending=[True, False, False, True, False]).reset_index(drop=True)


def nested_selection_g(R, NX, pid, y, groups, patients, seed=ep.SEED0):
    return {p: sorted(rank_frequencies_g(R, NX, pid, y, groups, [q for q in patients if q != p], seed)
                      ["index"].to_numpy()[:ep.TOP_M].tolist()) for p in patients}


def leaky_selection_g(R, NX, pid, y, groups, patients, seed=ep.SEED0):
    return sorted(rank_frequencies_g(R, NX, pid, y, groups, list(patients), seed)["index"].to_numpy()[:ep.TOP_M].tolist())


# ----------------------------------------------------------------------- evaluation
def evaluate_g(Xfun, pid, y, ref_groups, patients, sample_groups, weights=None,
               n_repeats=ep.N_REPEATS, weighting="patient_class"):
    """Outer leave-one-patient-out evaluation. Xfun(p) gives the feature matrix of fold p.
    weights: None (equal fusion) or dict fold -> fusion weight on the absolute branch.
    Returns one record per (repeat, evaluable test spectrum)."""
    wfun = weight_fn(weighting)
    recs = []
    for p in patients:
        X = Xfun(p)
        trp = [q for q in patients if q != p]
        A, M, ok = build_rep_g(X, pid, y, ref_groups, patients)
        tr = np.isin(pid, trp) & ok
        te = (pid == p) & ok
        if te.sum() == 0:
            continue
        w_tr = wfun(pid[tr], y[tr])
        wf = 0.5 if weights is None else weights[p]
        for rep in range(n_repeats):
            a, b, _ = ep.fit_predict(A[tr], M[tr], y[tr], w_tr, A[te], M[te], seed=rep)
            recs.append(pd.DataFrame(dict(Repeat=rep, Patient=p, Sample=sample_groups[te], y=y[te],
                                          Row=np.flatnonzero(te), p_abs=a, p_mah=b,
                                          p=wf * a + (1 - wf) * b)))
    return pd.concat(recs, ignore_index=True)


def oof_g(X, pid, y, groups, train_patients, seed=ep.SEED0):
    """Inner-LOPO out-of-fold branch probabilities for the outer-training patients."""
    A, M, ok = build_rep_g(X, pid, y, groups, train_patients)
    idx = np.flatnonzero(np.isin(pid, train_patients) & ok)
    pa, pm = np.full(len(y), np.nan), np.full(len(y), np.nan)
    for q in train_patients:
        te = idx[pid[idx] == q]
        tr = idx[pid[idx] != q]
        if te.size == 0:
            continue
        w = ep.patient_class_weights(pid[tr], y[tr])
        a, b, _ = ep.fit_predict(A[tr], M[tr], y[tr], w, A[te], M[te], seed)
        pa[te], pm[te] = a, b
    return pa, pm, idx


def inner_weights_g(Xfun, pid, y, groups, sample_groups, patients):
    """Fusion weight per outer fold chosen on inner out-of-fold predictions (sample-level BA,
    ties toward 0.5); the held-out patient is never used."""
    W = {}
    for p in patients:
        trp = [q for q in patients if q != p]
        pa, pm, idx = oof_g(Xfun(p), pid, y, groups, trp)
        best = None
        for w in np.round(np.linspace(0, 1, 11), 2):
            st = pd.DataFrame(dict(Sample=sample_groups[idx], y=y[idx], p=w * pa[idx] + (1 - w) * pm[idx])) \
                .groupby("Sample").agg(y=("y", "first"), p=("p", "mean"))
            ba = pooled(st)["BA"]
            cand = (ba, -abs(w - 0.5), w)
            best = cand if best is None or cand > best else best
        W[p] = float(best[2])
    return W


# -------------------------------------------------------------------------- metrics
def sample_table(recs, col="p", keep=None):
    """One row per tissue sample: label and probability averaged over spectra and repeats."""
    r = recs if keep is None else recs[recs.Sample.isin(keep)]
    return r.groupby(["Patient", "Sample"]).agg(y=("y", "first"), p=(col, "mean")).reset_index()


def pooled(st, thr=0.5):
    pred = (st.p >= thr).astype(int)
    tp = int(((pred == 1) & (st.y == 1)).sum()); n1 = int((st.y == 1).sum())
    tn = int(((pred == 0) & (st.y == 0)).sum()); n0 = int((st.y == 0).sum())
    sens = tp / n1 if n1 else np.nan
    spec = tn / n0 if n0 else np.nan
    auc = roc_auc_score(st.y, st.p) if st.y.nunique() == 2 else np.nan
    return dict(BA=0.5 * (sens + spec), Sens=sens, Spec=spec, TP=tp, N1=n1, TN=tn, N0=n0, AUC=auc)


def spectrum_ba(recs, col="p", keep=None):
    r = recs if keep is None else recs[recs.Sample.isin(keep)]
    vals = []
    for _, g in r.groupby("Repeat"):
        pred = (g[col] >= 0.5).astype(int)
        vals.append(0.5 * ((pred[g.y == 1] == 1).mean() + (pred[g.y == 0] == 0).mean()))
    return float(np.mean(vals))


def repeat_sd(recs, col="p", keep=None):
    r = recs if keep is None else recs[recs.Sample.isin(keep)]
    return float(np.std([pooled(sample_table(g, col))["BA"] for _, g in r.groupby("Repeat")], ddof=1))


def summary(recs, col="p", keep=None):
    st = sample_table(recs, col, keep)
    out = pooled(st)
    out.update(SpectrumBA=spectrum_ba(recs, col, keep), RepeatSD=repeat_sd(recs, col, keep))
    return out, st


def cluster_boot(tables, B=10000, seed=ep.SEED0, thr=0.5):
    """Patient-level cluster bootstrap of the pooled sample-level balanced accuracy.
    tables: dict name -> sample table (all tables contain the same samples).
    The same resampled patients are used for every table, so differences are paired.
    Resamples without any healthy or any tumor sample are discarded.
    Returns percentile 95% CIs of each BA and of all pairwise differences."""
    names = list(tables)
    pats = np.asarray(sorted(tables[names[0]].Patient.unique()))
    counts = {}
    for n in names:
        t = tables[n]
        pred = (t.p >= thr).astype(int)
        c = np.zeros((len(pats), 4))
        for k, q in enumerate(pats):
            m = (t.Patient == q).values
            c[k] = [((pred == 1) & (t.y == 1) & m).sum(), ((t.y == 1) & m).sum(),
                    ((pred == 0) & (t.y == 0) & m).sum(), ((t.y == 0) & m).sum()]
        counts[n] = c
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(pats), size=(B, len(pats)))
    draws = {}
    for n in names:
        s = counts[n][idx].sum(1)                     # B x 4
        with np.errstate(invalid="ignore", divide="ignore"):
            draws[n] = 0.5 * (s[:, 0] / s[:, 1] + s[:, 2] / s[:, 3])
    valid = np.all([np.isfinite(v) for v in draws.values()], axis=0)
    draws = {n: v[valid] for n, v in draws.items()}
    ci = {n: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for n, v in draws.items()}
    diff = {}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            d = draws[a] - draws[b]
            diff[f"{a} - {b}"] = (float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))
    return ci, diff, int(valid.sum())

Writing eis_groups.py


In [4]:
%%writefile cole_analysis.py
"""
cole_analysis.py -- physics-based (Cole-model) patient-referenced representation.

Each measured spectrum is fitted, on its ORIGINAL frequency points between 1 kHz and
1 MHz (no interpolation), with the Cole model

    Z(w) = R_inf + (R_0 - R_inf) / (1 + (j w tau)^alpha),

giving theta = [ln R_0, ln R_inf, log10 f_c, alpha] with f_c = 1 / (2 pi tau).
The four parameters replace the selected frequencies in the patient-referenced
framework (absolute deviation + Ledoit-Wolf Mahalanobis + equal fusion, LOPO,
30 repetitions). No data-driven frequency selection is involved.
"""
import json
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import least_squares

import eis_pipeline as ep

OUT = "outputs_cole"
os.makedirs(OUT, exist_ok=True)
FMIN, FMAX = 1e3, 1e6


def iterate_spectra(raw):
    """Same ordering as ep.harmonize (patients by (len, name); list-valued classes)."""
    for p_index, pkey in enumerate(sorted(raw.keys(), key=lambda s: (len(s), s)), start=1):
        for ckey, meas in raw[pkey].items():
            if not isinstance(meas, (list, tuple)):
                continue
            label = ep._class_label(ckey)
            for m in meas:
                f, z = ep._measurement_arrays(m)
                o = np.argsort(f)
                yield f"P{p_index}", label, f[o], z[o]


def cole(params, w):
    r_inf, d_r, log_tau, alpha = params
    return r_inf + d_r / (1.0 + (1j * w * 10.0 ** log_tau) ** alpha)


def fit_cole(f, z):
    m = (f >= FMIN * 0.999) & (f <= FMAX * 1.001)
    f, z = f[m], z[m]
    w = 2 * np.pi * f
    scale = np.abs(z)

    def resid(p):
        d = (cole(p, w) - z) / scale
        return np.concatenate([d.real, d.imag])

    r_inf0 = max(float(np.real(z[-1])) * 0.9, 1e-3)
    d_r0 = max(float(np.real(z[0])) - r_inf0, 1e-3)
    f_peak = f[np.argmax(-np.imag(z))]
    best = None
    for a0 in (0.6, 0.8, 0.95):
        for lt0 in (np.log10(1 / (2 * np.pi * f_peak)), -4.5, -5.5):
            r = least_squares(resid, x0=[r_inf0, d_r0, lt0, a0],
                              bounds=([1e-6, 1e-6, -9.0, 0.1], [1e4, 1e4, -1.0, 1.0]),
                              x_scale=[r_inf0, d_r0, 1.0, 0.1], max_nfev=4000)
            if best is None or r.cost < best.cost:
                best = r
    r_inf, d_r, log_tau, alpha = best.x
    rel_rmse = float(np.sqrt(np.mean(best.fun ** 2)))
    return dict(R0=r_inf + d_r, Rinf=r_inf, fc=1.0 / (2 * np.pi * 10.0 ** log_tau), alpha=alpha,
                rel_rmse=rel_rmse, n_points=int(f.size))


if __name__ == "__main__":
    raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
    GRID = ep.candidate_grid()
    R, NX, pid, y, _ = ep.harmonize(raw, GRID)
    PATIENTS = [f"P{i}" for i in range(1, 7)]

    rows = []
    for (p, lab, f, z), (pp, yy) in zip(iterate_spectra(raw), zip(pid, y)):
        assert p == pp and lab == yy
        d = fit_cole(f, z)
        d.update(Patient=p, Label=lab)
        rows.append(d)
    fits = pd.DataFrame(rows)
    fits.to_csv(f"{OUT}/cole_fits.csv", index=False)
    print("fit quality (relative RMSE, %):")
    print((100 * fits.groupby("Patient").rel_rmse.describe()[["50%", "max"]]).round(2))
    print("points used per patient:", fits.groupby("Patient").n_points.first().to_dict())

    # --- physiology: per patient medians by class --------------------------------------
    phys = fits.groupby(["Patient", "Label"])[["R0", "Rinf", "fc", "alpha"]].median().unstack("Label")
    print(phys.round(4))
    phys.to_csv(f"{OUT}/cole_medians.csv")
    sign = {}
    for prm in ("R0", "Rinf", "fc", "alpha"):
        h, t = phys[(prm, 0)], phys[(prm, 1)]
        sign[prm] = dict(tumor_lower=int((t < h).sum()), median_ratio=float(np.median(t / h)),
                         wilcoxon_p=float(stats.wilcoxon(t, h).pvalue))
    print("tumor vs healthy across patients:", json.dumps(sign, indent=1))
    pooled = fits.groupby("Label")[["R0", "Rinf", "fc", "alpha"]].median()
    print("pooled medians:\n", pooled.round(4))

    # --- classification in Cole-parameter space -------------------------------------------
    theta = np.column_stack([np.log(fits.R0), np.log(fits.Rinf), np.log10(fits.fc), fits.alpha])
    Rc, NXc = theta[:, [0, 2]], theta[:, [1, 3]]       # build_vectors -> [th0, th1, th2, th3]
    res_cole = ep.evaluate(Rc, NXc, pid, y, PATIENTS, {p: [0, 1] for p in PATIENTS})
    res_cole.to_csv(f"{OUT}/cole_lopo_long.csv", index=False)
    summ = {c: ep.summarize(res_cole, c) for c in ("Absolute", "Mahalanobis", "Fusion")}
    tab = pd.DataFrame([dict(Configuration=c, MeanBA=s["MeanBA"], AUC=s["AUC"], WorstBA=s["WorstBA"],
                             SD=s["InterPatientSD"], Sens=s["Sensitivity"], Spec=s["Specificity"])
                        for c, s in summ.items()])
    print(tab.round(4))
    pp = summ["Fusion"]["per_patient"].loc[PATIENTS, "BA"]
    print("per patient (Cole fusion):", pp.round(4).to_dict())

    sel = {k: [int(v) for v in vals] for k, vals in json.load(open("outputs_extra/selected.json")).items()}
    res_nested = ep.evaluate(R, NX, pid, y, PATIENTS, sel)
    ppn = ep.summarize(res_nested, "Fusion")["per_patient"].loc[PATIENTS, "BA"]
    d = pp.values - ppn.values
    rng = np.random.default_rng(ep.SEED0)
    bs = d[rng.integers(0, 6, size=(10000, 6))].mean(1)
    nz = int(np.sum(np.abs(d) > 1e-12))
    w = stats.wilcoxon(pp.values, ppn.values) if nz else None
    comp = dict(cole_minus_nested=float(d.mean()), ci=[float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))],
                W=None if w is None else float(w.statistic), p=None if w is None else float(w.pvalue), n_nontied=nz,
                per_patient=dict(zip(PATIENTS, np.round(d, 4).tolist())))
    print("Cole fusion minus nested-frequency fusion:", json.dumps(comp, indent=1))
    json.dump(dict(sign=sign, table=tab.to_dict(orient="records"), per_patient=pp.to_dict(), comparison=comp,
                   pooled=pooled.to_dict()), open(f"{OUT}/cole_summary.json", "w"), indent=1, default=float)

Writing cole_analysis.py


In [5]:
%%writefile ijs_figures.py
"""Figures for the revised manuscript (Iranian Journal of Science)."""
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle

matplotlib.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 12.5, "axes.titlesize": 13.5, "axes.labelsize": 12.5,
    "xtick.labelsize": 11.5, "ytick.labelsize": 11.5, "legend.fontsize": 11,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":", "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 400,
})
C_H, C_T = "#2E7DD1", "#D1442E"
C_NAIVE, C_SA = "#8A94A6", "#E07A2E"
C_ABS, C_MAH, C_FUS = "#4C8BE2", "#9B7BD4", "#3E7C59"


# ------------------------------------------------------------------- Fig. 1 schematic
def schematic():
    """Fig. 1: spectrum-wise, sample-wise, and size-matched sample-wise referencing (one patient, two healthy samples)."""
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 2.45))
    REF, REF2 = "#F2C14E", "#9AD1A8"

    def sample_box(ax, x, col, label, query=None, ref=lambda k: False, ref_col=REF, faded=False):
        w = h = 0.9
        y = 0.55
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                    fc="white", ec=col, lw=1.6, alpha=0.5 if faded else 1))
        ax.text(x + w / 2, y + h + 0.07, label, ha="center", va="bottom", fontsize=12.5, color=col)
        for k in range(9):
            cx = x + 0.15 + (k % 3) * (w - 0.30) / 2
            cy = y + 0.15 + (k // 3) * (h - 0.30) / 2
            fc, ec = (ref_col if ref(k) else "white"), col
            if query is not None and k == query:
                fc, ec = "black", "black"
            ax.add_patch(Circle((cx, cy), 0.06, fc=fc, ec=ec, lw=1.0, alpha=0.5 if faded else 1))

    specs = [("(a) Spectrum-wise (naive)", "naive",
              "Healthy query vs. all other healthy\nspectra, incl. repeats of its own sample"),
             ("(b) Sample-wise (corrected)", "sample",
              "Healthy query vs. healthy spectra\nof the other samples only"),
             ("(c) Size-matched sample-wise", "matched",
              "Tumor query vs. one healthy sample\nper version (H1 or H2), averaged")]
    for ax, (title, mode, txt) in zip(axes, specs):
        ax.set_xlim(0, 3.5); ax.set_ylim(-0.42, 1.9); ax.axis("off"); ax.set_aspect("equal")
        ax.set_title(title, fontsize=13.5, pad=1)
        if mode == "naive":
            sample_box(ax, 0.1, C_H, "H1", query=4, ref=lambda k: k != 4)
            sample_box(ax, 1.3, C_H, "H2", ref=lambda k: True)
            sample_box(ax, 2.5, C_T, "T1", faded=True)
        elif mode == "sample":
            sample_box(ax, 0.1, C_H, "H1", query=4)
            sample_box(ax, 1.3, C_H, "H2", ref=lambda k: True)
            sample_box(ax, 2.5, C_T, "T1", faded=True)
        else:
            sample_box(ax, 0.1, C_H, "H1", ref=lambda k: True, ref_col=REF2)
            sample_box(ax, 1.3, C_H, "H2", ref=lambda k: True)
            sample_box(ax, 2.5, C_T, "T1", query=4)
        ax.text(1.75, 0.18, txt, ha="center", va="top", fontsize=11.5)
    h = [plt.Line2D([], [], marker="o", ls="", ms=8, mfc="black", mec="black"),
         plt.Line2D([], [], marker="o", ls="", ms=8, mfc=REF, mec="0.3"),
         plt.Line2D([], [], marker="o", ls="", ms=8, mfc=REF2, mec="0.3")]
    fig.legend(h, ["query spectrum", "reference spectrum", "reference of the second version in (c)"],
               loc="lower center", ncol=3, fontsize=11.5, bbox_to_anchor=(0.5, -0.03), handletextpad=0.3)
    fig.tight_layout(rect=(0, 0.09, 1, 1), w_pad=0.4)
    return fig


# --------------------------------------------------- Fig. 2 repeated-spectra structure
def sample_structure(specs_p1, dist_df):
    """specs_p1: list of (label, sample, f, z) for one patient; dist_df: kind, d."""
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.7))
    fig.subplots_adjust(wspace=0.28)
    ax = axes[0]
    hs = sorted({s for l, s, *_ in specs_p1 if l == 0}); ts = sorted({s for l, s, *_ in specs_p1 if l == 1})
    cmh = plt.cm.Blues(np.linspace(0.45, 0.95, len(hs))); cmt = plt.cm.Reds(np.linspace(0.45, 0.95, len(ts)))
    col = {**{s: cmh[i] for i, s in enumerate(hs)}, **{s: cmt[i] for i, s in enumerate(ts)}}
    for lab, s, f, z in specs_p1:
        m = (f >= 1e3) & (f <= 1e6)
        ax.loglog(f[m] / 1e3, np.abs(z[m]), "-", lw=0.6, alpha=0.55, color=col[s])
    ax.set_xlabel("Frequency [kHz]", fontsize=13); ax.set_ylabel(r"$|Z|$  [$\Omega$]", fontsize=13); ax.tick_params(labelsize=12)
    ax.set_title("(a) P1: spectra by tissue sample", fontsize=13.5)
    ax.plot([], [], color=cmh[-1], label=f"healthy ({len(hs)} samples)")
    ax.plot([], [], color=cmt[-1], label=f"tumor ({len(ts)} samples)")
    ax.set_yticks([2, 3, 5, 10, 20, 30]); ax.set_yticklabels(["2", "3", "5", "10", "20", "30"])
    ax.minorticks_off()
    ax.legend(fontsize=11.5, loc="upper right")
    ax = axes[1]
    kinds = (("within", "own sample", C_H), ("between", "other healthy sample(s)", "#7FA7D9"), ("tumor", "tumor", C_T))
    pats = [p for p in ("P1", "P4", "P5") if (dist_df.Patient == p).any()]
    pos, data, cols = [], [], []
    for i, p in enumerate(pats):
        for k, (kind, _, c) in enumerate(kinds):
            d = dist_df[(dist_df.Patient == p) & (dist_df.kind == kind)].d.values
            pos.append(i * 4 + k); data.append(d); cols.append(c)
    bp = ax.boxplot(data, positions=pos, widths=0.7, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", lw=1.2))
    for patch, c in zip(bp["boxes"], cols):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    for x_, d, c in zip(pos, data, cols):
        jit = np.random.default_rng(int(x_)).uniform(-0.2, 0.2, d.size)
        ax.plot(x_ + jit, d, "o", ms=1.8, alpha=0.35, color=c)
    ax.set_yscale("log")
    ax.set_xticks([i * 4 + 1 for i in range(len(pats))])
    ax.set_xticklabels([f"{p}" for p in pats])
    ax.set_ylabel("Mahalanobis distance", fontsize=13); ax.tick_params(labelsize=12)
    for kind, lab, c in kinds:
        ax.plot([], [], "s", color=c, alpha=0.6, label=("healthy vs. " + lab) if kind != "tumor" else "tumor vs. healthy samples")
    ax.set_ylim(top=ax.get_ylim()[1] * 30)
    ax.legend(fontsize=11.5, loc="upper left", ncol=1)
    ax.set_title("(b) Distance to healthy reference", fontsize=13.5)
    fig.tight_layout()
    return fig


# --------------------------------------------------------- Fig. 3 factorial design
def factorial(vals, cis, counts):
    """vals/cis/counts keyed by (referencing, selection)."""
    fig, ax = plt.subplots(figsize=(5.8, 4.0))
    xs = {"leaky": 0, "nested": 1}
    for refk, lab, c, off in (("naive", "Spectrum-wise referencing (naive)", C_NAIVE, -0.04),
                              ("sa", "Sample-wise referencing (corrected)", C_SA, 0.04)):
        xx = [xs["leaky"] + off, xs["nested"] + off]
        yy = [vals[(refk, "leaky")], vals[(refk, "nested")]]
        lo = [vals[(refk, s)] - cis[(refk, s)][0] for s in ("leaky", "nested")]
        hi = [cis[(refk, s)][1] - vals[(refk, s)] for s in ("leaky", "nested")]
        ax.errorbar(xx, yy, yerr=[lo, hi], fmt="-o", color=c, ms=7, lw=1.8, capsize=4, label=lab)
        for x_, s in zip(xx, ("leaky", "nested")):
            ax.annotate(f"{vals[(refk, s)]:.3f}  ({counts[(refk, s)]})", (x_, vals[(refk, s)]),
                        textcoords="offset points", xytext=(0, 9) if refk == "naive" else (0, -16),
                        ha="center", fontsize=9, color=c,
                        bbox=dict(fc="white", ec="none", alpha=0.85, pad=0.5))
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Leaky selection\n(all six patients)", "Nested selection\n(training patients only)"])
    ax.set_xlim(-0.45, 1.45); ax.set_ylim(0.4, 1.08)
    ax.set_ylabel("Sample-level balanced accuracy")
    ax.axhline(0.5, color="0.5", lw=0.8, ls="--")
    ax.legend(fontsize=10, loc="lower left")
    fig.tight_layout()
    return fig


# ------------------------------------------------- Fig. 4 representation reversal
def representations(vals, cis):
    """vals[(features, referencing, branch)]"""
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8), sharey=True)
    branches = (("abs", "Absolute", C_ABS), ("mah", "Mahalanobis", C_MAH), ("fusion", "Equal fusion", C_FUS))
    for ax, (feat, title) in zip(axes, (("freq", "(a) Nested frequency selection"), ("cole", "(b) Cole parameters"))):
        for k, (b, lab, c) in enumerate(branches):
            for j, (refk, mk) in enumerate((("naive", "o"), ("sa", "s"))):
                x = j + (k - 1) * 0.18
                v = vals[(feat, refk, b)]
                lo, hi = cis[(feat, refk, b)]
                ax.errorbar([x], [v], yerr=[[v - lo], [hi - v]], fmt=mk, color=c, ms=7, capsize=3,
                            label=lab if j == 0 else None)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Spectrum-wise\n(naive)", "Sample-wise\n(corrected)"])
        ax.set_xlim(-0.5, 1.5); ax.set_ylim(0, 1.05)
        ax.axhline(0.5, color="0.5", lw=0.8, ls="--")
        ax.set_title(title)
    axes[0].set_ylabel("Sample-level balanced accuracy")
    axes[0].legend(fontsize=10, loc="lower left")
    fig.tight_layout()
    return fig


# --------------------------------------------------------- Fig. 5 selected frequencies
def frequencies(rows):
    """rows: list of (label, list_of_kHz, colour, group)."""
    fig, ax = plt.subplots(figsize=(8.6, 4.6))
    yt, yl = [], []
    yv = 0
    last = None
    for lab, fk, c, grp in rows:
        if last is not None and grp != last:
            yv += 0.6
        ax.plot(fk, [yv] * len(fk), "o", color=c, ms=6)
        yt.append(yv); yl.append(lab)
        yv += 1
        last = grp
    ax.set_xscale("log"); ax.set_xlim(0.8, 1300)
    ax.set_yticks(yt); ax.set_yticklabels(yl, fontsize=10.5); ax.invert_yaxis()
    ax.set_xlabel("Frequency [kHz]")
    ax.axvspan(10, 34, color="0.9", lw=0, zorder=0)
    ax.axvspan(100, 1000, color="#FBE3CF", lw=0, zorder=0)
    fig.tight_layout()
    return fig


# --------------------------------------------------------------- Fig. S1 drift
def drift(fits):
    fig, ax = plt.subplots(figsize=(6.0, 3.8))
    for s_, g in fits.groupby("Sample", sort=False):
        if len(g) < 6:
            continue
        x = np.linspace(0, 1, len(g))
        ax.plot(x, np.log(g.R0.values) - np.log(g.R0.values[0]), lw=0.8, alpha=0.7,
                color=C_H if g.Label.iloc[0] == 0 else C_T)
    ax.plot([], [], color=C_H, label="healthy samples"); ax.plot([], [], color=C_T, label="tumor samples")
    ax.set_xlabel("Normalized acquisition order within tissue sample")
    ax.set_ylabel(r"$\ln R_0 - \ln R_0^{\mathrm{first}}$")
    ax.legend(fontsize=10)
    fig.tight_layout()
    return fig


def frequencies_icc(rows, f_khz, icc):
    """(a) between-sample ICC of ln|Z| among healthy spectra; (b) selected frequency sets (side by side)."""
    fig, (ax0, ax) = plt.subplots(1, 2, figsize=(9.6, 4.3), gridspec_kw=dict(width_ratios=[1, 1.45], wspace=0.08))
    ax0.semilogx(f_khz, icc, "-", color="0.2", lw=2.2)
    ax0.set_ylabel("ICC of ln|Z| (healthy spectra)", fontsize=13)
    ax0.set_ylim(0.5, 1.0)
    ax0.set_title("(a) Between-sample share of variance", fontsize=13.5)
    for a in (ax0, ax):
        a.axvspan(10, 34, color="0.88", lw=0, zorder=0)
        a.axvspan(100, 1000, color="#FBE3CF", lw=0, zorder=0)
        a.set_xscale("log"); a.set_xlim(0.8, 1300)
        a.set_xlabel("Frequency [kHz]", fontsize=13)
        a.tick_params(labelsize=12)
    yt, yl = [], []
    yv, last = 0, None
    for lab, fk, c, grp in rows:
        if last is not None and grp != last:
            yv += 0.6
        ax.plot(fk, [yv] * len(fk), "o", color=c, ms=7)
        yt.append(yv); yl.append(lab)
        yv += 1
        last = grp
    ax.set_yticks(yt); ax.set_yticklabels(yl, fontsize=12); ax.invert_yaxis()
    ax.yaxis.tick_right()
    ax.set_title("(b) Frequencies retained", fontsize=13.5)
    return fig


## 3. Data

In [6]:
import os
if not os.path.isdir("electrical-impedance-data-bladder"):
    os.system("git clone -q https://github.com/cayobro/electrical-impedance-data-bladder.git")
print(sorted(os.listdir("electrical-impedance-data-bladder")))

['.git', '.gitignore', 'README.md', 'class_reduced_measurement.py', 'dataset.pickle', 'dataset_without_modules.pickle', 'load_data.py']


In [7]:
import json
import os
import time

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import least_squares
from sklearn.covariance import LedoitWolf

import importlib
import eis_pipeline as ep
import eis_groups as eg
from cole_analysis import iterate_spectra, fit_cole, cole, FMIN, FMAX

OUT = "outputs_ijs"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()


def tick(m):
    print(f"[{time.time() - T0:6.0f}s] {m}", flush=True)


raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
GRID = ep.candidate_grid()
R, NX, pid, y, meta = ep.harmonize(raw, GRID)
P = [f"P{i}" for i in range(1, 7)]
G = eg.sample_ids(raw)                 # tissue samples
GS = eg.spectrum_groups(len(y))        # every spectrum its own group
khz = lambda idx: [round(GRID[j] / 1e3, 3) for j in idx]
RES = {}

## 4. Hierarchical structure (Table 1) and evaluable samples

In [8]:
df = pd.DataFrame(dict(Patient=pid, y=y, G=G))
t1 = []
for p in P:
    d = df[df.Patient == p]
    t1.append(dict(Patient=p,
                   HealthySamples=int(d[d.y == 0].G.nunique()), HealthySpectra=int((d.y == 0).sum()),
                   TumorSamples=int(d[d.y == 1].G.nunique()), TumorSpectra=int((d.y == 1).sum()),
                   FrequencyPoints=meta.set_index("Patient").loc[p, "FrequencyPoints"]))
T1 = pd.DataFrame(t1)
T1.to_csv(f"{OUT}/Table1_structure.csv", index=False)
print(T1)
spp = df.groupby("G").size()
RES["structure"] = dict(n_spectra=int(len(y)), n_samples=int(df.G.nunique()),
                        n_healthy_samples=int(df[df.y == 0].G.nunique()),
                        n_tumor_samples=int(df[df.y == 1].G.nunique()),
                        spectra_per_sample_median=float(spp.median()), spectra_per_sample_min=int(spp.min()),
                        spectra_per_sample_max=int(spp.max()))

_, _, ok_any = eg.build_rep_g(R[:, :1], pid, y, G, P)
EVAL = sorted(set(G[ok_any]))
ev = df[df.G.isin(EVAL)].drop_duplicates("G")
RES["evaluable"] = dict(n=len(EVAL), healthy=int((ev.y == 0).sum()), tumor=int((ev.y == 1).sum()),
                        healthy_patients=sorted(ev[ev.y == 0].Patient.unique().tolist()))
print("evaluable samples:", RES["evaluable"])

  Patient  HealthySamples  ...  TumorSpectra  FrequencyPoints
0      P1               6  ...           110              100
1      P2               1  ...            90              100
2      P3               1  ...           180               50
3      P4               2  ...            45               50
4      P5               2  ...            20               50
5      P6               1  ...            14               20

[6 rows x 6 columns]
evaluable samples: {'n': 26, 'healthy': 10, 'tumor': 16, 'healthy_patients': ['P1', 'P4', 'P5']}


## 5. Frequency selection under both referencing designs (Fig. 3b, Sect. 3.3; about 5-8 minutes)

In [9]:
SEL_CACHE = f"{OUT}/selections_idx.json"
if os.path.exists(SEL_CACHE):
    SEL = json.load(open(SEL_CACHE))
    print("selections loaded from cache")
else:
    tick("nested ranking, spectrum-wise referencing ...")
    SEL = {}
    SEL["naive_nested"] = eg.nested_selection_g(R, NX, pid, y, GS, P)
    tick("leaky ranking, spectrum-wise referencing ...")
    SEL["naive_leaky"] = eg.leaky_selection_g(R, NX, pid, y, GS, P)
    tick("nested ranking, sample-wise referencing ...")
    SEL["sa_nested"] = eg.nested_selection_g(R, NX, pid, y, G, P)
    tick("leaky ranking, sample-wise referencing ...")
    SEL["sa_leaky"] = eg.leaky_selection_g(R, NX, pid, y, G, P)
    tick("nested ranking without P2 (sensitivity) ...")
    SEL["sa_nested_noP2"] = eg.nested_selection_g(R, NX, pid, y, G, [p for p in P if p != "P2"])
    SEL = json.loads(json.dumps(SEL, default=int))
    json.dump(SEL, open(SEL_CACHE, "w"), indent=1)
RES["selected_khz"] = {k: ({p: khz(v) for p, v in s.items()} if isinstance(s, dict) else khz(s)) for k, s in SEL.items()}
CACHE_FITS = f"{OUT}/cole_fits.csv"
print(json.dumps(RES["selected_khz"], indent=1))

VEIL_HZ = np.array([3.55e3, 568.99e3, 655.13e3, 754.31e3, 868.51e3])
Rv, NXv, *_ = ep.harmonize(raw, VEIL_HZ)
GEN = sorted({ep.nearest_grid_index(GRID, f) for f in ep.FIXED_FIVE_HZ})

Xsel = lambda s: (lambda p: ep.build_vectors(R, NX, s[p] if isinstance(s, dict) else s))

selections loaded from cache
{
 "naive_nested": {
  "P1": [
   2.024,
   2.683,
   14.563,
   16.768,
   22.23
  ],
  "P2": [
   10.985,
   12.649,
   14.563,
   16.768,
   19.307
  ],
  "P3": [
   14.563,
   19.307,
   25.595,
   29.471,
   33.932
  ],
  "P4": [
   10.985,
   12.649,
   14.563,
   19.307,
   25.595
  ],
  "P5": [
   10.985,
   12.649,
   14.563,
   19.307,
   29.471
  ],
  "P6": [
   120.679,
   138.95,
   159.986,
   184.207,
   212.095
  ]
 },
 "naive_leaky": [
  10.985,
  12.649,
  14.563,
  16.768,
  25.595
 ],
 "sa_nested": {
  "P1": [
   138.95,
   184.207,
   212.095,
   494.171,
   868.511
  ],
  "P2": [
   159.986,
   184.207,
   372.759,
   429.193,
   1000.0
  ],
  "P3": [
   159.986,
   429.193,
   568.987,
   655.129,
   1000.0
  ],
  "P4": [
   429.193,
   494.171,
   568.987,
   655.129,
   754.312
  ],
  "P5": [
   8.286,
   12.649,
   14.563,
   16.768,
   19.307
  ],
  "P6": [
   138.95,
   184.207,
   212.095,
   372.759,
   568.987
  ]
 },
 "sa_lea

## 6. Cole-model fits and identifiability

In [10]:
tick("fitting the Cole model ...")
specs = list(iterate_spectra(raw))
rows = []
for (p, lab, f, z), p2, y2 in zip(specs, pid, y):
    assert p == p2 and lab == y2
    d = fit_cole(f, z)
    d.update(Patient=p, Label=lab)
    rows.append(d)
fits = pd.DataFrame(rows)
fits["Sample"] = G
fits.to_csv(f"{OUT}/cole_fits.csv", index=False)
THETA = np.column_stack([np.log(fits.R0), np.log(fits.Rinf), np.log10(fits.fc), fits.alpha])
q = 100 * fits.rel_rmse.quantile([0.5, 0.95, 1.0]).values
RES["cole_fit"] = dict(rmse_median=q[0], rmse_p95=q[1], rmse_max=q[2])

# identifiability: f_c outside the fitted window and Jacobian standard errors
se = []
for r, (_, _, f, z) in zip(fits.itertuples(), specs):
    m = (f >= FMIN * .999) & (f <= FMAX * 1.001)
    f, z = f[m], z[m]
    w, sc = 2 * np.pi * f, np.abs(z)
    res = lambda prm: np.concatenate([((cole(prm, w) - z) / sc).real, ((cole(prm, w) - z) / sc).imag])
    o = least_squares(res, x0=[r.Rinf, r.R0 - r.Rinf, np.log10(1 / (2 * np.pi * r.fc)), r.alpha],
                      bounds=([1e-6, 1e-6, -9, .1], [1e4, 1e4, -1, 1]), max_nfev=200)
    cov = np.linalg.pinv(o.jac.T @ o.jac) * (2 * o.cost / max(o.jac.shape[0] - 4, 1))
    se.append(np.sqrt(np.clip(np.diag(cov), 0, None)))
se = np.asarray(se)
outside = ((fits.fc < 1e3) | (fits.fc > 1e6)).values
RES["cole_ident"] = dict(outside_pct=float(100 * outside.mean()), outside_n=int(outside.sum()),
                         outside_by={f"{a}-{'H' if b == 0 else 'T'}": int(v) for (a, b), v in fits[outside].groupby(["Patient", "Label"]).size().items()},
                         se_log10fc_median=float(np.median(se[:, 2])), se_alpha_median=float(np.median(se[:, 3])))
print("Cole identifiability:", RES["cole_ident"])

[     0s] fitting the Cole model ...
Cole identifiability: {'outside_pct': 7.6923076923076925, 'outside_n': 59, 'outside_by': {'P1-H': 16, 'P4-H': 40, 'P6-H': 3}, 'se_log10fc_median': 0.015722540108809722, 'se_alpha_median': 0.007978790752706686}


## 7. Factorial design, comparators and representations (Tables 2 and 3, Sects. 3.2-3.4)

In [11]:
CONF = {
    # name: (feature function, referencing groups, weights, weighting, patients)
    "naive_nested": (Xsel(SEL["naive_nested"]), GS),
    "naive_leaky": (Xsel(SEL["naive_leaky"]), GS),
    "sa_nested": (Xsel(SEL["sa_nested"]), G),
    "sa_leaky": (Xsel(SEL["sa_leaky"]), G),
    "sa_published": (lambda p: ep.build_vectors(Rv, NXv, [0, 1, 2, 3, 4]), G),
    "sa_generic": (Xsel(GEN), G),
    "naive_cole": (lambda p: THETA, GS),
    "sa_cole": (lambda p: THETA, G),
}
RECS, SUM, TAB = {}, {}, {}
for name, (fx, grp) in CONF.items():
    tick(f"evaluate {name} ...")
    RECS[name] = eg.evaluate_g(fx, pid, y, grp, P, G)
    for col, tag in (("p", "fusion"), ("p_abs", "abs"), ("p_mah", "mah")):
        s, st = eg.summary(RECS[name], col, keep=EVAL)
        SUM[f"{name}|{tag}"], TAB[f"{name}|{tag}"] = s, st

tick("inner-fold fusion weights (sample-wise) ...")
W = {}
for name in ("sa_nested", "sa_cole"):
    fx, grp = CONF[name]
    W[name] = eg.inner_weights_g(fx, pid, y, grp, G, P)
    r = eg.evaluate_g(fx, pid, y, grp, P, G, weights=W[name])
    SUM[f"{name}|innerw"], TAB[f"{name}|innerw"] = eg.summary(r, "p", keep=EVAL)
RES["inner_weights"] = W

# bridge to the originally reported metric (spectrum-wise, all 767 spectra, mean patient BA)
bridge = []
for _, g in RECS["naive_nested"].groupby("Repeat"):
    bridge.append(np.mean([ep.patient_metrics(h.y.values, h.p.values)["BA"] for _, h in g.groupby("Patient")]))
s_all, _ = eg.summary(RECS["naive_nested"], "p", keep=None)
RES["bridge"] = dict(original_metric=float(np.mean(bridge)), naive_all29=s_all)

[    36s] evaluate naive_nested ...
[    52s] evaluate naive_leaky ...
[    67s] evaluate sa_nested ...
[    81s] evaluate sa_leaky ...
[    95s] evaluate sa_published ...
[   109s] evaluate sa_generic ...
[   123s] evaluate naive_cole ...
[   139s] evaluate sa_cole ...
[   153s] inner-fold fusion weights (sample-wise) ...


## 8. Sensitivity analyses (Table 4: class-balanced weighting, P2 excluded, ln R0 and ln Rinf only)

In [12]:
tick("sensitivity: class-balanced weighting ...")
for name in ("sa_nested", "sa_cole"):
    fx, grp = CONF[name]
    r = eg.evaluate_g(fx, pid, y, grp, P, G, weighting="class_balanced")
    for col, tag in (("p", "fusion"), ("p_abs", "abs")):
        SUM[f"{name}|{tag}|cb"], TAB[f"{name}|{tag}|cb"] = eg.summary(r, col, keep=EVAL)

tick("sensitivity: Cole resistances only ...")
TH2 = THETA[:, :2]
r = eg.evaluate_g(lambda p: TH2, pid, y, G, P, G)
for col, tag in (("p", "fusion"), ("p_abs", "abs"), ("p_mah", "mah")):
    SUM[f"sa_coleR|{tag}"], TAB[f"sa_coleR|{tag}"] = eg.summary(r, col, keep=EVAL)

tick("sensitivity: without P2 ...")
P5 = [p for p in P if p != "P2"]
keep2 = np.asarray([p != "P2" for p in pid])
EVAL_noP2 = [s for s in EVAL if not s.startswith("P2:")]
sel5 = SEL["sa_nested_noP2"]
RES["selected_khz"]["sa_nested_noP2"] = {p: khz(v) for p, v in sel5.items()}
for name, fx, grp in (("noP2_sa_nested", Xsel(sel5), G), ("noP2_sa_cole", lambda p: THETA, G),
                      ("noP2_naive_nested", Xsel({p: SEL["naive_nested"][p] for p in P5}), GS)):
    r = eg.evaluate_g(fx, pid, y, grp, P5, G)
    for col, tag in (("p", "fusion"), ("p_abs", "abs")):
        SUM[f"{name}|{tag}"], TAB[f"{name}|{tag}"] = eg.summary(r, col, keep=EVAL_noP2)

[   186s] sensitivity: class-balanced weighting ...
[   213s] sensitivity: Cole resistances only ...
[   227s] sensitivity: without P2 ...


## 9. Patient-level cluster bootstrap (all confidence intervals and paired differences)

In [13]:
tick("patient-level cluster bootstrap ...")
main = ["naive_leaky|fusion", "naive_nested|fusion", "sa_leaky|fusion", "sa_nested|fusion",
        "sa_published|fusion", "sa_generic|fusion",
        "naive_nested|abs", "naive_nested|mah", "sa_nested|abs", "sa_nested|mah",
        "naive_cole|fusion", "naive_cole|abs", "naive_cole|mah",
        "sa_cole|fusion", "sa_cole|abs", "sa_cole|mah", "sa_cole|innerw", "sa_nested|innerw"]
CI, DIFF, NB = eg.cluster_boot({k: TAB[k] for k in main})
RES["boot_valid"] = NB

[   262s] patient-level cluster bootstrap ...


## 10. Per-patient sample outcomes (Table S2)

In [14]:
pp = []
for k in ("naive_nested|fusion", "sa_nested|fusion", "sa_published|fusion", "sa_cole|abs", "sa_cole|innerw"):
    st = TAB[k]
    pred = (st.p >= .5).astype(int)
    for p in P:
        m = st.Patient == p
        pp.append(dict(Config=k, Patient=p,
                       TumorCorrect=int(((pred == 1) & (st.y == 1) & m).sum()), Tumor=int(((st.y == 1) & m).sum()),
                       HealthyCorrect=int(((pred == 0) & (st.y == 0) & m).sum()), Healthy=int(((st.y == 0) & m).sum())))
PP = pd.DataFrame(pp)
PP.to_csv(f"{OUT}/per_patient_samples.csv", index=False)

## 11. Variance decomposition and Mahalanobis distances (Table S1, Fig. 2b, Sect. 3.1)

In [15]:
tick("within- versus between-sample structure ...")
dist = []
for p in P:
    h = np.flatnonzero((pid == p) & (y == 0))
    samples = sorted(set(G[h]))
    for i in np.flatnonzero(pid == p):
        own = h[(G[h] == G[i]) & (h != i)]
        other = h[G[h] != G[i]]
        if y[i] == 0 and len(samples) >= 2 and own.size >= 2 and other.size >= 2:
            for kind, pool in (("within", own), ("between", other)):
                lw = LedoitWolf().fit(THETA[pool])
                dd = THETA[i] - lw.location_
                dist.append(dict(Patient=p, kind=kind, d=float(np.sqrt(dd @ lw.precision_ @ dd))))
        if y[i] == 1 and h.size >= 2:
            lw = LedoitWolf().fit(THETA[h])
            dd = THETA[i] - lw.location_
            dist.append(dict(Patient=p, kind="tumor", d=float(np.sqrt(dd @ lw.precision_ @ dd))))
DIST = pd.DataFrame(dist)
DIST.to_csv(f"{OUT}/mahalanobis_within_between.csv", index=False)
dmed = DIST.groupby("kind").d.median()
RES["distances"] = dict(within=float(dmed["within"]), between=float(dmed["between"]), tumor=float(dmed["tumor"]),
                        patients_with_between=sorted(DIST[DIST.kind == "between"].Patient.unique().tolist()))
print("median Mahalanobis distance:", RES["distances"])


def icc1(values, groups):
    """One-way random-effects ICC(1) with unequal group sizes."""
    dfv = pd.DataFrame(dict(v=values, g=groups))
    k = dfv.g.nunique(); N = len(dfv)
    if k < 2:
        return np.nan
    gm = dfv.groupby("g").v.agg(["mean", "size"])
    grand = dfv.v.mean()
    msb = (gm["size"] * (gm["mean"] - grand) ** 2).sum() / (k - 1)
    msw = ((dfv.v - dfv.g.map(gm["mean"])) ** 2).sum() / (N - k)
    n0 = (N - (gm["size"] ** 2).sum() / N) / (k - 1)
    return float((msb - msw) / (msb + (n0 - 1) * msw))


icc = {}
names = ["ln R0", "ln Rinf", "log10 fc", "alpha"]
for c, lab in ((0, "healthy"), (1, "tumor")):
    for j, nm in enumerate(names):
        vals, grps = [], []
        for p in P:
            m = (pid == p) & (y == c)
            if len(set(G[m])) >= 2:
                vals.extend(THETA[m, j] - THETA[m, j].mean())
                grps.extend(G[m])
        icc[f"{lab}|{nm}"] = icc1(np.asarray(vals), np.asarray(grps))
m_bar = float(spp.mean())
RES["icc"] = icc
RES["design_effect"] = dict(m_bar=m_bar,
                            deff_healthy_lnR0=1 + (m_bar - 1) * icc["healthy|ln R0"],
                            n_eff=float(len(y) / (1 + (m_bar - 1) * icc["healthy|ln R0"])))
print("ICC:", {k: round(v, 3) for k, v in icc.items()})

[   262s] within- versus between-sample structure ...
median Mahalanobis distance: {'within': 1.2622278708856391, 'between': 4.893892997942829, 'tumor': 31.934354819849283, 'patients_with_between': ['P1', 'P4', 'P5']}
ICC: {'healthy|ln R0': 0.975, 'healthy|ln Rinf': 0.573, 'healthy|log10 fc': 0.967, 'healthy|alpha': 0.905, 'tumor|ln R0': 0.992, 'tumor|ln Rinf': 0.901, 'tumor|log10 fc': 0.984, 'tumor|alpha': 0.971}


## 12. Acquisition-order drift (Sect. 3.1)

In [16]:
rh = []
for s_, g in fits.groupby("Sample", sort=False):
    if len(g) >= 6:
        idx = np.arange(len(g))
        k = max(1, len(g) // 5)
        rh.append(dict(Sample=s_, n=len(g), rho=stats.spearmanr(idx, np.log(g.R0)).statistic,
                       drift=float(np.log(g.R0.iloc[-k:]).median() - np.log(g.R0.iloc[:k]).median())))
RH = pd.DataFrame(rh)
RH.to_csv(f"{OUT}/drift_per_sample.csv", index=False)
medR0 = fits.groupby(["Patient", "Label"]).R0.median().unstack()
RES["drift"] = dict(n_samples=int(len(RH)), median_abs_rho=float(RH.rho.abs().median()),
                    frac_abs_rho_gt_08=float((RH.rho.abs() > 0.8).mean()),
                    median_abs_drift_lnR0=float(RH.drift.abs().median()),
                    median_abs_contrast_lnR0=float(np.abs(np.log(medR0[1] / medR0[0])).median()))
print("drift:", RES["drift"])

drift: {'n_samples': 29, 'median_abs_rho': 0.9607142857142855, 'frac_abs_rho_gt_08': 0.6206896551724138, 'median_abs_drift_lnR0': 0.0570718612064649, 'median_abs_contrast_lnR0': 0.9553747036317588}


## 13. ICC of ln|Z| by frequency (Fig. 3a)

In [17]:
L = np.log(np.hypot(R, NX))
icc_f = []
for j, fr in enumerate(GRID):
    vals, grps = [], []
    for p in ("P1", "P4", "P5"):
        h = (pid == p) & (y == 0)
        vals += list(L[h, j] - L[h, j].mean()); grps += list(G[h])
    icc_f.append(dict(f_kHz=fr / 1e3, ICC=icc1(np.asarray(vals), np.asarray(grps))))
ICCF = pd.DataFrame(icc_f)
ICCF.to_csv(f"{OUT}/frequency_icc.csv", index=False)
RES["icc_freq"] = dict(first=float(ICCF.ICC.iloc[0]), last=float(ICCF.ICC.iloc[-1]),
                       monotone=bool(np.all(np.diff(ICCF.ICC.values) < 0)))
print("ICC of ln|Z|: 1 kHz", round(RES["icc_freq"]["first"], 3), "| 1 MHz", round(RES["icc_freq"]["last"], 3),
      "| monotone decreasing:", RES["icc_freq"]["monotone"])

ICC of ln|Z|: 1 kHz 0.95 | 1 MHz 0.601 | monotone decreasing: True


## 14. Misclassified samples

In [18]:
st = TAB["sa_nested|fusion"].copy()
st["n"] = st.Sample.map(fits.groupby("Sample").size())
st["ok"] = (st.p >= .5).astype(int) == st.y
p1t = st[(st.Patient == "P1") & (st.y == 1)].p
p1h_wrong = st[(st.Patient == "P1") & (st.y == 0) & (~st.ok)].p
RES["misclassified"] = dict(n_min=int(st[~st.ok].n.min()), n_max=int(st[~st.ok].n.max()),
                            mean_n_correct=float(st[st.ok].n.mean()), n_correct=int(st.ok.sum()),
                            p1_tumor_min=float(p1t.min()), p1_tumor_max=float(p1t.max()),
                            p1_healthy_wrong=float(p1h_wrong.iloc[0]) if len(p1h_wrong) else None)
print("misclassified:", RES["misclassified"])

misclassified: {'n_min': 6, 'n_max': 14, 'mean_n_correct': 28.636363636363637, 'n_correct': 22, 'p1_tumor_min': 0.5076922178268433, 'p1_tumor_max': 0.5081905126571655, 'p1_healthy_wrong': 0.5066224932670593}


## 15. Save results

In [19]:
def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


RES["summary"] = SUM
RES["ci"] = CI
RES["diff"] = DIFF
json.dump(clean(RES), open(f"{OUT}/ijs_results.json", "w"), indent=1, default=float)
pd.DataFrame([dict(Config=k, **{m: v for m, v in s.items()}) for k, s in SUM.items()]).to_csv(f"{OUT}/summary_all.csv", index=False)
for k, st in TAB.items():
    st.to_csv(f"{OUT}/sampletable_{k.replace('|', '_')}.csv", index=False)
tick("done")

[   263s] done


## 16. Additional analyses: permutation null, fully nested pipeline selection, size-matched referencing, P2 label swap, logistic regression, simulation
These analyses are implemented as stand-alone scripts (written below) because they are long: about 25 minutes for the
1000-permutation null, 12 minutes for the real-data analyses (fully nested selection re-ranks frequencies in all 30 inner folds), 5 minutes for the partially nested comparison, 30 minutes for the simulation, about 1 hour for the robustness analyses of the fully nested selection, 15 minutes for the extended simulation, and about 10 hours for the selection-aware permutation test (100 permutations, each repeating the complete nested selection) on two CPU cores; the second-revision analyses (`ijs_revision2_extras.py`) take less than a minute. Each script
checkpoints its progress and resumes after an interruption. If the result files already exist in `outputs_ijs/revision`
(for example from the code repository), they are loaded; delete them to recompute everything from scratch.

In [20]:
%%writefile rev_common.py
"""
rev_common.py -- shared helpers for the revision analyses.

* lw_fit / fast_rep : numerically identical, faster re-implementation of
  eg.build_rep_g (closed-form Ledoit-Wolf as in scikit-learn);
* run_eval          : outer leave-one-patient-out evaluation on precomputed
  representations (identical to eg.evaluate_g for fold-independent features);
* LogitClf / backend: L2 logistic-regression backend that can replace XGBoost
  everywhere ep.fit_predict is used.
"""
from contextlib import contextmanager

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import eis_pipeline as ep
import eis_groups as eg


# ------------------------------------------------------------------ Ledoit-Wolf
def lw_fit(X):
    """Ledoit-Wolf location and precision (same estimator as sklearn.covariance.LedoitWolf)."""
    n, d = X.shape
    mu = X.mean(0)
    Xc = X - mu
    emp = Xc.T @ Xc / n
    if d == 1:
        s, m = 0.0, float(emp[0, 0])
    else:
        X2 = Xc ** 2
        tr = X2.sum(0) / n
        m = tr.sum() / d
        beta_ = (X2.T @ X2).sum()
        delta_ = ((Xc.T @ Xc) ** 2).sum() / n ** 2
        beta = (beta_ / n - delta_) / (d * n)
        delta = (delta_ - 2.0 * m * tr.sum() + d * m * m) / d
        beta = min(beta, delta)
        s = 0.0 if beta == 0 else beta / delta
    cov = (1.0 - s) * emp
    cov[np.diag_indices(d)] += s * m
    return mu, np.linalg.pinv(cov, hermitian=True)


def fast_rep(X, pid, y, groups, patients):
    """Same output as eg.build_rep_g."""
    n, d = X.shape
    A = np.full((n, d), np.nan)
    M = np.full((n, 1), np.nan)
    for p in patients:
        rows = np.flatnonzero(pid == p)
        healthy = np.flatnonzero((pid == p) & (y == 0))
        cache = {}
        for i in rows:
            pool = healthy[groups[healthy] != groups[i]]
            if pool.size == 0:
                continue
            key = pool.tobytes()
            if key not in cache:
                ref = X[pool]
                med = np.median(ref, axis=0)
                cache[key] = (med, *lw_fit(ref)) if ref.shape[0] >= 2 else (med, None, None)
            med, mu, prec = cache[key]
            A[i] = np.abs(X[i] - med)
            if mu is None:
                M[i, 0] = 0.0
            else:
                dd = X[i] - mu
                M[i, 0] = float(np.sqrt(max(dd @ prec @ dd, 0.0)))
    return A, M, ~np.isnan(A).any(1)


# ------------------------------------------------------------------ evaluation
def run_eval(A, M, ok, pid, y, sgroups, patients, seeds, weights=None, test_patients=None):
    recs = []
    for p in (patients if test_patients is None else test_patients):
        trp = [q for q in patients if q != p]
        tr = np.isin(pid, trp) & ok
        te = (pid == p) & ok
        if te.sum() == 0:
            continue
        w = ep.patient_class_weights(pid[tr], y[tr])
        wf = 0.5 if weights is None else weights[p]
        for s in seeds:
            a, b, _ = ep.fit_predict(A[tr], M[tr], y[tr], w, A[te], M[te], seed=int(s))
            recs.append(pd.DataFrame(dict(Repeat=int(s), Patient=p, Sample=sgroups[te], y=y[te],
                                          Row=np.flatnonzero(te), p_abs=a, p_mah=b, p=wf * a + (1 - wf) * b)))
    return pd.concat(recs, ignore_index=True)


def sample_ba(recs, col, keep):
    st = eg.sample_table(recs, col, keep)
    return eg.pooled(st)["BA"], st


# ------------------------------------------------------------------ classifier backends
class LogitClf:
    """L2 logistic regression on log(1 + feature), standardized; deterministic."""

    def __init__(self, seed=0, C=1.0):
        self.C = C

    def fit(self, X, y, sample_weight=None):
        Z = np.log1p(np.abs(X))
        self.sc = StandardScaler().fit(Z)
        self.m = LogisticRegression(C=self.C, max_iter=5000).fit(self.sc.transform(Z), y, sample_weight=sample_weight)
        return self

    def predict_proba(self, X):
        return self.m.predict_proba(self.sc.transform(np.log1p(np.abs(X))))


@contextmanager
def backend(kind):
    old = ep.make_classifier
    if kind == "logit":
        ep.make_classifier = lambda seed: LogitClf(seed)
    try:
        yield
    finally:
        ep.make_classifier = old


def matched_rep(X, pid, y, G, patients):
    """Reference-size-matched sample-aware representation.
    A healthy query is referenced against the patient's other healthy samples (n_h - 1 samples).
    A tumor query is referenced n_h times, each time leaving one healthy sample out, so that both
    classes see references built from the same number of samples. Returns (row index, A, M) for
    every (query, reference version); predictions are averaged over versions."""
    idx, As, Ms = [], [], []
    for p in patients:
        healthy = np.flatnonzero((pid == p) & (y == 0))
        hs = np.unique(G[healthy])
        cache = {}
        for i in np.flatnonzero(pid == p):
            if y[i] == 0:
                pools = [healthy[G[healthy] != G[i]]]
            elif hs.size >= 2:
                pools = [healthy[G[healthy] != s] for s in hs]
            else:
                pools = [healthy]
            for pool in pools:
                if pool.size == 0:
                    continue
                key = pool.tobytes()
                if key not in cache:
                    ref = X[pool]
                    cache[key] = (np.median(ref, axis=0), *lw_fit(ref)) if ref.shape[0] >= 2 else (np.median(ref, axis=0), None, None)
                med, mu, prec = cache[key]
                dd = X[i] - (mu if mu is not None else med)
                idx.append(i)
                As.append(np.abs(X[i] - med))
                Ms.append(0.0 if mu is None else float(np.sqrt(max(dd @ prec @ dd, 0.0))))
    return np.asarray(idx), np.asarray(As), np.asarray(Ms)[:, None]


def run_eval_matched(X, pid, y, G, patients, seeds, test_patients=None):
    idx, A, M = matched_rep(X, pid, y, G, patients)
    pv, yv = pid[idx], y[idx]
    recs = []
    for p in (patients if test_patients is None else test_patients):
        trp = [q for q in patients if q != p]
        tr = np.isin(pv, trp)
        te = pv == p
        if te.sum() == 0:
            continue
        w = ep.patient_class_weights(pv[tr], yv[tr])
        for s in seeds:
            a, b, _ = ep.fit_predict(A[tr], M[tr], yv[tr], w, A[te], M[te], seed=int(s))
            d = pd.DataFrame(dict(Row=idx[te], p_abs=a, p_mah=b)).groupby("Row").mean().reset_index()
            d["p"] = 0.5 * d.p_abs + 0.5 * d.p_mah
            d["Repeat"], d["Patient"], d["Sample"], d["y"] = int(s), p, G[d.Row], y[d.Row]
            recs.append(d)
    return pd.concat(recs, ignore_index=True)


def permute_within_patient(y, pid, G, rng):
    """Shuffle class labels among the tissue samples of each patient (spectra follow their sample)."""
    y2 = y.copy()
    for p in np.unique(pid):
        m = pid == p
        samples = np.unique(G[m])
        lab = np.asarray([y[m & (G == s)][0] for s in samples])
        new = dict(zip(samples, rng.permutation(lab)))
        for s in samples:
            y2[m & (G == s)] = new[s]
    return y2


def evaluable(pid, y, G, patients):
    """Tissue samples with at least one same-patient healthy reference sample other than themselves."""
    _, _, ok = fast_rep(np.zeros((len(y), 1)) + np.arange(len(y))[:, None] * 0.0, pid, y, G, patients)
    return sorted(set(G[ok]))

Writing rev_common.py


In [21]:
%%writefile ijs_permutation.py
"""
ijs_permutation.py -- negative control: within-patient permutation of tissue-sample labels.

Under the null hypothesis that impedance carries no class information beyond the patient,
class labels are exchangeable among the tissue samples of a patient. Labels are therefore
shuffled among samples within each patient (all spectra of a sample keep a common label),
and the complete referencing + classification pipeline is re-run. The null distribution of the
sample-level balanced accuracy is obtained for both referencing designs and for the two
selection-free feature sets (Cole parameters; frequencies published by Veil et al.).
A design that is valid under the null must give null BA centered on 0.5.
"""
import json
import os
import sys
import time
from multiprocessing import Pool

import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import rev_common as rc

OUT = "outputs_ijs/revision"
os.makedirs(OUT, exist_ok=True)
N_PERM = int(sys.argv[1]) if len(sys.argv) > 1 else 1000

raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
R, NX, pid, y, meta = ep.harmonize(raw, ep.candidate_grid())
P = [f"P{i}" for i in range(1, 7)]
G = eg.sample_ids(raw)
GS = eg.spectrum_groups(len(y))
fits = pd.read_csv("outputs_ijs/cole_fits.csv")
FEATS = {
    "cole": np.column_stack([np.log(fits.R0), np.log(fits.Rinf), np.log10(fits.fc), fits.alpha]),
    "published": ep.build_vectors(*ep.harmonize(raw, np.array([3.55e3, 568.99e3, 655.13e3, 754.31e3, 868.51e3]))[:2],
                                  [0, 1, 2, 3, 4]),
}
DESIGNS = {"naive": GS, "sa": G}


def one(args):
    k, yk = args
    ep.XGB_PARAMS["n_jobs"] = 1
    keep = rc.evaluable(pid, yk, G, P)
    out = dict(perm=k, n_eval=len(keep),
               n_eval_healthy=int(len({s for s in keep if yk[G == s][0] == 0})))
    for fn, X in FEATS.items():
        for dn, grp in DESIGNS.items():
            A, M, ok = rc.fast_rep(X, pid, yk, grp, P)
            r = rc.run_eval(A, M, ok, pid, yk, G, P, seeds=[k])
            for col, tag in (("p", "fusion"), ("p_abs", "abs"), ("p_mah", "mah")):
                out[f"{fn}|{dn}|{tag}"] = rc.sample_ba(r, col, keep)[0]
    return out


if __name__ == "__main__":
    t0 = time.time()
    rng = np.random.default_rng(ep.SEED0 + 7)
    jobs = [(0, y.copy())] + [(k, rc.permute_within_patient(y, pid, G, rng)) for k in range(1, N_PERM + 1)]
    part = f"{OUT}/permutation_null_partial.csv"            # checkpoint: resume after interruption
    done = set(pd.read_csv(part).perm) if os.path.exists(part) else set()
    todo = [j for j in jobs if j[0] not in done]
    print(f"{len(done)} permutations already done, {len(todo)} to go", flush=True)
    with Pool(2) as pool:
        for i, r in enumerate(pool.imap_unordered(one, todo, chunksize=4)):
            pd.DataFrame([r]).to_csv(part, mode="a", header=not os.path.exists(part), index=False)
            if i % 50 == 0:
                print(f"[{time.time() - t0:6.0f}s] {i}/{len(todo)}", flush=True)
    rows = pd.read_csv(part).drop_duplicates("perm").to_dict("records")
    df = pd.DataFrame(rows).sort_values("perm").reset_index(drop=True)
    df.to_csv(f"{OUT}/permutation_null.csv", index=False)
    obs = json.load(open("outputs_ijs/ijs_results.json"))["summary"]
    null = df[df.perm > 0]
    res = dict(n_perm=int(len(null)))
    for c in [c for c in df.columns if "|" in c]:
        fn, dn, tag = c.split("|")
        key = {"cole": f"{dn}_cole|{tag}", "published": f"{dn}_published|{tag}"}[fn]
        o = obs.get(key, {}).get("BA", float(df.loc[df.perm == 0, c].iloc[0]))
        res[c] = dict(null_mean=float(null[c].mean()), null_sd=float(null[c].std(ddof=1)),
                      null_q025=float(null[c].quantile(.025)), null_q975=float(null[c].quantile(.975)),
                      frac_null_ge_0p75=float((null[c] >= .75).mean()),
                      observed=float(o), p_value=float((1 + (null[c] >= o - 1e-12).sum()) / (1 + len(null))))
    json.dump(res, open(f"{OUT}/permutation_results.json", "w"), indent=1)
    print(json.dumps(res, indent=1))
    print(f"done in {time.time() - t0:.0f}s")

Writing ijs_permutation.py


In [22]:
%%writefile ijs_revision.py
"""
ijs_revision.py -- additional real-data analyses for the revised IJS manuscript.

R1  Fully nested pipeline selection: inside every outer fold, the representation
    (nested-selected frequencies or Cole parameters) and the branch (fusion, absolute,
    Mahalanobis) are chosen by inner leave-one-patient-out on the five training patients only;
    the chosen pipeline is then applied once to the held-out patient. Within the inner loop,
    the frequencies are re-ranked in every inner fold on the four inner-training patients, so
    every data-driven choice is nested at both levels. With --partial-nesting, the outer-fold
    frequency ranking is reused in the inner folds instead (incomplete nesting, for comparison);
    results go to *_v2_outer_ranking files and the checkpoint is not touched. This turns the
    exploratory Cole-absolute result into a selection-honest estimate.
R2  Reference-size-matched sample-aware referencing (rc.matched_rep).
R3  P2 label-swap sensitivity: the complete factorial re-run with the P2 class labels exchanged.
R4  Logistic-regression backend: the complete factorial (ranking and evaluation) re-run with
    L2 logistic regression instead of XGBoost.
Outputs: outputs_ijs/revision/revision_results.json and CSV tables.
"""
import json
import os
import pickle
import sys
import time

import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import rev_common as rc

eg.build_rep_g = rc.fast_rep          # identical results (verified), several times faster
OUT = "outputs_ijs/revision"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
tick = lambda m: print(f"[{time.time() - T0:6.0f}s] {m}", flush=True)

raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
GRID = ep.candidate_grid()
R, NX, pid, y, meta = ep.harmonize(raw, GRID)
P = [f"P{i}" for i in range(1, 7)]
G = eg.sample_ids(raw)
GS = eg.spectrum_groups(len(y))
fits = pd.read_csv("outputs_ijs/cole_fits.csv")
THETA = np.column_stack([np.log(fits.R0), np.log(fits.Rinf), np.log10(fits.fc), fits.alpha])
Rv, NXv, *_ = ep.harmonize(raw, np.array([3.55e3, 568.99e3, 655.13e3, 754.31e3, 868.51e3]))
XPUB = ep.build_vectors(Rv, NXv, [0, 1, 2, 3, 4])
SEL = json.load(open("outputs_ijs/selections_idx.json"))
EVAL = rc.evaluable(pid, y, G, P)
SEEDS = range(ep.N_REPEATS)
khz = lambda idx: [round(float(GRID[j]) / 1e3, 3) for j in idx]
Xsel = lambda s: (lambda p: ep.build_vectors(R, NX, s[p] if isinstance(s, dict) else s))
RES, TAB = {}, {}


def summarize(name, recs, keep, cols=(("p", "fusion"), ("p_abs", "abs"), ("p_mah", "mah"))):
    for col, tag in cols:
        s, st = eg.summary(recs, col, keep=keep)
        RES.setdefault("summary", {})[f"{name}|{tag}"] = s
        TAB[f"{name}|{tag}"] = st


def boot(names, key):
    ci, diff, nb = eg.cluster_boot({k: TAB[k] for k in names})
    RES.setdefault("ci", {}).update({k: v for k, v in ci.items()})
    RES.setdefault("diff", {})[key] = diff
    RES.setdefault("boot_valid", {})[key] = nb


def factorial_selections(yy, cache_key):
    """Four selections of the factorial design, cached per outer fold so that an interrupted
    run resumes where it stopped."""
    path = f"{OUT}/selections_{cache_key}.json"
    S = json.load(open(path)) if os.path.exists(path) else {}
    for key, grp in (("naive_nested", GS), ("naive_leaky", GS), ("sa_nested", G), ("sa_leaky", G)):
        if key.endswith("leaky"):
            if key not in S:
                S[key] = [int(v) for v in eg.leaky_selection_g(R, NX, pid, yy, grp, P)]
        else:
            S.setdefault(key, {})
            for p in P:
                if p not in S[key]:
                    trp = [q for q in P if q != p]
                    S[key][p] = sorted(int(v) for v in eg.rank_frequencies_g(R, NX, pid, yy, grp, trp)["index"].to_numpy()[:ep.TOP_M])
                    json.dump(S, open(path, "w"), indent=1)
        json.dump(S, open(path, "w"), indent=1)
        tick(f"  selection {cache_key}/{key} ready")
    return S


CKPT = f"{OUT}/revision_checkpoint.pkl"
if os.path.exists(CKPT):
    RES, TAB = pickle.load(open(CKPT, "rb"))
    print("resuming; completed sections:", RES.get("_done", []))


def todo(name):
    return name not in RES.get("_done", [])


def done(name):
    RES.setdefault("_done", []).append(name)
    pickle.dump((RES, TAB), open(CKPT, "wb"))


# =========================================================== R1 nested pipeline selection
INNER_SEL_PATH = f"{OUT}/selections_inner.json"
INNER_SEL = json.load(open(INNER_SEL_PATH)) if os.path.exists(INNER_SEL_PATH) else {}
INNER_HOW = {}


def rank_frequencies_lsoo(train_patients, seed=ep.SEED0):
    """Fallback ranking when patient-level inner validation is impossible (only one of the
    training patients has two or more healthy samples): leave-one-tissue-sample-out validation
    within the training patients, scored with the same criterion D(f) over the patients whose
    held-out predictions contain both classes."""
    rows = []
    for j in range(R.shape[1]):
        X = ep.build_vectors(R, NX, [j])
        A, M, ok = rc.fast_rep(X, pid, y, G, train_patients)
        pool = np.isin(pid, train_patients) & ok
        pred = np.full(len(y), np.nan)
        for smp in np.unique(G[pool]):
            te = pool & (G == smp)
            tr = pool & (G != smp)
            if len(np.unique(y[tr])) < 2:
                continue
            w = ep.patient_class_weights(pid[tr], y[tr])
            _, _, pf = ep.fit_predict(A[tr], M[tr], y[tr], w, A[te], M[te], seed)
            pred[te] = pf
        bas, aucs = [], []
        for q in train_patients:
            m = (pid == q) & np.isfinite(pred)
            if len(np.unique(y[m])) < 2:
                continue
            mm = ep.patient_metrics(y[m], pred[m])
            bas.append(mm["BA"]); aucs.append(mm["AUC"])
        ba = np.asarray(bas)
        sd = float(ba.std(ddof=1)) if ba.size > 1 else 0.0
        rows.append(dict(index=j, MeanBA=float(ba.mean()), WorstBA=float(ba.min()), SD=sd, MeanAUC=float(np.nanmean(aucs)),
                         Score=float(np.sqrt((1 - ba.mean()) ** 2 + (1 - ba.min()) ** 2 + sd ** 2))))
    return pd.DataFrame(rows).sort_values(["Score", "MeanBA", "WorstBA", "SD", "MeanAUC"],
                                          ascending=[True, False, False, True, False]).reset_index(drop=True)


def n_scorable(train_patients):
    """Number of training patients that can be scored by patient-level inner validation."""
    multi = [v for v in train_patients if len(np.unique(G[(pid == v) & (y == 0)])) >= 2]
    return len(multi) if len(multi) >= 2 else 0


def inner_selection(p, q):
    """Top-5 frequencies ranked on the outer training patients of fold p without inner test
    patient q (sample-wise referencing): by leave-one-patient-out when at least two of them have
    two or more healthy samples, otherwise by leave-one-tissue-sample-out; cached."""
    key = f"{p}|{q}"
    if key not in INNER_SEL:
        tr4 = [v for v in P if v not in (p, q)]
        if n_scorable(tr4):
            ranking, how = eg.rank_frequencies_g(R, NX, pid, y, G, tr4), "LOPO"
        else:
            ranking, how = rank_frequencies_lsoo(tr4), "LOSO"
        INNER_SEL[key] = sorted(int(v) for v in ranking["index"].to_numpy()[:ep.TOP_M])
        INNER_HOW[key] = how
        json.dump(INNER_SEL, open(INNER_SEL_PATH, "w"), indent=1)
        tick(f"  inner selection {key} ({INNER_HOW[key]}): {khz(INNER_SEL[key])} kHz")
    return INNER_SEL[key]


PARTIAL = "--partial-nesting" in sys.argv
if PARTIAL:
    RES, TAB = {}, {}

if PARTIAL or todo("R1"):
    tick("R1 nested pipeline selection ...")
    CANDS = [(rep, br) for rep in ("freq", "cole") for br in ("fusion", "abs", "mah")]   # original design first
    chosen, recs1, inner_tab = {}, [], []
    for p in P:
        trp = [q for q in P if q != p]
        feats = {"freq": ep.build_vectors(R, NX, SEL["sa_nested"][p]), "cole": THETA}
        scores = {}
        for rep in feats:
            parts = []
            for q in trp:
                # inner fold q: frequencies are re-ranked on the four inner-training patients only,
                # so q influences neither the frequencies, nor the classifier, nor the reference
                if rep == "freq" and PARTIAL:
                    Xq = feats["freq"]          # outer-fold ranking reused: q helped choose the frequencies
                elif rep == "freq":
                    Xq = ep.build_vectors(R, NX, inner_selection(p, q))
                else:
                    Xq = THETA
                A, M, ok = rc.fast_rep(Xq, pid, y, G, trp)
                parts.append(rc.run_eval(A, M, ok, pid, y, G, trp, seeds=[ep.SEED0], test_patients=[q]))
            r = pd.concat(parts, ignore_index=True)
            keep_in = sorted(set(G[np.isin(pid, trp)]) & set(r.Sample))
            for col, br in (("p", "fusion"), ("p_abs", "abs"), ("p_mah", "mah")):
                m = eg.pooled(eg.sample_table(r, col, keep_in))
                scores[(rep, br)] = (m["BA"], m["AUC"] if np.isfinite(m["AUC"]) else 0.0)
                inner_tab.append(dict(Fold=p, Representation=rep, Branch=br, InnerBA=m["BA"], InnerAUC=m["AUC"]))
        best = max(CANDS, key=lambda c: (round(scores[c][0], 10), round(scores[c][1], 10), -CANDS.index(c)))
        chosen[p] = dict(representation=best[0], branch=best[1], inner_BA=scores[best][0])
        X = feats[best[0]]
        A, M, ok = rc.fast_rep(X, pid, y, G, P)
        r = rc.run_eval(A, M, ok, pid, y, G, P, seeds=SEEDS, test_patients=[p])
        r["p_sel"] = r[{"fusion": "p", "abs": "p_abs", "mah": "p_mah"}[best[1]]]
        recs1.append(r)
    sfx = "_v2_outer_ranking" if PARTIAL else ""
    pd.DataFrame(inner_tab).to_csv(f"{OUT}/nested_pipeline_inner_scores{sfx}.csv", index=False)
    recs1 = pd.concat(recs1, ignore_index=True)
    summarize("nested_pipeline", recs1, EVAL, cols=(("p_sel", "selected"),))
    RES["nested_pipeline_choice"] = chosen
    if PARTIAL:
        ci, _, _ = eg.cluster_boot({"nested_pipeline|selected": TAB["nested_pipeline|selected"]})
        RES["ci"] = ci
        json.dump(json.loads(json.dumps(RES, default=float)), open(f"{OUT}/revision_results_v2_outer_ranking.json", "w"), indent=1)
        TAB["nested_pipeline|selected"].to_csv(f"{OUT}/sampletable_partial_pipeline_selected.csv", index=False)
        print("partial nesting:", RES["summary"]["nested_pipeline|selected"], ci)
        sys.exit(0)
    RES["inner_selected_khz"] = {k: khz(v) for k, v in INNER_SEL.items()}
    RES["inner_selection_method"] = {k: ("LOPO" if n_scorable([v for v in P if v not in k.split("|")]) else "LOSO") for k in INNER_SEL}
    print(chosen)
    done("R1")

# ======================================================= R2 reference-size-matched design
if todo("R2"):
    tick("R2 reference-size-matched referencing ...")
    for name, fx in (("matched_nested", Xsel(SEL["sa_nested"])), ("matched_cole", lambda p: THETA),
                     ("matched_published", lambda p: XPUB)):
        if name == "matched_nested":
            r = pd.concat([rc.run_eval_matched(fx(p), pid, y, G, P, SEEDS, test_patients=[p]) for p in P], ignore_index=True)
        else:
            r = rc.run_eval_matched(fx(None), pid, y, G, P, SEEDS)
        summarize(name, r, EVAL)

    # reference the main-analysis tables for the bootstrap
    for k in ("naive_nested|fusion", "naive_leaky|fusion", "sa_nested|fusion", "sa_leaky|fusion",
              "sa_cole|abs", "sa_cole|fusion", "sa_published|fusion", "sa_nested|abs", "sa_nested|mah"):
        TAB[k] = pd.read_csv(f"outputs_ijs/sampletable_{k.replace('|', '_')}.csv")
    boot(["nested_pipeline|selected", "sa_nested|fusion", "sa_cole|abs", "naive_nested|fusion",
          "matched_nested|fusion", "matched_nested|abs", "matched_nested|mah",
          "matched_cole|fusion", "matched_cole|abs", "matched_published|fusion", "sa_published|fusion"], "main_extra")
    done("R2")

# ======================================================= R3 P2 label swap
if todo("R3"):
    tick("R3 P2 label swap: selections ...")
    y2 = y.copy()
    y2[pid == "P2"] = 1 - y2[pid == "P2"]
    EVAL2 = rc.evaluable(pid, y2, G, P)
    ev2 = pd.DataFrame(dict(G=G, y=y2, pid=pid)).drop_duplicates("G").set_index("G").loc[EVAL2]
    RES["p2swap_evaluable"] = dict(n=len(EVAL2), healthy=int((ev2.y == 0).sum()), tumor=int((ev2.y == 1).sum()),
                                   healthy_patients=sorted(ev2[ev2.y == 0].pid.unique().tolist()))
    S2 = factorial_selections(y2, "p2swap")
    RES["p2swap_selected_khz"] = {k: ({p: khz(v) for p, v in s.items()} if isinstance(s, dict) else khz(s)) for k, s in S2.items()}
    tick("R3 P2 label swap: evaluations ...")
    for name, fx, grp in (("p2_naive_nested", Xsel(S2["naive_nested"]), GS), ("p2_naive_leaky", Xsel(S2["naive_leaky"]), GS),
                          ("p2_sa_nested", Xsel(S2["sa_nested"]), G), ("p2_sa_leaky", Xsel(S2["sa_leaky"]), G),
                          ("p2_sa_cole", lambda p: THETA, G), ("p2_sa_published", lambda p: XPUB, G)):
        summarize(name, eg.evaluate_g(fx, pid, y2, grp, P, G), EVAL2)
    boot(["p2_naive_leaky|fusion", "p2_naive_nested|fusion", "p2_sa_leaky|fusion", "p2_sa_nested|fusion",
          "p2_sa_cole|abs", "p2_sa_published|fusion"], "p2swap")
    done("R3")

# ======================================================= R4 logistic-regression backend
if todo("R4"):
    tick("R4 logistic regression: selections ...")
    with rc.backend("logit"):
        S3 = factorial_selections(y, "logit")
        RES["logit_selected_khz"] = {k: ({p: khz(v) for p, v in s.items()} if isinstance(s, dict) else khz(s)) for k, s in S3.items()}
        tick("R4 logistic regression: evaluations ...")
        for name, fx, grp in (("lr_naive_nested", Xsel(S3["naive_nested"]), GS), ("lr_naive_leaky", Xsel(S3["naive_leaky"]), GS),
                              ("lr_sa_nested", Xsel(S3["sa_nested"]), G), ("lr_sa_leaky", Xsel(S3["sa_leaky"]), G),
                              ("lr_naive_cole", lambda p: THETA, GS), ("lr_sa_cole", lambda p: THETA, G),
                              ("lr_sa_published", lambda p: XPUB, G)):
            summarize(name, eg.evaluate_g(fx, pid, y, grp, P, G, n_repeats=1), EVAL)
    boot(["lr_naive_leaky|fusion", "lr_naive_nested|fusion", "lr_sa_leaky|fusion", "lr_sa_nested|fusion",
          "lr_naive_cole|abs", "lr_sa_cole|abs", "lr_naive_cole|fusion", "lr_sa_cole|fusion", "lr_sa_published|fusion"], "logit")
    done("R4")


# ======================================================= save
def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


json.dump(clean(RES), open(f"{OUT}/revision_results.json", "w"), indent=1, default=float)
pd.DataFrame([dict(Config=k, **v) for k, v in RES["summary"].items()]).to_csv(f"{OUT}/revision_summary.csv", index=False)
for k, st in TAB.items():
    st.to_csv(f"{OUT}/sampletable_{k.replace('|', '_')}.csv", index=False)
for k, s in RES["summary"].items():
    ci = RES["ci"].get(k)
    print(f"{k:32s} BA={s['BA']:.3f} ({s['TP']}/{s['N1']}, {s['TN']}/{s['N0']}) AUC={s['AUC']:.3f}"
          + (f"  CI {ci[0]:.3f}-{ci[1]:.3f}" if ci else ""))
tick("done")

Writing ijs_revision.py


In [23]:
%%writefile ijs_simulation.py
"""
ijs_simulation.py -- does the pseudo-replication inflation generalize beyond the bladder data?

Hierarchical generator (whitened four-dimensional feature space, analogous to the four Cole
parameters):
    patient offset      mu_p  ~ N(0, 4 I)                          (removed by referencing)
    healthy sample      theta_s = mu_p + b_s,            b_s ~ N(0, I)
    tumor sample        theta_s = mu_p + delta_p + b_s,  delta_p ~ N(delta, 0.25 I)
    spectrum            x_ij    = theta_s + e_ij,        e_ij ~ N(0, (1 - ICC)/ICC I)
with |delta| = Delta. Each patient contributes n_h healthy and n_t tumor tissue samples with
m spectra each. ICC is the share of the within-patient variance that is between samples; the
between-sample variance is held at 1 so that the separability of tissue samples (the quantity
that matters clinically) does not change with ICC; only the redundancy of repeated spectra does.

For every replicate the patient-referenced pipeline (absolute + Mahalanobis branches,
equal fusion, patient-class weights, leave-one-patient-out) is run with
    naive referencing        (leave-one-spectrum-out) and
    sample-aware referencing (leave-one-tissue-sample-out).
The cross-validated sample-level BA (the estimate) is compared with the TRUE deployment BA of
the same pipeline: the classifier trained on all simulated patients is applied to 40 fresh
patients, in which a NEW query tissue sample (healthy or tumor) is referenced against all of the
patient's existing healthy samples, as it would be in clinical use. Optimism = estimate - truth.
A third design, reference-size-matched sample-aware referencing (rc.matched_rep), removes the
asymmetry that a healthy query sees n_h - 1 reference samples whereas a tumor query sees n_h.
"""
import json
import os
import sys
import time
from itertools import product
from multiprocessing import Pool

import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import rev_common as rc

OUT = "outputs_ijs/revision"
os.makedirs(OUT, exist_ok=True)
D = 4
N_FRESH = 40


def simulate(rng, n_pat, n_h, n_t, m, icc, delta, offset=0):
    dvec = np.full(D, delta / np.sqrt(D))
    X, pid, y, G = [], [], [], []
    for p in range(n_pat):
        mu = rng.normal(0, 2.0, D)
        dp = dvec + rng.normal(0, 0.5, D)
        for c, ns in ((0, n_h), (1, n_t)):
            for s in range(ns):
                th = mu + c * dp + rng.normal(0, 1.0, D)
                X.append(th + rng.normal(0, np.sqrt((1 - icc) / icc), (m, D)))
                pid += [f"P{p + offset}"] * m
                y += [c] * m
                G += [f"P{p + offset}:{c}{s}"] * m
    return np.vstack(X), np.asarray(pid), np.asarray(y), np.asarray(G)


def fit_branches(A, M, y, w, seed):
    a = ep.make_classifier(seed).fit(A, y, sample_weight=w)
    b = ep.make_classifier(seed + 1).fit(M, y, sample_weight=w)
    return a, b


def pooled_ba(G, y, p):
    st = pd.DataFrame(dict(Patient=0, Sample=G, y=y, p=p)).groupby("Sample").agg(y=("y", "first"), p=("p", "mean"))
    return eg.pooled(st)["BA"]


def deployment_set(rng, n_h, n_t, m, icc, delta):
    """Fresh patients as in clinical use: n_h healthy REFERENCE samples plus one new healthy and
    n_t tumor QUERY samples; every query is referenced against all n_h reference samples."""
    X, pid, y, G = simulate(rng, N_FRESH, n_h + 1, n_t, m, icc, delta, offset=10_000)
    query_h = np.asarray([g.endswith(f":0{n_h}") for g in G])          # last healthy sample = query
    yref = y.copy()
    yref[query_h] = 2                                                  # excluded from the reference pool
    A, M, ok = rc.fast_rep(X, pid, yref, G, np.unique(pid))
    q = ok & (query_h | (y == 1))
    return A[q], M[q], y[q], G[q]


def replicate(args):
    cell, rep, kind = args
    ep.XGB_PARAMS["n_jobs"] = 1
    n_pat, n_h, n_t, m, icc, delta = cell
    rng = np.random.default_rng([ep.SEED0, rep, int(1000 * icc), m, int(100 * delta), n_pat, n_h])
    X, pid, y, G = simulate(rng, n_pat, n_h, n_t, m, icc, delta)
    Af, Mf, yf, Gf = deployment_set(rng, n_h, n_t, m, icc, delta)
    pats = np.unique(pid)
    keep = sorted(set(G[rc.fast_rep(X, pid, y, G, pats)[2]]))
    out = dict(n_pat=n_pat, n_h=n_h, n_t=n_t, m=m, icc=icc, delta=delta, rep=rep, backend=kind)
    with rc.backend(kind):
        for dn in ("naive", "sa", "matched"):
            if dn == "matched":
                r = rc.run_eval_matched(X, pid, y, G, pats, seeds=[rep])
                idx, A, M = rc.matched_rep(X, pid, y, G, pats)
                yt, pt = y[idx], pid[idx]
            else:
                grp = np.asarray([f"s{i}" for i in range(len(y))]) if dn == "naive" else G
                A, M, ok = rc.fast_rep(X, pid, y, grp, pats)
                r = rc.run_eval(A, M, ok, pid, y, G, pats, seeds=[rep])
                A, M, yt, pt = A[ok], M[ok], y[ok], pid[ok]
            out[f"{dn}_est"] = rc.sample_ba(r, "p", keep)[0]
            out[f"{dn}_est_abs"] = rc.sample_ba(r, "p_abs", keep)[0]
            ca, cm = fit_branches(A, M, yt, ep.patient_class_weights(pt, yt), rep)
            pa, pm = ca.predict_proba(Af)[:, 1], cm.predict_proba(Mf)[:, 1]
            out[f"{dn}_true"] = pooled_ba(Gf, yf, 0.5 * pa + 0.5 * pm)
            out[f"{dn}_true_abs"] = pooled_ba(Gf, yf, pa)
    return out


def run(jobs, name, procs=2):
    """Run replicates in parallel; results are appended to a partial file so that an interrupted
    run resumes where it stopped."""
    t0 = time.time()
    part = f"{OUT}/sim_{name}_partial.csv"
    keycols = ["n_pat", "n_h", "n_t", "m", "icc", "delta", "rep", "backend"]
    done = set()
    if os.path.exists(part):
        done = set(map(tuple, pd.read_csv(part)[keycols].astype(str).values.tolist()))
    todo = [j for j in jobs if tuple(map(str, (*j[0], j[1], j[2]))) not in done]
    print(f"{name}: {len(jobs) - len(todo)} done, {len(todo)} to go", flush=True)
    with Pool(procs) as pool:
        for i, r in enumerate(pool.imap_unordered(replicate, todo, chunksize=2)):
            pd.DataFrame([r]).to_csv(part, mode="a", header=not os.path.exists(part), index=False)
            if i % 200 == 0:
                print(f"[{time.time() - t0:6.0f}s] {name} {i}/{len(todo)}", flush=True)
    df = pd.read_csv(part).drop_duplicates(keycols)
    df.to_csv(f"{OUT}/sim_{name}.csv", index=False)
    return df


if __name__ == "__main__":
    which = sys.argv[1] if len(sys.argv) > 1 else "all"
    DELTA = 4.0
    if which in ("grid", "all"):
        cells = [(6, 3, 3, m, icc, d) for icc, m, d in product([0.1, 0.25, 0.5, 0.75, 0.9, 0.97], [1, 5, 15, 30], [0.0, DELTA])]
        run([(c, r, "logit") for c in cells for r in range(30)], "grid")
    if which in ("xgb", "all"):
        cells = [(6, 3, 3, 15, icc, d) for icc in (0.1, 0.5, 0.9, 0.97) for d in (0.0, DELTA)]
        run([(c, r, "xgb") for c in cells for r in range(20)], "xgb")
    if which in ("precision", "all"):
        cells = [(n, nh, 3, 15, 0.95, DELTA) for n in (6, 12, 24, 48) for nh in (2, 3)]
        run([(c, r, "logit") for c in cells for r in range(100)], "precision")

Writing ijs_simulation.py


In [24]:
%%writefile sim_summary.py
"""sim_summary.py -- numbers of the simulation study quoted in the manuscript."""
import json

import numpy as np
import pandas as pd

REV = "outputs_ijs/revision"
g = pd.read_csv(f"{REV}/sim_grid.csv")
x = pd.read_csv(f"{REV}/sim_xgb.csv")
pr = pd.read_csv(f"{REV}/sim_precision.csv")
for d in (g, x, pr):
    for dn in ("naive", "sa", "matched"):
        d[f"{dn}_opt"] = d[f"{dn}_est"] - d[f"{dn}_true"]

cell = g.groupby(["delta", "icc", "m"])[[c for c in g.columns if c.endswith(("_est", "_true", "_opt"))]].mean()
S = {}
for dl in (0.0, 4.0):
    c = cell.loc[dl]
    rep = c[c.index.get_level_values("m") >= 5]
    tag = "null" if dl == 0 else "effect"
    S[tag] = dict(
        naive_opt_min=float(rep.naive_opt.min()), naive_opt_max=float(rep.naive_opt.max()),
        naive_opt_icc01_m5=float(c.loc[(0.1, 5), "naive_opt"]),
        naive_opt_icc097_m30=float(c.loc[(0.97, 30), "naive_opt"]),
        naive_est_icc097_m15=float(c.loc[(0.97, 15), "naive_est"]),
        naive_true_icc097_m15=float(c.loc[(0.97, 15), "naive_true"]),
        sa_est_icc097_m15=float(c.loc[(0.97, 15), "sa_est"]),
        sa_true_icc097_m15=float(c.loc[(0.97, 15), "sa_true"]),
        matched_est_icc097_m15=float(c.loc[(0.97, 15), "matched_est"]),
        true_icc097_m15=float(c.loc[(0.97, 15), "matched_true"]),
        sa_opt_min=float(rep.sa_opt.min()), sa_opt_max=float(rep.sa_opt.max()),
        sa_opt_m1_min=float(c[c.index.get_level_values("m") == 1].sa_opt.min()),
        sa_opt_m1_max=float(c[c.index.get_level_values("m") == 1].sa_opt.max()),
        matched_opt_min=float(c.matched_opt.min()), matched_opt_max=float(c.matched_opt.max()),
        matched_abs_mean=float(c.matched_opt.abs().mean()),
    )
# monotonic increase of naive optimism with ICC at m = 15 and 30
for m in (15, 30):
    for dl in (0.0, 4.0):
        v = cell.loc[dl].xs(m, level="m").naive_opt.values
        S.setdefault("naive_monotone_in_icc", {})[f"m{m}_d{dl:g}"] = bool(np.all(np.diff(v) > -0.02))

xc = x.groupby(["delta", "icc"])[[c for c in x.columns if c.endswith(("_est", "_true", "_opt"))]].mean()
S["xgb"] = dict(naive_opt_null_icc097=float(xc.loc[(0.0, 0.97), "naive_opt"]),
                naive_est_null_icc097=float(xc.loc[(0.0, 0.97), "naive_est"]),
                naive_opt_effect_icc097=float(xc.loc[(4.0, 0.97), "naive_opt"]),
                matched_opt_min=float(xc.matched_opt.min()), matched_opt_max=float(xc.matched_opt.max()),
                naive_opt_icc01_null=float(xc.loc[(0.0, 0.1), "naive_opt"]))
q = pr.groupby(["n_pat", "n_h"]).matched_est.quantile([.025, .975]).unstack()
w = (q[0.975] - q[0.025])
S["precision"] = {f"n{n}_h{h}": dict(lo=float(q.loc[(n, h), 0.025]), hi=float(q.loc[(n, h), 0.975]), width=float(w.loc[(n, h)]))
                  for n, h in q.index}
S["precision_true_mean"] = float(pr.matched_true.mean())
S["precision_sa_width"] = {f"n{n}_h{h}": float(v) for (n, h), v in
                           (pr.groupby(["n_pat", "n_h"]).sa_est.quantile(.975) - pr.groupby(["n_pat", "n_h"]).sa_est.quantile(.025)).items()}
json.dump(S, open(f"{REV}/simulation_summary.json", "w"), indent=1)
cell.round(3).to_csv(f"{REV}/simulation_cell_means.csv")
print(json.dumps(S, indent=1))

Writing sim_summary.py


In [25]:
%%writefile ijs_revision_figs.py
"""Figures 6 and 7 of the revised manuscript: permutation null and simulation study."""
import json
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import ijs_figures  # noqa: F401  (shared rcParams)
from ijs_figures import C_NAIVE, C_SA

REV = "outputs_ijs/revision"
FIG = "outputs_ijs/figs"
os.makedirs(FIG, exist_ok=True)
C_MATCH = "#3E7C59"


def permutation_figure():
    df = pd.read_csv(f"{REV}/permutation_null.csv")
    res = json.load(open(f"{REV}/permutation_results.json"))
    null = df[df.perm > 0]
    panels = [("cole|{}|abs", "(a) Cole, absolute"),
              ("cole|{}|fusion", "(b) Cole, equal fusion"),
              ("published|{}|fusion", "(c) Published set, fusion")]
    fig, axes = plt.subplots(1, 4, figsize=(11.0, 3.5), sharex=True)
    bins = np.linspace(0, 1, 26)
    for ax, (key, title) in zip(axes[:3], panels):
        for dn, col in (("naive", C_NAIVE), ("sa", C_SA)):
            k = key.format(dn)
            ax.hist(null[k], bins=bins, color=col, alpha=0.65)
            ax.axvline(res[k]["observed"], color=col, lw=2.6, ls="-")
        ax.set_title(title, fontsize=13.5)
    path = f"{REV}/selection_permutation_null.csv"
    ax = axes[3]
    if os.path.exists(path):
        sp = pd.read_csv(path)
        sn, so = sp[sp.perm > 0], sp[sp.perm == 0].iloc[0]
        for c, col in (("BA_prespecified", "#4F6D8F"), ("BA_selected", C_SA)):
            ax.hist(sn[c], bins=bins, color=col, alpha=0.65)
            ax.axvline(so[c], color=col, lw=2.6)
    ax.set_title("(d) Selection-aware", fontsize=13.5)
    for ax in axes:
        ax.axvline(0.5, color="0.2", lw=1.2, ls="--")
        ax.set_xlim(0, 1.02); ax.set_xticks([0, 0.5, 1]); ax.tick_params(labelsize=12)
        ax.set_xlabel("Balanced accuracy", fontsize=12.5)
    axes[0].set_ylabel("Permutations", fontsize=13)
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    h = [Patch(color=C_NAIVE, alpha=.65), Patch(color=C_SA, alpha=.65), Patch(color="#4F6D8F", alpha=.65),
         Line2D([], [], color="0.3", lw=2.6), Line2D([], [], color="0.2", lw=1.2, ls="--")]
    fig.legend(h, ["null, spectrum-wise (a-c)", "null, sample-wise (a-c); fully nested (d)", "null, pre-specified (d)",
                   "observed", "chance"], loc="upper center", ncol=5, fontsize=11.5, bbox_to_anchor=(0.5, 1.10),
               handlelength=1.4, columnspacing=1.0)
    fig.tight_layout()
    fig.savefig(f"{FIG}/Fig4_Permutation_Null.png", bbox_inches="tight")
    plt.close(fig)


def simulation_figure():
    g = pd.read_csv(f"{REV}/sim_grid.csv")
    pr = pd.read_csv(f"{REV}/sim_precision.csv")
    eff = g.delta.max()
    fig = plt.figure(figsize=(9.6, 6.6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.1], hspace=0.55, wspace=0.42)
    norm = TwoSlopeNorm(vmin=-0.15, vcenter=0, vmax=0.45)
    iccs = sorted(g.icc.unique())
    ms = sorted(g.m.unique())
    for k, (dn, title) in enumerate((("naive", "(a) Spectrum-wise"), ("sa", "(b) Sample-wise"),
                                     ("matched", "(c) Size-matched"))):
        ax = fig.add_subplot(gs[0, k])
        d = g[g.delta == eff].assign(opt=lambda t: t[f"{dn}_est"] - t[f"{dn}_true"])
        mat = d.groupby(["m", "icc"]).opt.mean().unstack().loc[ms, iccs].values
        im = ax.imshow(mat, cmap="RdBu_r", norm=norm, aspect="auto", origin="lower")
        ax.set_xticks(range(len(iccs)), [f"{v:g}" for v in iccs], fontsize=11)
        ax.set_yticks(range(len(ms)), [str(v) for v in ms], fontsize=12.5)
        ax.set_xlabel("ICC", fontsize=13)
        if k == 0:
            ax.set_ylabel("Spectra per sample, m", fontsize=13.5)
        ax.set_title(title, fontsize=14)
        ax.grid(False)
    cax = fig.add_axes([0.92, 0.57, 0.012, 0.3])
    cb = fig.colorbar(im, cax=cax); cb.set_label("optimism (estimate - truth)", fontsize=13); cb.ax.tick_params(labelsize=12)
    # (d) null distribution of the estimate
    ax = fig.add_subplot(gs[1, 0])
    d0 = g[(g.delta == 0) & (g.m == 15)]
    for dn, col, lab in (("naive", C_NAIVE, "spectrum-wise"), ("sa", C_SA, "sample-wise"),
                         ("matched", C_MATCH, "size-matched")):
        s = d0.groupby("icc")[f"{dn}_est"].mean()
        ax.plot(s.index, s.values, "o-", color=col, label=lab, ms=6)
    s = d0.groupby("icc")["sa_true"].mean()
    ax.plot(s.index, s.values, "k--", lw=1, label="truth")
    ax.set_ylim(0.3, 1.02)
    ax.set_xlabel("ICC")
    ax.set_ylabel("Estimated BA", fontsize=13)
    ax.set_title("(d) No class difference", fontsize=14)
    ax.set_ylim(0.3, 1.05)
    ax.legend(fontsize=12, loc="upper left")
    # (e) precision vs number of patients
    ax = fig.add_subplot(gs[1, 1:])
    for nh, mk in ((2, "s"), (3, "o")):
        for dn, col, off in (("sa", C_SA, -0.04), ("matched", C_MATCH, 0.04)):
            d = pr[pr.n_h == nh]
            q = d.groupby("n_pat")[f"{dn}_est"].quantile([.025, .5, .975]).unstack()
            x = np.log2(q.index.values) + off + (0.12 if nh == 3 else -0.12)
            ax.errorbar(x, q[0.5], yerr=[q[0.5] - q[0.025], q[0.975] - q[0.5]], fmt=mk, color=col, ms=6,
                        capsize=2.5, lw=1.2, label=f"{'sample-wise' if dn == 'sa' else 'size-matched'}, {nh} healthy samples")
    t = pr.groupby("n_pat")[["sa_true", "matched_true"]].mean().mean(axis=1)
    ax.plot(np.log2(t.index.values), t.values, "k--", lw=1, label="mean true BA")
    ax.set_xticks(np.log2(sorted(pr.n_pat.unique())), [str(v) for v in sorted(pr.n_pat.unique())])
    ax.set_xlabel("Number of patients")
    ax.set_ylabel("Estimated BA", fontsize=13)
    ax.set_title("(e) Precision (ICC 0.95, m = 15)", fontsize=14)
    ax.set_ylim(0.5, 1.0)
    ax.legend(fontsize=11.5, ncol=2, loc="upper center", bbox_to_anchor=(0.45, -0.17), columnspacing=1.0, handletextpad=0.4)
    fig.savefig(f"{FIG}/Fig5_Simulation.png", bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    if os.path.exists(f"{REV}/permutation_results.json"):
        permutation_figure()
    if os.path.exists(f"{REV}/sim_precision.csv"):
        simulation_figure()


In [26]:
%%writefile nested_core.py
"""
nested_core.py -- fully nested pipeline selection as a reusable function.

full_nested(yk, ...) runs, for labels yk:
  outer leave-one-patient-out; in every outer fold
    * frequencies ranked on the five outer-training patients (for the outer model),
    * inner leave-one-patient-out over the five training patients, in which the frequency
      candidate is re-ranked on the four inner-training patients of every inner fold
      (patient-level ranking when at least two of them have >= 2 healthy samples, otherwise a
      fallback chosen by `fallback`),
    * six candidates (frequencies | Cole) x (fusion | absolute | Mahalanobis) compared by inner
      sample-level BA (ties: inner AUC, then the pre-specified frequency fusion),
    * the chosen candidate refitted on the five training patients and applied once to the
      held-out patient.
Options
  fallback : "loso"    leave-one-tissue-sample-out ranking within the four patients (main analysis)
             "outer"   reuse the outer-fold ranking (favours the frequency candidate)
             "generic" fixed decade-spaced set (no data-driven ranking)
  design   : "sa" sample-wise referencing; "matched" size-matched sample-wise referencing
             (ranking, inner comparison, and outer evaluation all use the same design)
  patients : subset of patients (e.g. without P2)
With design="sa", fallback="loso" and seeds=range(30) the result is identical to ijs_revision.py R1.
"""
import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import rev_common as rc

GRID = ep.candidate_grid()
CANDS = [(rep, br) for rep in ("freq", "cole") for br in ("fusion", "abs", "mah")]
BRCOL = {"fusion": "p", "abs": "p_abs", "mah": "p_mah"}


class Data:
    def __init__(self, raw, fits):
        self.R, self.NX, self.pid, self.y, _ = ep.harmonize(raw, GRID)
        self.G = eg.sample_ids(raw)
        self.THETA = np.column_stack([np.log(fits.R0), np.log(fits.Rinf), np.log10(fits.fc), fits.alpha])
        self.generic = sorted({int(np.argmin(np.abs(GRID - f))) for f in (1e3, 1e4, 1e5, 5e5, 1e6)})


def reps(D, X, yk, patients, design):
    """Row index, absolute-deviation and Mahalanobis representations (one row per query version)."""
    if design == "sa":
        A, M, ok = rc.fast_rep(X, D.pid, yk, D.G, patients)
        idx = np.flatnonzero(ok)
        return idx, A[ok], M[ok]
    return rc.matched_rep(X, D.pid, yk, D.G, patients)


def fit_eval(D, rep, yk, train_patients, test_mask_fn, seed):
    """Fit on rows of train_patients, predict rows selected by test_mask_fn(row_pid, row_group);
    predictions of query versions are averaged per spectrum."""
    idx, A, M = rep
    pv, gv, yv = D.pid[idx], D.G[idx], yk[idx]
    te = test_mask_fn(pv, gv)
    tr = np.isin(pv, train_patients) & ~te
    if te.sum() == 0 or len(np.unique(yv[tr])) < 2:
        return None
    w = ep.patient_class_weights(pv[tr], yv[tr])
    a, b, pf = ep.fit_predict(A[tr], M[tr], yv[tr], w, A[te], M[te], seed=int(seed))
    d = pd.DataFrame(dict(Row=idx[te], p_abs=a, p_mah=b, p_rank=pf)).groupby("Row").mean().reset_index()
    d["p"] = 0.5 * d.p_abs + 0.5 * d.p_mah
    d["y"], d["Patient"], d["Sample"] = yk[d.Row], D.pid[d.Row], D.G[d.Row]
    return d


def _score(bas, aucs, j):
    ba = np.asarray(bas)
    sd = float(ba.std(ddof=1)) if ba.size > 1 else 0.0
    return dict(index=j, MeanBA=float(ba.mean()), WorstBA=float(ba.min()), SD=sd, MeanAUC=float(np.nanmean(aucs)),
                Score=float(np.sqrt((1 - ba.mean()) ** 2 + (1 - ba.min()) ** 2 + sd ** 2)))


def _top(rows):
    df = pd.DataFrame(rows).sort_values(["Score", "MeanBA", "WorstBA", "SD", "MeanAUC"],
                                        ascending=[True, False, False, True, False])
    return sorted(int(v) for v in df["index"].to_numpy()[:ep.TOP_M])


def rank_lopo(D, yk, train, design, seed=ep.SEED0):
    rows = []
    for j in range(D.R.shape[1]):
        rep = reps(D, ep.build_vectors(D.R, D.NX, [j]), yk, train, design)
        bas, aucs = [], []
        for q in train:
            if len(np.unique(yk[rep[0]][D.pid[rep[0]] == q])) < 2:
                continue
            d = fit_eval(D, rep, yk, train, lambda pv, gv: pv == q, seed)
            if d is None:
                continue
            m = ep.patient_metrics(d.y.to_numpy(), d.p_rank.to_numpy())
            bas.append(m["BA"]); aucs.append(m["AUC"])
        rows.append(_score(bas, aucs, j))
    return _top(rows)


def rank_loso(D, yk, train, design, seed=ep.SEED0):
    rows = []
    for j in range(D.R.shape[1]):
        rep = reps(D, ep.build_vectors(D.R, D.NX, [j]), yk, train, design)
        parts = []
        for smp in np.unique(D.G[rep[0]]):
            d = fit_eval(D, rep, yk, train, lambda pv, gv: gv == smp, seed)
            if d is not None:
                parts.append(d)
        d = pd.concat(parts, ignore_index=True)
        bas, aucs = [], []
        for q in train:
            dq = d[d.Patient == q]
            if dq.y.nunique() < 2:
                continue
            m = ep.patient_metrics(dq.y.to_numpy(), dq.p_rank.to_numpy())
            bas.append(m["BA"]); aucs.append(m["AUC"])
        rows.append(_score(bas, aucs, j))
    return _top(rows)


def scorable(D, train, yk):
    multi = [v for v in train if len(np.unique(D.G[(D.pid == v) & (yk == 0)])) >= 2]
    return len(multi) >= 2


def full_nested(D, yk, patients, fallback="loso", design="sa", seeds=(ep.SEED0,), cache=None, keep=None):
    cache = {} if cache is None else cache
    pts = list(patients)
    keep = rc.evaluable(D.pid, yk, D.G, pts) if keep is None else keep
    chosen, inner_rows, recs, inner_method = {}, [], [], {}
    for p in pts:
        trp = [q for q in pts if q != p]
        k_out = ("outer", design, p)
        if k_out not in cache:
            cache[k_out] = rank_lopo(D, yk, trp, design)
        Fout = cache[k_out]
        scores = {}
        for rep_name in ("freq", "cole"):
            parts = []
            for q in trp:
                tr4 = [v for v in trp if v != q]
                if rep_name == "cole":
                    X = D.THETA
                else:
                    if scorable(D, tr4, yk):
                        k_in, method = ("lopo", design, p, q), "LOPO"
                        if k_in not in cache:
                            cache[k_in] = rank_lopo(D, yk, tr4, design)
                        F = cache[k_in]
                    elif fallback == "loso":
                        k_in, method = ("loso", design, p, q), "LOSO"
                        if k_in not in cache:
                            cache[k_in] = rank_loso(D, yk, tr4, design)
                        F = cache[k_in]
                    elif fallback == "outer":
                        F, method = Fout, "outer"
                    elif fallback == "generic":
                        F, method = D.generic, "generic"
                    else:
                        raise ValueError(fallback)
                    inner_method[f"{p}|{q}"] = method
                    X = ep.build_vectors(D.R, D.NX, F)
                rep = reps(D, X, yk, trp, design)
                d = fit_eval(D, rep, yk, trp, lambda pv, gv: pv == q, ep.SEED0)
                if d is not None:
                    parts.append(d)
            r = pd.concat(parts, ignore_index=True)
            keep_in = sorted(set(D.G[np.isin(D.pid, trp)]) & set(r.Sample))
            for br, col in BRCOL.items():
                m = eg.pooled(eg.sample_table(r.assign(Repeat=0), col, keep_in))
                scores[(rep_name, br)] = (m["BA"], m["AUC"] if np.isfinite(m["AUC"]) else 0.0)
                inner_rows.append(dict(Fold=p, Representation=rep_name, Branch=br, InnerBA=m["BA"], InnerAUC=m["AUC"]))
        best = max(CANDS, key=lambda c: (round(scores[c][0], 10), round(scores[c][1], 10), -CANDS.index(c)))
        chosen[p] = dict(representation=best[0], branch=best[1], inner_BA=scores[best][0])
        X = ep.build_vectors(D.R, D.NX, Fout) if best[0] == "freq" else D.THETA
        rep = reps(D, X, yk, pts, design)
        for s in seeds:
            d = fit_eval(D, rep, yk, trp, lambda pv, gv: pv == p, s)
            if d is not None:
                d["Repeat"] = int(s)
                d["p_sel"] = d[BRCOL[best[1]]]
                recs.append(d)
    recs = pd.concat(recs, ignore_index=True)
    st = eg.sample_table(recs, "p_sel", keep)
    return dict(choice=chosen, inner=pd.DataFrame(inner_rows), records=recs, sample_table=st,
                metrics=eg.pooled(st), inner_method=inner_method)

Writing nested_core.py


In [27]:
%%writefile ijs_nested_sensitivity.py
"""
ijs_nested_sensitivity.py -- sensitivity of the fully nested pipeline selection.

V1 fallback="outer"   : the six inner folds without patient-level ranking reuse the outer-fold
                        ranking (favours the frequency candidate)
V2 fallback="generic" : these six inner folds use a fixed decade-spaced frequency set
V3 design="matched"   : size-matched sample-wise referencing in ranking, inner comparison and
                        outer evaluation
V4 P2 labels exchanged
V5 P2 excluded
All with 30 XGBoost seeds for the outer evaluation; patient-level cluster bootstrap CIs.
Rankings already computed by ijs_revision.py (same procedure, verified identical) are reused.
Output: outputs_ijs/revision/nested_sensitivity.json and sample tables.
"""
import json
import os
import time

import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import nested_core as nc
import rev_common as rc

OUT = "outputs_ijs/revision"
T0 = time.time()
tick = lambda m: print(f"[{time.time() - T0:6.0f}s] {m}", flush=True)
raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
D = nc.Data(raw, pd.read_csv("outputs_ijs/cole_fits.csv"))
P = [f"P{i}" for i in range(1, 7)]
SEEDS = list(range(ep.N_REPEATS))
RESF = f"{OUT}/nested_sensitivity.json"
RES = json.load(open(RESF)) if os.path.exists(RESF) else {}

# cache of the main analysis (sample-wise referencing, labels as given)
SEL = json.load(open("outputs_ijs/selections_idx.json"))["sa_nested"]
INNER = json.load(open(f"{OUT}/selections_inner.json"))
METHOD = json.load(open(f"{OUT}/revision_results.json"))["inner_selection_method"]
CACHE_MAIN = {("outer", "sa", p): sorted(SEL[p]) for p in P}
for k, v in INNER.items():
    p, q = k.split("|")
    CACHE_MAIN[(METHOD[k].lower(), "sa", p, q)] = v


def record(name, r, keep):
    ci, _, nb = eg.cluster_boot({name: r["sample_table"]})
    m = r["metrics"]
    RES[name] = dict(choice=r["choice"], inner_method=r["inner_method"], BA=m["BA"], TP=m["TP"], N1=m["N1"],
                     TN=m["TN"], N0=m["N0"], AUC=m["AUC"], CI=list(ci[name]),
                     n_cole=sum(v["representation"] == "cole" for v in r["choice"].values()),
                     n_abs=sum(v["branch"] == "abs" for v in r["choice"].values()))
    r["sample_table"].to_csv(f"{OUT}/sampletable_nested_{name}.csv", index=False)
    r["inner"].to_csv(f"{OUT}/nested_inner_{name}.csv", index=False)
    json.dump(json.loads(json.dumps(RES, default=float)), open(RESF, "w"), indent=1)
    tick(f"{name}: BA={m['BA']:.3f} ({m['TP']}/{m['N1']}, {m['TN']}/{m['N0']}) CI {ci[name]} choice "
         f"{ {p: (v['representation'], v['branch']) for p, v in r['choice'].items()} }")


y2 = D.y.copy()
y2[D.pid == "P2"] = 1 - y2[D.pid == "P2"]
P5 = [p for p in P if p != "P2"]
runs = [("fallback_outer", dict(yk=D.y, patients=P, fallback="outer", design="sa", cache=dict(CACHE_MAIN))),
        ("fallback_generic", dict(yk=D.y, patients=P, fallback="generic", design="sa", cache=dict(CACHE_MAIN))),
        ("matched", dict(yk=D.y, patients=P, fallback="loso", design="matched", cache={})),
        ("p2_exchanged", dict(yk=y2, patients=P, fallback="loso", design="sa", cache={})),
        ("p2_excluded", dict(yk=D.y, patients=P5, fallback="loso", design="sa", cache={}))]
for name, kw in runs:
    if name in RES:
        tick(f"{name}: done before")
        continue
    tick(f"{name} ...")
    keep = rc.evaluable(D.pid, kw["yk"], D.G, kw["patients"])
    r = nc.full_nested(D, kw["yk"], kw["patients"], fallback=kw["fallback"], design=kw["design"],
                       seeds=SEEDS, cache=kw["cache"], keep=keep)
    record(name, r, keep)
tick("done")

Writing ijs_nested_sensitivity.py


In [28]:
%%writefile ijs_selection_permutation.py
"""
ijs_selection_permutation.py -- selection-aware permutation test.

For every permutation of tissue-sample labels within patients, the COMPLETE fully nested
procedure of nested_core.full_nested is repeated: outer and inner frequency ranking (with the
leave-one-tissue-sample-out fallback), comparison of the six candidate pipelines in the inner
folds, and evaluation of the chosen pipeline on the held-out patient. The statistic is the
sample-level balanced accuracy of the selected pipeline (one XGBoost seed, as for the observed
value computed here as permutation 0). The pre-specified frequency-based classifier (nested
frequencies, equal fusion) is recorded from the same runs.
Usage: python ijs_selection_permutation.py [B=100]; resumes from its checkpoint.
"""
import json
import os
import sys
import time

import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import nested_core as nc
import rev_common as rc

OUT = "outputs_ijs/revision"
B = int(sys.argv[1]) if len(sys.argv) > 1 else 100
raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
D = nc.Data(raw, pd.read_csv("outputs_ijs/cole_fits.csv"))
P = [f"P{i}" for i in range(1, 7)]


def prespecified(D, yk, cache, keep):
    """Nested frequencies (outer-fold ranking from the cache), equal fusion, sample-wise referencing."""
    recs = []
    for p in P:
        trp = [q for q in P if q != p]
        rep = nc.reps(D, ep.build_vectors(D.R, D.NX, cache[("outer", "sa", p)]), yk, P, "sa")
        d = nc.fit_eval(D, rep, yk, trp, lambda pv, gv: pv == p, ep.SEED0)
        if d is not None:
            recs.append(d.assign(Repeat=0))
    return eg.pooled(eg.sample_table(pd.concat(recs, ignore_index=True), "p", keep))["BA"]


if __name__ == "__main__":
    t0 = time.time()
    rng = np.random.default_rng(ep.SEED0 + 11)
    labels = [D.y.copy()] + [rc.permute_within_patient(D.y, D.pid, D.G, rng) for _ in range(B)]
    part = f"{OUT}/selection_permutation_partial.csv"
    done = set(pd.read_csv(part).perm) if os.path.exists(part) else set()
    print(f"{len(done)} done, {B + 1 - len(done)} to go", flush=True)
    for k, yk in enumerate(labels):
        if k in done:
            continue
        cache = {}
        keep = rc.evaluable(D.pid, yk, D.G, P)
        r = nc.full_nested(D, yk, P, seeds=[ep.SEED0], cache=cache, keep=keep)
        row = dict(perm=k, BA_selected=r["metrics"]["BA"], BA_prespecified=prespecified(D, yk, cache, keep),
                   n_cole=sum(v["representation"] == "cole" for v in r["choice"].values()),
                   n_abs=sum(v["branch"] == "abs" for v in r["choice"].values()),
                   n_mah=sum(v["branch"] == "mah" for v in r["choice"].values()))
        pd.DataFrame([row]).to_csv(part, mode="a", header=not os.path.exists(part), index=False)
        print(f"[{time.time() - t0:7.0f}s] perm {k}: {row}", flush=True)
    df = pd.read_csv(part).drop_duplicates("perm").sort_values("perm").reset_index(drop=True)
    df.to_csv(f"{OUT}/selection_permutation_null.csv", index=False)
    obs, null = df[df.perm == 0].iloc[0], df[df.perm > 0]
    res = dict(n_perm=int(len(null)))
    for c in ("BA_selected", "BA_prespecified"):
        res[c] = dict(observed=float(obs[c]), null_mean=float(null[c].mean()), null_sd=float(null[c].std(ddof=1)),
                      null_q025=float(null[c].quantile(.025)), null_q975=float(null[c].quantile(.975)),
                      null_max=float(null[c].max()),
                      p_value=float((1 + (null[c] >= obs[c] - 1e-12).sum()) / (1 + len(null))))
    res["null_choice"] = dict(mean_folds_cole=float(null.n_cole.mean()), mean_folds_abs=float(null.n_abs.mean()),
                              frac_all_cole_abs=float(((null.n_cole == 6) & (null.n_abs == 6)).mean()))
    json.dump(res, open(f"{OUT}/selection_permutation_results.json", "w"), indent=1)
    print(json.dumps(res, indent=1))

Writing ijs_selection_permutation.py


In [29]:
%%writefile ijs_reviewer_extras.py
"""
ijs_reviewer_extras.py -- additional robustness analyses requested in review.

E1 per-patient confusion matrices and leave-one-patient-out influence (pooled BA recomputed
   with each patient's samples removed) for the main configurations
E2 stability of frequency selection (band shares, selection counts, pairwise Jaccard index)
E3 Cole identifiability: absolute Cole branch re-evaluated without the spectra whose
   characteristic frequency lies outside the fitted window
E4 stability of predictions across the 30 XGBoost seeds (per-seed BA, samples changing class)
E5 design effect and effective sample size for every Cole parameter and class, with the
   Kish correction for unequal numbers of spectra per sample
Output: outputs_ijs/revision/reviewer_extras.json and CSV tables.
"""
import json
from itertools import combinations

import numpy as np
import pandas as pd

import eis_pipeline as ep
import eis_groups as eg
import rev_common as rc
from cole_analysis import FMIN, FMAX

OUT = "outputs_ijs"
REV = f"{OUT}/revision"
raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
GRID = ep.candidate_grid()
R, NX, pid, y, _ = ep.harmonize(raw, GRID)
G = eg.sample_ids(raw)
P = [f"P{i}" for i in range(1, 7)]
fits = pd.read_csv(f"{OUT}/cole_fits.csv")
THETA = np.column_stack([np.log(fits.R0), np.log(fits.Rinf), np.log10(fits.fc), fits.alpha])
SEL = json.load(open(f"{OUT}/selections_idx.json"))
EVAL = rc.evaluable(pid, y, G, P)
X = {}

# ------------------------------------------------------------------ E1 per patient and influence
tables = {"Spectrum-wise, nested frequencies": f"{OUT}/sampletable_naive_nested_fusion.csv",
          "Sample-wise, nested frequencies": f"{OUT}/sampletable_sa_nested_fusion.csv",
          "Sample-wise, published set": f"{OUT}/sampletable_sa_published_fusion.csv",
          "Fully nested pipeline": f"{REV}/sampletable_nested_pipeline_selected.csv"}
rows, infl = [], []
for name, path in tables.items():
    st = pd.read_csv(path)
    st["pred"] = (st.p >= 0.5).astype(int)
    for p in P:
        s = st[st.Patient == p]
        tp, fn = int(((s.y == 1) & (s.pred == 1)).sum()), int(((s.y == 1) & (s.pred == 0)).sum())
        tn, fp = int(((s.y == 0) & (s.pred == 0)).sum()), int(((s.y == 0) & (s.pred == 1)).sum())
        rows.append(dict(Config=name, Patient=p, TP=tp, FN=fn, TN=tn, FP=fp,
                         BA=(tp / (tp + fn) + tn / (tn + fp)) / 2 if (tp + fn) and (tn + fp) else np.nan))
        rest = st[st.Patient != p]
        infl.append(dict(Config=name, Removed=p, BA=eg.pooled(rest)["BA"]))
per_patient = pd.DataFrame(rows)
influence = pd.DataFrame(infl)
per_patient.to_csv(f"{REV}/per_patient_confusion.csv", index=False)
influence.to_csv(f"{REV}/leave_one_patient_out_influence.csv", index=False)
X["influence"] = {c: dict(min=float(g.BA.min()), max=float(g.BA.max()),
                          min_patient=g.loc[g.BA.idxmin(), "Removed"], max_patient=g.loc[g.BA.idxmax(), "Removed"])
                  for c, g in influence.groupby("Config")}

# ------------------------------------------------------------------ E2 frequency stability
inner = json.load(open(f"{REV}/selections_inner.json"))
sets = {"sample-wise outer": list(SEL["sa_nested"].values()), "sample-wise inner": list(inner.values()),
        "spectrum-wise outer": list(SEL["naive_nested"].values())}
stab = {}
for k, ss in sets.items():
    f = np.concatenate([GRID[s] for s in ss]) / 1e3
    jac = [len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in combinations(ss, 2)]
    counts = pd.Series(np.concatenate(ss)).value_counts()
    stab[k] = dict(n_sets=len(ss), share_below_10k=float((f < 10).mean()), share_10_100k=float(((f >= 10) & (f < 100)).mean()),
                   share_above_100k=float((f >= 100).mean()), mean_jaccard=float(np.mean(jac)),
                   n_distinct=int(counts.size), max_count=int(counts.max()),
                   most_frequent_khz=float(GRID[counts.index[0]] / 1e3))
X["frequency_stability"] = stab

# ------------------------------------------------------------------ E3 Cole identifiability
inwin = ((fits.fc >= FMIN) & (fits.fc <= FMAX)).to_numpy()
rows_ = np.flatnonzero(inwin)
X["cole_out_of_window"] = dict(n=int((~inwin).sum()),
                               per_sample={g: int(n) for g, n in pd.Series(G[~inwin]).value_counts().items()})
p_, y_, G_, T_ = pid[rows_], y[rows_], G[rows_], THETA[rows_]
A, M, ok = rc.fast_rep(T_, p_, y_, G_, P)
keep_ = rc.evaluable(p_, y_, G_, P)
r = rc.run_eval(A, M, ok, p_, y_, G_, P, seeds=range(ep.N_REPEATS))
m = eg.pooled(eg.sample_table(r, "p_abs", keep_))
st_main = pd.read_csv(f"{OUT}/sampletable_sa_cole_abs.csv")
mis = st_main[(st_main.p >= 0.5).astype(int) != st_main.y].Sample.tolist()
X["cole_in_window_only"] = dict(BA=m["BA"], TP=m["TP"], N1=m["N1"], TN=m["TN"], N0=m["N0"], n_samples=len(keep_),
                                misclassified_main=mis,
                                misclassified_main_out_of_window=int(sum(X["cole_out_of_window"]["per_sample"].get(s, 0) for s in mis)))

# ------------------------------------------------------------------ E4 seed stability
def per_seed(recs, col):
    bas, preds = [], []
    for s, g in recs.groupby("Repeat"):
        st = eg.sample_table(g, col, EVAL)
        bas.append(eg.pooled(st)["BA"])
        preds.append(st.set_index("Sample").p.ge(0.5).astype(int))
    pr = pd.concat(preds, axis=1)
    return dict(BA_min=float(min(bas)), BA_max=float(max(bas)), BA_sd=float(np.std(bas, ddof=1)),
                n_samples_changing_class=int((pr.nunique(axis=1) > 1).sum()))

A, M, ok = rc.fast_rep(THETA, pid, y, G, P)
rc_cole = rc.run_eval(A, M, ok, pid, y, G, P, seeds=range(ep.N_REPEATS))
rf = eg.evaluate_g(lambda p: ep.build_vectors(R, NX, SEL["sa_nested"][p]), pid, y, G, P, G)
X["seed_stability"] = {"Cole absolute (fully nested choice)": per_seed(rc_cole, "p_abs"),
                       "Nested frequencies, equal fusion": per_seed(rf, "p")}

# ------------------------------------------------------------------ E5 design effect
icc = json.load(open(f"{OUT}/ijs_results.json"))["icc"]
m_s = pd.Series(G).value_counts()
mbar, cv2 = float(m_s.mean()), float(m_s.var(ddof=0) / m_s.mean() ** 2)
de = {}
for k, v in icc.items():
    d0 = 1 + (mbar - 1) * v
    d1 = 1 + ((1 + cv2) * mbar - 1) * v
    de[k] = dict(ICC=v, DE=d0, N_eff=len(y) / d0, DE_kish=d1, N_eff_kish=len(y) / d1)
X["design_effect"] = dict(m_bar=mbar, cv2=cv2, per_parameter=de,
                          N_eff_min=min(v["N_eff"] for v in de.values()), N_eff_max=max(v["N_eff"] for v in de.values()),
                          N_eff_kish_min=min(v["N_eff_kish"] for v in de.values()),
                          N_eff_kish_max=max(v["N_eff_kish"] for v in de.values()))

# ------------------------------------------------------------------ E6 published set without P2
P5 = [p for p in P if p != "P2"]
XPUB = ep.build_vectors(*ep.harmonize(raw, np.array([3.55e3, 568.99e3, 655.13e3, 754.31e3, 868.51e3]))[:2], [0, 1, 2, 3, 4])
rp = eg.evaluate_g(lambda p: XPUB, pid, y, G, P5, G)
mp = eg.pooled(eg.sample_table(rp, "p", rc.evaluable(pid, y, G, P5)))
X["published_without_P2"] = dict(BA=mp["BA"], TP=mp["TP"], N1=mp["N1"], TN=mp["TN"], N0=mp["N0"])

json.dump(json.loads(json.dumps(X, default=float)), open(f"{REV}/reviewer_extras.json", "w"), indent=1)
print(json.dumps(json.loads(json.dumps(X, default=float)), indent=1))
print(per_patient.to_string())

Writing ijs_reviewer_extras.py


In [30]:
%%writefile ijs_simulation_ext.py
"""
ijs_simulation_ext.py -- robustness of the simulation to departures from the base model.

Scenarios (logistic regression, six patients with three healthy and three tumor samples,
m = 15 spectra per sample, ICC in {0.5, 0.9, 0.97}, Delta in {0, 4}, 30 replicates):
  base           the generator of ijs_simulation.py
  correlated     between-sample and within-sample deviations correlated across the four
                 features (AR(1) correlation 0.7), marginal variances unchanged
  heteroscedastic within-sample noise scaled per tissue sample by a log-normal factor
                 (log-SD 0.75, unit mean square), so that repeatability differs between samples
  heterogeneous  patient-specific tumor effects: delta_p = u_p * delta + N(0, I), u_p ~ U(0, 2)
Output: outputs_ijs/revision/sim_ext.csv and sim_ext_summary.json
"""
import json
import sys
from multiprocessing import Pool

import numpy as np
import pandas as pd

import ijs_simulation as S

L = np.linalg.cholesky(np.array([[0.7 ** abs(i - j) for j in range(S.D)] for i in range(S.D)]))
SCEN = {"name": "base"}


def simulate_scen(rng, n_pat, n_h, n_t, m, icc, delta, offset=0):
    sc = SCEN["name"]
    dvec = np.full(S.D, delta / np.sqrt(S.D))
    X, pid, y, G = [], [], [], []
    for p in range(n_pat):
        mu = rng.normal(0, 2.0, S.D)
        if sc == "heterogeneous":
            dp = rng.uniform(0, 2) * dvec + rng.normal(0, 1.0, S.D)
        else:
            dp = dvec + rng.normal(0, 0.5, S.D)
        for c, ns in ((0, n_h), (1, n_t)):
            for s in range(ns):
                b = rng.normal(0, 1.0, S.D)
                e = rng.normal(0, np.sqrt((1 - icc) / icc), (m, S.D))
                if sc == "correlated":
                    b, e = L @ b, e @ L.T
                if sc == "heteroscedastic":
                    e = e * np.exp(0.75 * rng.normal() - 0.75 ** 2)
                X.append(mu + c * dp + b + e)
                pid += [f"P{p + offset}"] * m
                y += [c] * m
                G += [f"P{p + offset}:{c}{s}"] * m
    return np.vstack(X), np.asarray(pid), np.asarray(y), np.asarray(G)


def replicate(args):
    cell, rep, kind, sc = args
    SCEN["name"] = sc
    S.simulate = simulate_scen
    out = S.replicate((cell, rep, kind))
    out["scenario"] = sc
    return out


if __name__ == "__main__":
    jobs = [((6, 3, 3, 15, icc, d), r, "logit", sc)
            for sc in ("base", "correlated", "heteroscedastic", "heterogeneous")
            for icc in (0.5, 0.9, 0.97) for d in (0.0, 4.0) for r in range(30)]
    procs = int(sys.argv[1]) if len(sys.argv) > 1 else 2
    with Pool(procs) as pool:
        rows = pool.map(replicate, jobs, chunksize=4)
    df = pd.DataFrame(rows)
    df.to_csv(f"{S.OUT}/sim_ext.csv", index=False)
    for dn in ("naive", "sa", "matched"):
        df[f"{dn}_opt"] = df[f"{dn}_est"] - df[f"{dn}_true"]
    cell = df.groupby(["scenario", "delta", "icc"])[["naive_opt", "sa_opt", "matched_opt", "naive_est", "matched_true"]].mean()
    cell.reset_index().to_csv(f"{S.OUT}/sim_ext_cell_means.csv", index=False)
    summ = {}
    for sc, g in cell.groupby(level="scenario"):
        summ[sc] = dict(naive_opt_min=float(g.naive_opt.min()), naive_opt_max=float(g.naive_opt.max()),
                        matched_opt_min=float(g.matched_opt.min()), matched_opt_max=float(g.matched_opt.max()),
                        sa_opt_min=float(g.sa_opt.min()), sa_opt_max=float(g.sa_opt.max()),
                        naive_est_null_icc097=float(g.loc[(sc, 0.0, 0.97), "naive_est"]))
    json.dump(summ, open(f"{S.OUT}/sim_ext_summary.json", "w"), indent=1)
    print(cell.round(3).to_string())
    print(json.dumps(summ, indent=1))

Writing ijs_simulation_ext.py


In [31]:
%%writefile make_supp_tables.py
"""Tables S1-S9 of Online Resource 1, written as LaTeX (compact layout for the 25-page limit)."""
import json
import os
import pandas as pd

REV = os.environ.get("REV_DIR", "outputs_ijs/revision")
OUTF = os.environ.get("SUPP_OUT", "outputs_ijs/supp_tables.tex")
L = []
R = json.load(open(f"{REV}/revision_results.json"))
S, CI = R["summary"], R["ci"]
RP = json.load(open(f"{REV}/revision_results_v2_outer_ranking.json"))
PATS = ["P1", "P2", "P3", "P4", "P5", "P6"]


def h3(v):
    """Three decimals, rounding halves up (as in the printed article); tiny float error is removed first."""
    from decimal import Decimal, ROUND_HALF_UP
    return str(Decimal(f"{float(v):.10f}").quantize(Decimal("0.001"), rounding=ROUND_HALF_UP))


def s3(v):
    """Signed three-decimal value, halves rounded away from zero."""
    t = h3(abs(v))
    if t == "0.000":
        return t
    return ("-" if float(v) < 0 else "+") + t


def begin(caption, spec, header, size=r"\footnotesize", sep="4pt"):
    L.append(r"\begin{table}[H]" "\n" r"\centering" "\n" r"\caption{" + caption + "}\n" + size + "\n"
             r"\setlength{\tabcolsep}{" + sep + "}\n" r"\begin{tabular}{" + spec + "}\n" r"\toprule" "\n" + header + "\n" r"\midrule")


def end():
    L.append(r"\bottomrule" "\n" r"\end{tabular}" "\n" r"\end{table}" "\n")


# ---------------------------------------------------------------- S1 ICC and design effect
X = json.load(open(f"{REV}/reviewer_extras.json"))["design_effect"]
pn = {"ln R0": r"$\ln R_0$", "ln Rinf": r"$\ln R_{\infty}$", "log10 fc": r"$\log_{10} f_c$", "alpha": r"$\alpha$"}
begin(r"Intraclass correlation (ICC) of Cole parameters between tissue samples (after removal of patient means), design effect (DE), "
      r"and effective number of independent observations among the 767 spectra, without and with the Kish correction "
      r"(mean " + f"{X['m_bar']:.1f}" + r" spectra per sample, squared coefficient of variation " + f"{X['cv2']:.2f}" + r"). "
      r"ICC and DE are class-specific, but $N_{\mathrm{eff}}=767/\mathrm{DE}$ applies each DE to all 767 spectra: it is the effective size "
      r"of the whole dataset if every spectrum had that ICC, not the number of independent observations per class. The figures describe the "
      r"redundancy of repeats for one parameter at a time; for generalization, the six patients remain the top level of the hierarchy.",
      "llccccc", r"Class & Parameter & ICC & DE & $N_{\mathrm{eff}}$ & DE (Kish) & $N_{\mathrm{eff}}$ (Kish) \\")
for k, v in X["per_parameter"].items():
    cl, par = k.split("|")
    L.append(f"{cl.capitalize()} & {pn[par]} & {h3(v['ICC'])} & {v['DE']:.1f} & {v['N_eff']:.1f} & {v['DE_kish']:.1f} & {v['N_eff_kish']:.1f} \\\\")
end()

# ---------------------------------------------------------------- S2 per patient and influence
pp = pd.read_csv(f"{REV}/per_patient_confusion.csv")
inf = pd.read_csv(f"{REV}/leave_one_patient_out_influence.csv")
begin(r"Per-patient outcomes and influence. Left: correctly classified tumor (T) and healthy (H) samples of the pre-specified "
      r"classifier (nested frequencies, equal fusion, sample-wise referencing) and of the fully nested pipeline; healthy samples "
      r"of P2, P3 and P6 are not evaluable under sample-wise referencing. Right: pooled sample-level balanced accuracy recomputed "
      r"with the samples of the listed patient removed. Patient-level BA: sensitivity and specificity computed within each patient and "
      r"averaged over patients (specificity over P1, P4, and P5).",
      "lcccccccc",
      r" & \multicolumn{2}{c}{Pre-specified} & \multicolumn{2}{c}{Fully nested} & \multicolumn{4}{c}{BA without the patient} \\" "\n"
      r"Patient & T & H & T & H & \shortstack{Spectrum-wise,\\nested} & \shortstack{Pre-\\specified} & \shortstack{Published\\set} & \shortstack{Fully\\nested} \\",
      sep="3pt")


def th(cfg, p):
    r = pp[(pp.Config == cfg) & (pp.Patient == p)].iloc[0]
    t = f"{r.TP}/{r.TP + r.FN}" if r.TP + r.FN else "--"
    h = f"{r.TN}/{r.TN + r.FP}" if r.TN + r.FP else "--"
    return t, h


for p in PATS:
    a = th("Sample-wise, nested frequencies", p)
    b = th("Fully nested pipeline", p)
    vals = [inf[(inf.Config == c) & (inf.Removed == p)].BA.iloc[0] for c in
            ("Spectrum-wise, nested frequencies", "Sample-wise, nested frequencies", "Sample-wise, published set", "Fully nested pipeline")]
    L.append(f"{p} & {a[0]} & {a[1]} & {b[0]} & {b[1]} & " + " & ".join(f"{h3(v)}" for v in vals) + r" \\")


def patient_ba(cfg):
    d = pp[pp.Config == cfg]
    se = (d.TP / (d.TP + d.FN)).dropna().mean()
    sp = (d.TN / (d.TN + d.FP)).dropna().mean()
    return 0.5 * (se + sp)


L.append(r"\midrule")
L.append(r"\multicolumn{9}{l}{Patient-level BA: pre-specified " + h3(patient_ba("Sample-wise, nested frequencies")) +
         ", fully nested " + h3(patient_ba("Fully nested pipeline")) + r"} \\")
end()

# ---------------------------------------------------------------- S3 permutation null of fixed configurations
PR = json.load(open(f"{REV}/permutation_results.json"))
begin(r"Permutation null of the sample-level balanced accuracy for fixed configurations (" + f"{PR['n_perm']}" +
      r" within-patient permutations of tissue-sample labels). Null: mean and 2.5th--97.5th percentiles; $p$: permutation "
      r"$p$-value of the observed balanced accuracy.",
      "lllcccc", r"Features & Output & Referencing & Null mean & Null 95\% range & Observed & $p$ \\", sep="3.5pt")
fn = {"cole": "Cole", "published": "Published set"}
on = {"fusion": "Equal fusion", "abs": "Absolute", "mah": "Mahalanobis"}
for f in ("cole", "published"):
    for o in ("fusion", "abs", "mah"):
        for i, ref in enumerate(("naive", "sa")):
            v = PR[f"{f}|{ref}|{o}"]
            L.append(f"{fn[f] if (o == 'fusion' and i == 0) else ''} & {on[o] if i == 0 else ''} & "
                     f"{'Spectrum-wise' if ref == 'naive' else 'Sample-wise'} & {h3(v['null_mean'])} & "
                     f"{h3(v['null_q025'])}--{h3(v['null_q975'])} & {h3(v['observed'])} & {h3(v['p_value'])} \\\\")
end()

# ---------------------------------------------------------------- S4 nested pipeline inner scores
inner = pd.read_csv(f"{REV}/nested_pipeline_inner_scores.csv")
choice = R["nested_pipeline_choice"]
lab = {("freq", "fusion"): "Freq.\\ fusion", ("freq", "abs"): "Freq.\\ abs.", ("freq", "mah"): "Freq.\\ Mah.",
       ("cole", "fusion"): "Cole fusion", ("cole", "abs"): "Cole abs.", ("cole", "mah"): "Cole Mah."}
cols = list(lab)
begin(r"Fully nested pipeline selection (Algorithm~S1): inner leave-one-patient-out sample-level balanced accuracy of the six candidate "
      r"pipelines within each outer fold (five training patients, sample-wise referencing; frequencies re-ranked in every inner fold). "
      r"Freq.: five selected frequencies; Cole: Cole parameters; abs.: absolute-deviation branch; Mah.: Mahalanobis branch; fusion: equal "
      r"fusion of both. The candidate with the highest inner BA (in bold) was refitted and evaluated once on the held-out patient.",
      "l" + "c" * len(cols), "Held-out patient & " + " & ".join(lab[c] for c in cols) + r" \\")
for p in sorted(inner.Fold.unique()):
    d = inner[inner.Fold == p].set_index(["Representation", "Branch"]).InnerBA
    ch = (choice[p]["representation"], choice[p]["branch"])
    cells = [(r"\textbf{%s}" if c == ch else "%s") % h3(d[c]) for c in cols]
    L.append(f"{p} & " + " & ".join(cells) + r" \\")
end()

# ---------------------------------------------------------------- S5 fully nested selection: variants
NS = json.load(open(f"{REV}/nested_sensitivity.json"))
ab = {("cole", "abs"): "C-a", ("cole", "fusion"): "C-f", ("cole", "mah"): "C-m",
      ("freq", "abs"): "F-a", ("freq", "fusion"): "F-f", ("freq", "mah"): "F-m"}
main_ = dict(choice=R["nested_pipeline_choice"], **{k: S["nested_pipeline|selected"][k] for k in ("BA", "TP", "N1", "TN", "N0")},
             CI=CI["nested_pipeline|selected"])
part_ = dict(choice=RP["nested_pipeline_choice"], **{k: RP["summary"]["nested_pipeline|selected"][k] for k in ("BA", "TP", "N1", "TN", "N0")},
             CI=RP["ci"]["nested_pipeline|selected"])
variants = [("Primary (fallback: sample ranking)", main_), ("Fallback: fixed set", NS["fallback_generic"]),
            ("Fallback: outer ranking", NS["fallback_outer"]), ("Outer ranking, all inner folds", part_),
            ("Size-matched referencing", NS["matched"]), ("P2 excluded", NS["p2_excluded"]), ("P2 labels exchanged", NS["p2_exchanged"])]
begin(r"Fully nested pipeline selection under alternative treatments: pipeline chosen in each outer fold (C: Cole parameters, "
      r"F: frequencies; a: absolute branch, f: equal fusion, m: Mahalanobis branch), correctly classified tumor and healthy samples, "
      r"and sample-level balanced accuracy with patient-level cluster bootstrap 95\% CI. Fallback rows differ only in the six inner folds "
      r"in which patient-level frequency ranking was impossible (primary: leave-one-sample-out ranking within the inner-training patients; "
      r"fixed set: decade-spaced frequencies; outer ranking: frequencies of the outer fold, a deliberately leaky diagnostic).",
      "lccccccccc", r"Variant & P1 & P2 & P3 & P4 & P5 & P6 & Tumor & Healthy & BA (95\% CI) \\", sep="3pt")
for name, v in variants:
    ch = [ab[(v["choice"][p]["representation"], v["choice"][p]["branch"])] if p in v["choice"] else "--" for p in PATS]
    L.append(f"{name} & " + " & ".join(ch) + f" & {v['TP']}/{v['N1']} & {v['TN']}/{v['N0']} & {h3(v['BA'])} ({h3(v['CI'][0])}--{h3(v['CI'][1])}) \\\\")
end()

# ---------------------------------------------------------------- S6 additional real-data analyses
S["partial_pipeline|selected"] = RP["summary"]["nested_pipeline|selected"]
CI["partial_pipeline|selected"] = RP["ci"]["nested_pipeline|selected"]
rows = [("Pipeline selection", "nested_pipeline|selected", "Fully nested"),
        ("", "partial_pipeline|selected", "Outer-fold ranking reused"),
        ("Size-matched referencing", "matched_nested|fusion", "Nested frequencies, fusion"),
        ("", "matched_nested|abs", "Nested frequencies, absolute"),
        ("", "matched_nested|mah", "Nested frequencies, Mahalanobis"),
        ("", "matched_cole|fusion", "Cole, fusion"), ("", "matched_cole|abs", "Cole, absolute"),
        ("", "matched_cole|mah", "Cole, Mahalanobis"), ("", "matched_published|fusion", "Published set, fusion"),
        ("P2 labels exchanged", "p2_naive_leaky|fusion", "Spectrum-wise, leaky"),
        ("", "p2_naive_nested|fusion", "Spectrum-wise, nested"), ("", "p2_sa_leaky|fusion", "Sample-wise, leaky"),
        ("", "p2_sa_nested|fusion", "Sample-wise, nested"), ("", "p2_sa_cole|abs", "Sample-wise, Cole absolute"),
        ("", "p2_sa_published|fusion", "Sample-wise, published set"),
        ("Logistic regression", "lr_naive_leaky|fusion", "Spectrum-wise, leaky"),
        ("", "lr_naive_nested|fusion", "Spectrum-wise, nested"), ("", "lr_sa_leaky|fusion", "Sample-wise, leaky"),
        ("", "lr_sa_nested|fusion", "Sample-wise, nested"), ("", "lr_sa_nested|abs", "Sample-wise, nested, absolute"),
        ("", "lr_sa_nested|mah", "Sample-wise, nested, Mahalanobis"),
        ("", "lr_naive_cole|fusion", "Spectrum-wise, Cole fusion"), ("", "lr_naive_cole|abs", "Spectrum-wise, Cole absolute"),
        ("", "lr_sa_cole|fusion", "Sample-wise, Cole fusion"), ("", "lr_sa_cole|abs", "Sample-wise, Cole absolute"),
        ("", "lr_sa_cole|mah", "Sample-wise, Cole Mahalanobis"), ("", "lr_sa_published|fusion", "Sample-wise, published set")]
begin(r"Additional real-data analyses (sample level). Except for the P2 label exchange (15 tumor, 12 healthy samples), 16 tumor "
      r"and 10 healthy samples were evaluated. Spectrum-wise/sample-wise: referencing design; leaky/nested: frequency selection with or "
      r"without the test patient; fusion: equal fusion of the absolute and Mahalanobis branches. CI: patient-level cluster bootstrap 95\% interval.",
      "llccccc", r"Analysis & Configuration & Tumor & Healthy & BA & 95\% CI & AUC \\", size=r"\scriptsize", sep="3.5pt")
for a, k, c in rows:
    s = S[k]
    ci = CI.get(k)
    cis = f"{h3(ci[0])}--{h3(ci[1])}" if ci else "--"
    L.append(f"{a} & {c} & {s['TP']}/{s['N1']} & {s['TN']}/{s['N0']} & {h3(s['BA'])} & {cis} & {h3(s['AUC'])} \\\\")
end()

# ---------------------------------------------------------------- S7 simulation grid (logistic regression)
g = pd.read_csv(f"{REV}/sim_grid.csv")
for dn in ("naive", "sa", "matched"):
    g[f"{dn}_opt"] = g[f"{dn}_est"] - g[f"{dn}_true"]
c = g.groupby(["delta", "icc", "m"]).mean(numeric_only=True).reset_index()
begin(r"Simulation study with logistic regression (six patients with three healthy and three tumor samples, 30 replicates per "
      r"condition): mean leave-one-patient-out estimate of the sample-level balanced accuracy for spectrum-wise referencing (SW est.), "
      r"true deployment balanced accuracy of the size-matched pipeline (True, for reference only), and optimism for spectrum-wise (SW), "
      r"sample-wise (SA), and size-matched (SM) referencing. Each optimism is the estimate of a design minus the true deployment balanced "
      r"accuracy of the pipeline trained with that same design, not minus the True column. $\Delta=0$: no class difference.",
      "cc|cccc|ccccc",
      r" & & \multicolumn{4}{c|}{$\Delta=0$} & \multicolumn{5}{c}{$\Delta=4$} \\" "\n"
      r"ICC & $m$ & SW est. & SW opt. & SA opt. & SM opt. & SW est. & True & SW opt. & SA opt. & SM opt. \\",
      size=r"\scriptsize", sep="3.5pt")
for (icc, m), d in c.groupby(["icc", "m"]):
    r0 = d[d.delta == 0].iloc[0]
    r4 = d[d.delta == 4].iloc[0]
    L.append(f"{icc:g} & {int(m)} & {h3(r0.naive_est)} & {s3(r0.naive_opt)} & {s3(r0.sa_opt)} & {s3(r0.matched_opt)} & "
             f"{h3(r4.naive_est)} & {h3(r4.matched_true)} & {s3(r4.naive_opt)} & {s3(r4.sa_opt)} & {s3(r4.matched_opt)} \\\\")
end()

# ---------------------------------------------------------------- S8 simulation with XGBoost
x = pd.read_csv(f"{REV}/sim_xgb.csv")
for dn in ("naive", "sa", "matched"):
    x[f"{dn}_opt"] = x[f"{dn}_est"] - x[f"{dn}_true"]
c = x.groupby(["delta", "icc"]).mean(numeric_only=True).reset_index()
begin(r"Simulation study with XGBoost ($m=15$, 20 replicates per condition); SM est.: size-matched estimate; other abbreviations as in Table~S7.",
      "ccccccc", r"$\Delta$ & ICC & SW est. & SM est. & SW opt. & SA opt. & SM opt. \\", size=r"\scriptsize", sep="3pt")
for _, r in c.iterrows():
    L.append(f"{r.delta:g} & {r.icc:g} & {h3(r.naive_est)} & {h3(r.matched_est)} & {s3(r.naive_opt)} & {s3(r.sa_opt)} & {s3(r.matched_opt)} \\\\")
end()

# ---------------------------------------------------------------- S9 simulation scenarios
if os.path.exists(f"{REV}/sim_ext_cell_means.csv"):
    ce = pd.read_csv(f"{REV}/sim_ext_cell_means.csv")
    scn = {"base": "Base model", "correlated": "Correlated features", "heteroscedastic": "Heteroscedastic repeats",
           "heterogeneous": "Heterogeneous tumor effects"}
    begin(r"Simulation under departures from the base model (logistic regression, $m=15$, 30 replicates per condition): mean "
          r"optimism of spectrum-wise (SW), sample-wise (SA), and size-matched (SM) referencing.",
          "lc|ccc|ccc",
          r" & & \multicolumn{3}{c|}{$\Delta=0$} & \multicolumn{3}{c}{$\Delta=4$} \\" "\n"
          r"Scenario & ICC & SW & SA & SM & SW & SA & SM \\", size=r"\scriptsize", sep="3pt")
    for sc in ("base", "correlated", "heteroscedastic", "heterogeneous"):
        d = ce[ce.scenario == sc]
        for i, icc in enumerate(sorted(d.icc.unique())):
            r0 = d[(d.icc == icc) & (d.delta == 0)].iloc[0]
            r4 = d[(d.icc == icc) & (d.delta == 4)].iloc[0]
            L.append(f"{scn[sc] if i == 0 else ''} & {icc:g} & {s3(r0.naive_opt)} & {s3(r0.sa_opt)} & {s3(r0.matched_opt)} & "
                     f"{s3(r4.naive_opt)} & {s3(r4.sa_opt)} & {s3(r4.matched_opt)} \\\\")
    end()

# ---------------------------------------------------------------- compact layout: pair narrow tables side by side
text = "\n".join(L)
blocks = text.split(r"\begin{table}[H]")
head, blocks = blocks[0], [r"\begin{table}[H]" + b for b in blocks[1:]]


def as_minipage(b, w):
    b = b.replace(r"\begin{table}[H]", r"\begin{minipage}[t]{%s\textwidth}" % w, 1)
    return b.replace(r"\end{table}", r"\end{minipage}", 1).rstrip()


PAIRS = {(7, 8): (0.40, 0.56)}          # Tables S8 and S9 (0-based block indices)
out_blocks, skip = [], set()
for i, b in enumerate(blocks):
    if i in skip:
        continue
    pair = next((k for k in PAIRS if k[0] == i), None)
    if pair:
        w1, w2 = PAIRS[pair]
        out_blocks.append(r"\begin{table}[H]" "\n" + as_minipage(b, w1) + "\n" r"\hfill" "\n" +
                          as_minipage(blocks[pair[1]], w2) + "\n" r"\end{table}" "\n")
        skip.add(pair[1])
    else:
        out_blocks.append(b)
open(OUTF, "w").write(head + "".join(out_blocks))
print("written", OUTF)


In [ ]:
%%writefile make_fig3.py
"""Figure 3 (selected frequencies and between-sample ICC), compact side-by-side layout."""
import json, os
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import ijs_figures as fg
OUT = "outputs_ijs"; FIG = "outputs_ijs/figs"; os.makedirs(FIG, exist_ok=True)
r = json.load(open(f"{OUT}/ijs_results.json"))
sk = r["selected_khz"]; rows = []
for p in ["P1", "P2", "P3", "P4", "P5", "P6"]:
    rows.append((f"SW nested, {p} out", sk["naive_nested"][p], fg.C_NAIVE, "a"))
rows.append(("SW leaky", sk["naive_leaky"], "0.25", "b"))
for p in ["P1", "P2", "P3", "P4", "P5", "P6"]:
    rows.append((f"SA nested, {p} out", sk["sa_nested"][p], fg.C_SA, "c"))
rows.append(("SA leaky", sk["sa_leaky"], "#9A4A10", "d"))
rows.append(("Published set", [3.55, 568.99, 655.13, 754.31, 868.51], "black", "e"))
ic = pd.read_csv(f"{OUT}/frequency_icc.csv")
fg.frequencies_icc(rows, ic.f_kHz.values, ic.ICC.values).savefig(f"{FIG}/Fig3_Selected_Frequencies.png", bbox_inches="tight")


In [ ]:
%%writefile make_fig12.py
"""Figures 1 and 2 (enlarged fonts; Fig. 1 includes size-matched referencing)."""
import os
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import eis_pipeline as ep, eis_groups as eg, ijs_figures as fg
from cole_analysis import iterate_spectra
OUT = "outputs_ijs"; FIG = "outputs_ijs/figs"; os.makedirs(FIG, exist_ok=True)
fg.schematic().savefig(f"{FIG}/Fig1_Referencing_Designs.png", bbox_inches="tight")
raw = ep.load_raw_dataset("electrical-impedance-data-bladder/dataset_without_modules.pickle")
G = eg.sample_ids(raw)
specs = list(iterate_spectra(raw))
p1 = [(lab, g, f, z) for (p, lab, f, z), g in zip(specs, G) if p == "P1"]
dist = pd.read_csv(f"{OUT}/mahalanobis_within_between.csv")
fg.sample_structure(p1, dist).savefig(f"{FIG}/Fig2_Sample_Structure.png", bbox_inches="tight")


In [ ]:
%%writefile article_tables.py
"""article_tables.py -- Tables 1-4 of the article, rebuilt from the analysis outputs.

Usage (inside the notebook or from the command line):
    python article_tables.py [outputs_ijs]
Writes outputs_ijs/article_tables.xlsx and outputs_ijs/article_tables.tex, prints every table, and compares
every cell with the value printed in the article (half-up rounding to three decimals).
"""
import json
import os
import sys
from decimal import Decimal, ROUND_HALF_UP

import pandas as pd

OUT = "outputs_ijs"            # set from the command line when run as a script
REV = f"{OUT}/revision"


def h3(v, d=3):
    """Round half up (as printed in the article)."""
    if v is None or (isinstance(v, float) and v != v):
        return "--"
    q = Decimal(1).scaleb(-d)
    return str(Decimal(f"{float(v):.10f}").quantize(q, rounding=ROUND_HALF_UP))


def load():
    r = json.load(open(f"{OUT}/ijs_results.json"))
    R = json.load(open(f"{REV}/revision_results.json"))
    NS = json.load(open(f"{REV}/nested_sensitivity.json"))
    EX = json.load(open(f"{REV}/reviewer_extras.json"))
    return r, R, NS, EX


def table1():
    t = pd.read_csv(f"{OUT}/Table1_structure.csv")
    # a healthy sample is evaluable under sample-wise referencing only if the patient has another healthy sample
    t["EvaluableHealthy"] = t.HealthySamples.where(t.HealthySamples >= 2, 0)
    return t


def table2(r, R):
    S, C = r["summary"], r["ci"]
    RS, RC = R["summary"], R["ci"]
    rows = [("Spectrum-wise", "Leaky", S["naive_leaky|fusion"], C.get("naive_leaky|fusion")),
            ("Spectrum-wise", "Nested", S["naive_nested|fusion"], C.get("naive_nested|fusion")),
            ("Sample-wise", "Leaky", S["sa_leaky|fusion"], C.get("sa_leaky|fusion")),
            ("Sample-wise", "Nested", S["sa_nested|fusion"], C.get("sa_nested|fusion")),
            ("Size-matched", "Nested", RS["matched_nested|fusion"], RC.get("matched_nested|fusion"))]
    out = []
    for ref, sel, s, ci in rows:
        out.append({"Referencing": ref, "Selection": sel, "Tumor correct": f"{s['TP']}/{s['N1']}",
                    "Healthy correct": f"{s['TN']}/{s['N0']}",
                    "Sample BA (95% CI)": f"{h3(s['BA'])} ({h3(ci[0])}--{h3(ci[1])})",
                    "Sample AUC": h3(s["AUC"]), "Spectrum BA": h3(s["SpectrumBA"])})
    return pd.DataFrame(out)


def table3(r, R):
    S, RS = r["summary"], R["summary"]
    g = lambda D, k: D[k]["BA"] if k in D else None
    rows = [("Nested frequencies", "Spectrum-wise", S, "naive_nested", None),
            ("", "Sample-wise", S, "sa_nested", "sa_nested|innerw"),
            ("", "Size-matched", RS, "matched_nested", None),
            ("Leaky frequencies", "Spectrum-wise", S, "naive_leaky", None),
            ("", "Sample-wise", S, "sa_leaky", None),
            ("Published set", "Sample-wise", S, "sa_published", None),
            ("", "Size-matched", RS, "matched_published", None),
            ("Generic decade-spaced", "Sample-wise", S, "sa_generic", None),
            ("Cole parameters", "Spectrum-wise", S, "naive_cole", None),
            ("", "Sample-wise", S, "sa_cole", "sa_cole|innerw"),
            ("", "Size-matched", RS, "matched_cole", None)]
    out = []
    for feat, ref, D, k, iw in rows:
        out.append({"Features": feat, "Referencing": ref, "Absolute": h3(g(D, f"{k}|abs")),
                    "Mahalanobis": h3(g(D, f"{k}|mah")), "Equal fusion": h3(g(D, f"{k}|fusion")),
                    "Inner-fold weight": h3(S[iw]["BA"]) if iw else "--"})
    return pd.DataFrame(out)


def table4(r, R, NS, EX):
    S, RS = r["summary"], R["summary"]
    ba = lambda D, k: D[k]["BA"]
    rows = [("Primary analysis", ba(S, "sa_nested|fusion"), ba(RS, "nested_pipeline|selected"), ba(S, "sa_cole|abs"), ba(S, "sa_published|fusion")),
            ("P2 excluded", ba(S, "noP2_sa_nested|fusion"), NS["p2_excluded"]["BA"], ba(S, "noP2_sa_cole|abs"), EX["published_without_P2"]["BA"]),
            ("P2 labels exchanged", ba(RS, "p2_sa_nested|fusion"), NS["p2_exchanged"]["BA"], ba(RS, "p2_sa_cole|abs"), ba(RS, "p2_sa_published|fusion")),
            ("Size-matched referencing", ba(RS, "matched_nested|fusion"), NS["matched"]["BA"], ba(RS, "matched_cole|abs"), ba(RS, "matched_published|fusion")),
            ("Class-balanced weighting", ba(S, "sa_nested|fusion|cb"), None, ba(S, "sa_cole|abs|cb"), None),
            ("Fallback: outer-fold ranking", None, NS["fallback_outer"]["BA"], None, None),
            ("Fallback: fixed frequency set", None, NS["fallback_generic"]["BA"], None, None),
            ("Logistic regression", ba(RS, "lr_sa_nested|fusion"), None, ba(RS, "lr_sa_cole|abs"), ba(RS, "lr_sa_published|fusion")),
            ("ln R0 and ln Rinf only", None, None, ba(S, "sa_coleR|abs"), None),
            ("fc inside fitted window only", None, None, EX["cole_in_window_only"]["BA"], None)]
    return pd.DataFrame([{"Analysis": a, "Pre-specified": h3(b), "Fully nested pipeline": h3(c),
                          "Cole, absolute": h3(d), "Published set": h3(e)} for a, b, c, d, e in rows])


# values printed in the article (Tables 1-4), used to verify the rebuilt tables cell by cell
PRINTED = {
    "Table 1": [["6", "105", "6", "110", "100", "6"], ["1", "36", "2", "90", "100", "0"], ["1", "45", "4", "180", "50", "0"],
                ["2", "90", "1", "45", "50", "2"], ["2", "16", "2", "20", "50", "2"], ["1", "16", "1", "14", "20", "0"]],
    "Table 2": [["16/16", "10/10", "1.000 (1.000--1.000)", "1.000", "0.984"],
                ["15/16", "10/10", "0.969 (0.857--1.000)", "0.981", "0.965"],
                ["15/16", "8/10", "0.869 (0.667--0.964)", "0.875", "0.930"],
                ["15/16", "7/10", "0.819 (0.429--0.964)", "0.813", "0.912"],
                ["15/16", "7/10", "0.819 (0.429--0.964)", "0.813", "0.908"]],
    "Table 3": [["0.888", "0.969", "0.969", "--"], ["0.681", "0.344", "0.819", "0.681"], ["0.681", "0.344", "0.819", "--"],
                ["0.825", "1.000", "1.000", "--"], ["0.781", "0.344", "0.869", "--"], ["0.813", "0.563", "0.900", "--"],
                ["0.844", "0.719", "0.950", "--"], ["0.813", "0.506", "0.763", "--"], ["0.950", "0.650", "0.888", "--"],
                ["0.950", "0.438", "0.694", "0.856"], ["0.888", "0.306", "0.706", "--"]],
    "Table 4": [["0.819", "0.950", "0.950", "0.900"], ["0.864", "0.914", "0.914", "0.900"], ["0.583", "0.608", "0.842", "0.850"],
                ["0.819", "0.756", "0.888", "0.950"], ["0.869", "--", "0.856", "--"], ["--", "0.713", "--", "--"],
                ["--", "0.950", "--", "--"], ["0.744", "--", "0.850", "0.794"], ["--", "--", "0.950", "--"],
                ["--", "--", "0.950", "--"]],
}


def verify(tabs):
    rows = []
    for name, first_col in (("Table 1", 1), ("Table 2", 2), ("Table 3", 2), ("Table 4", 1)):
        df = tabs[name]
        for i, exp in enumerate(PRINTED[name]):
            got = [str(v) for v in df.iloc[i, first_col:first_col + len(exp)].tolist()]
            for j, (e, g_) in enumerate(zip(exp, got)):
                rows.append(dict(Table=name, Row=i + 1, Column=df.columns[first_col + j], Article=e, Computed=g_,
                                 Status="PASS" if e == g_ else "FAIL"))
    return pd.DataFrame(rows)


def main():
    r, R, NS, EX = load()
    tabs = {"Table 1": table1(), "Table 2": table2(r, R), "Table 3": table3(r, R), "Table 4": table4(r, R, NS, EX)}
    with pd.ExcelWriter(f"{OUT}/article_tables.xlsx") as xw:
        for k, v in tabs.items():
            v.to_excel(xw, sheet_name=k, index=False)
    with open(f"{OUT}/article_tables.tex", "w") as fh:
        for k, v in tabs.items():
            fh.write(f"% {k}\n" + v.to_latex(index=False, escape=False) + "\n")
    chk = verify(tabs)
    chk.to_csv(f"{OUT}/article_tables_check.csv", index=False)
    return tabs, chk


if __name__ == "__main__":
    if len(sys.argv) > 1:
        OUT = sys.argv[1]
        REV = f"{OUT}/revision"
    tabs, chk = main()
    pd.set_option("display.width", 200)
    for k, v in tabs.items():
        print(f"\n{k}\n{v.to_string(index=False)}")
    bad = chk[chk.Status == "FAIL"]
    print(f"\nTables 1-4: {(chk.Status == 'PASS').sum()} / {len(chk)} cells agree with the article")
    if len(bad):
        print(bad.to_string(index=False))


In [ ]:
%%writefile ijs_revision2_extras.py
"""Additional analyses requested in the second internal review (no new model fitting).

1. 2x2 interaction of referencing design and frequency selection, with the paired patient-level cluster bootstrap.
2. Patient-level summaries: per-patient sensitivity and specificity, macro-averaged over patients.
3. Acquisition-order drift versus tissue class (is drift a class-linked confounder?).
4. Exact Clopper-Pearson bound for the selection-aware permutation p-value.
5. Sample-level BA of all size-matched configurations (for Table 3 of the article).

Inputs: per-sample prediction tables and drift/Cole outputs of the main pipeline (folder `results/` of the package,
or `outputs_ijs/` when run inside the notebook). Output: JSON with every value printed in the revised article.
"""
import json
import os
import sys

import numpy as np
import pandas as pd
from scipy import stats

RES = sys.argv[1] if len(sys.argv) > 1 else "outputs_ijs"
REV = f"{RES}/revision"
SEED0 = 20260921
out = {}


def st(path):
    return pd.read_csv(path)


def pooled_ba(t, thr=0.5):
    pred = (t.p >= thr).astype(int)
    sens = ((pred == 1) & (t.y == 1)).sum() / (t.y == 1).sum()
    spec = ((pred == 0) & (t.y == 0)).sum() / (t.y == 0).sum()
    return 0.5 * (sens + spec)


def counts(t, pats, thr=0.5):
    pred = (t.p >= thr).astype(int)
    c = np.zeros((len(pats), 4))
    for k, q in enumerate(pats):
        m = (t.Patient == q).values
        c[k] = [((pred == 1) & (t.y == 1) & m).sum(), ((t.y == 1) & m).sum(),
                ((pred == 0) & (t.y == 0) & m).sum(), ((t.y == 0) & m).sum()]
    return c


def boot_draws(tables, B=10000, seed=SEED0):
    """Identical resampling scheme to eis_groups.cluster_boot (same seed, same discarding rule)."""
    names = list(tables)
    pats = np.asarray(sorted(tables[names[0]].Patient.unique()))
    C = {n: counts(tables[n], pats) for n in names}
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(pats), size=(B, len(pats)))
    D = {}
    for n in names:
        s = C[n][idx].sum(1)
        with np.errstate(invalid="ignore", divide="ignore"):
            D[n] = 0.5 * (s[:, 0] / s[:, 1] + s[:, 2] / s[:, 3])
    valid = np.all([np.isfinite(v) for v in D.values()], axis=0)
    return {n: v[valid] for n, v in D.items()}, int(valid.sum())


# ------------------------------------------------------------------ 1. interaction in the 2x2 factorial design
T = {"SW_leaky": st(f"{RES}/sampletable_naive_leaky_fusion.csv"), "SW_nested": st(f"{RES}/sampletable_naive_nested_fusion.csv"),
     "SA_leaky": st(f"{RES}/sampletable_sa_leaky_fusion.csv"), "SA_nested": st(f"{RES}/sampletable_sa_nested_fusion.csv")}
ba = {k: pooled_ba(v) for k, v in T.items()}
D, nvalid = boot_draws(T)
inter = (ba["SW_leaky"] - ba["SW_nested"]) - (ba["SA_leaky"] - ba["SA_nested"])
inter_d = (D["SW_leaky"] - D["SW_nested"]) - (D["SA_leaky"] - D["SA_nested"])
ref_nested = D["SW_nested"] - D["SA_nested"]
out["factorial"] = {
    "BA": ba, "valid_resamples": nvalid,
    "referencing_effect_nested_CI_check": [float(np.percentile(ref_nested, 2.5)), float(np.percentile(ref_nested, 97.5))],
    "interaction": inter, "interaction_CI": [float(np.percentile(inter_d, 2.5)), float(np.percentile(inter_d, 97.5))],
    "sum_of_main_effects": (ba["SW_nested"] - ba["SA_nested"]) + (ba["SA_leaky"] - ba["SA_nested"]),
    "combined_effect": ba["SW_leaky"] - ba["SA_nested"]}

# ------------------------------------------------------------------ 2. patient-level summaries
def patient_level(t):
    rows = []
    for q, g in t.groupby("Patient"):
        pred = (g.p >= 0.5).astype(int)
        nt, nh = int((g.y == 1).sum()), int((g.y == 0).sum())
        rows.append(dict(Patient=q, sens=((pred == 1) & (g.y == 1)).sum() / nt if nt else np.nan,
                         spec=((pred == 0) & (g.y == 0)).sum() / nh if nh else np.nan,
                         all_correct=bool((pred == g.y).all())))
    r = pd.DataFrame(rows)
    both = r.dropna()
    return dict(macro_sens=float(r.sens.mean()), macro_spec=float(r.spec.mean()),
                patient_BA=float(0.5 * (r.sens.mean() + r.spec.mean())),
                n_patients_all_correct=int(r.all_correct.sum()), n_patients=int(len(r)),
                n_patients_both_classes=int(len(both)),
                per_patient=r.to_dict(orient="records"))


PL = {"spectrum-wise, leaky (naive)": f"{RES}/sampletable_naive_leaky_fusion.csv",
      "pre-specified (sample-wise, nested)": f"{RES}/sampletable_sa_nested_fusion.csv",
      "Cole absolute, sample-wise": f"{RES}/sampletable_sa_cole_abs.csv",
      "fully nested pipeline": f"{REV}/sampletable_nested_pipeline_selected.csv",
      "fully nested, size-matched": f"{REV}/sampletable_nested_matched.csv",
      "pre-specified, size-matched": f"{REV}/sampletable_matched_nested_fusion.csv"}
out["patient_level"] = {k: patient_level(st(v)) for k, v in PL.items() if os.path.exists(v)}

# ------------------------------------------------------------------ 3. drift versus class
dr = pd.read_csv(f"{RES}/drift_per_sample.csv")
cf = pd.read_csv(f"{RES}/cole_fits.csv")
lab = cf.groupby("Sample").Label.first()
dr["Label"] = dr.Sample.map(lab)
h, t_ = dr[dr.Label == 0], dr[dr.Label == 1]
mw_rho = stats.mannwhitneyu(h.rho.abs(), t_.rho.abs())
mw_drift = stats.mannwhitneyu(h.drift.abs(), t_.drift.abs())
# per-patient tumour-healthy contrast of ln R0 from the first and from the last fifth of every series
cf["lnR0"] = np.log(cf.R0)
cf["pos"] = cf.groupby("Sample").cumcount()
cf["n"] = cf.groupby("Sample").Sample.transform("size")
first = cf[cf.pos < np.ceil(cf.n / 5)]
last = cf[cf.pos >= cf.n - np.ceil(cf.n / 5)]


def contrast(d):
    s = d.groupby(["Patient", "Sample", "Label"]).lnR0.mean().reset_index()
    m = s.groupby(["Patient", "Label"]).lnR0.mean().unstack()
    return (m[1] - m[0]).dropna()


cF, cL = contrast(first), contrast(last)
out["drift_vs_class"] = {
    "n_healthy_samples": int(len(h)), "n_tumor_samples": int(len(t_)),
    "median_abs_rho": {"healthy": float(h.rho.abs().median()), "tumor": float(t_.rho.abs().median())},
    "median_abs_drift_lnR0": {"healthy": float(h.drift.abs().median()), "tumor": float(t_.drift.abs().median())},
    "share_positive_drift": {"healthy": float((h.drift > 0).mean()), "tumor": float((t_.drift > 0).mean())},
    "mannwhitney_p_abs_rho": float(mw_rho.pvalue), "mannwhitney_p_abs_drift": float(mw_drift.pvalue),
    "contrast_first_fifth": cF.to_dict(), "contrast_last_fifth": cL.to_dict(),
    "max_abs_change_in_contrast": float((cF - cL).abs().max()),
    "sign_agreement": int((np.sign(cF) == np.sign(cL)).sum()), "n_patients": int(len(cF))}

# ------------------------------------------------------------------ 4. selection-aware permutation p-value
B, k = 100, 0
out["selection_aware_p"] = {"B": B, "exceed": k, "p_hat": (1 + k) / (1 + B),
                            "clopper_pearson_upper_95": float(stats.beta.ppf(0.95, k + 1, B - k)),
                            "clopper_pearson_two_sided_upper": float(stats.beta.ppf(0.975, k + 1, B - k))}

# ------------------------------------------------------------------ 5. size-matched configurations
out["size_matched_BA"] = {k: pooled_ba(st(f"{REV}/sampletable_matched_{k}.csv")) for k in
                          ("nested_abs", "nested_mah", "nested_fusion", "cole_abs", "cole_mah", "cole_fusion",
                           "published_abs", "published_mah", "published_fusion")}

json.dump(out, open(f"{REV}/revision2_extras.json", "w"), indent=1, default=float)
print(json.dumps({k: v for k, v in out.items() if k != "patient_level"}, indent=1, default=float))
for k, v in out["patient_level"].items():
    print(k, {a: b for a, b in v.items() if a != "per_patient"})


In [32]:
%%writefile ijs_paired_matched.py
"""
ijs_paired_matched.py -- paired patient-level cluster bootstrap of the difference between the
fully nested pipeline under ordinary sample-wise referencing and under size-matched referencing
(complete nested procedure repeated with size-matched references), and of the corresponding
difference for the pre-specified classifier and the absolute Cole branch.
Output: outputs_ijs/revision/paired_matched.json
"""
import json

import pandas as pd

import eis_groups as eg

REV = "outputs_ijs/revision"
T = {"nested_sa": pd.read_csv(f"{REV}/sampletable_nested_pipeline_selected.csv"),
     "nested_matched": pd.read_csv(f"{REV}/sampletable_nested_matched.csv"),
     "prespecified_sa": pd.read_csv("outputs_ijs/sampletable_sa_nested_fusion.csv"),
     "prespecified_matched": pd.read_csv(f"{REV}/sampletable_matched_nested_fusion.csv"),
     "coleabs_sa": pd.read_csv("outputs_ijs/sampletable_sa_cole_abs.csv"),
     "coleabs_matched": pd.read_csv(f"{REV}/sampletable_matched_cole_abs.csv")}
ci, diff, nvalid = eg.cluster_boot(T)
out = dict(n_valid=int(nvalid))
for k in ("nested", "prespecified", "coleabs"):
    a, b = f"{k}_sa", f"{k}_matched"
    d = diff[f"{a} - {b}"]
    out[k] = dict(BA_sa=eg.pooled(T[a])["BA"], BA_matched=eg.pooled(T[b])["BA"],
                  diff=eg.pooled(T[a])["BA"] - eg.pooled(T[b])["BA"], boot_mean=d[0], ci_low=d[1], ci_high=d[2],
                  ci_sa=list(ci[a]), ci_matched=list(ci[b]))
json.dump(out, open(f"{REV}/paired_matched.json", "w"), indent=1)
print(json.dumps(out, indent=1))

Writing ijs_paired_matched.py


In [33]:
import subprocess, sys
REVDIR = "outputs_ijs/revision"
steps = [("permutation_results.json", ["ijs_permutation.py", "1000"]),
         ("revision_results.json", ["ijs_revision.py"]),
         ("revision_results_v2_outer_ranking.json", ["ijs_revision.py", "--partial-nesting"]),
         ("sim_precision.csv", ["ijs_simulation.py", "all"]),
         ("nested_sensitivity.json", ["ijs_nested_sensitivity.py"]),
         ("reviewer_extras.json", ["ijs_reviewer_extras.py"]),
         ("sim_ext_summary.json", ["ijs_simulation_ext.py", "2"]),
         ("selection_permutation_results.json", ["ijs_selection_permutation.py", "100"]),
         ("paired_matched.json", ["ijs_paired_matched.py"]),
         ("revision2_extras.json", ["ijs_revision2_extras.py", "outputs_ijs"])]
for target, cmd in steps:
    if os.path.exists(f"{REVDIR}/{target}"):
        print(f"{target}: found (delete it to recompute)")
    else:
        print("running", " ".join(cmd), flush=True)
        subprocess.run([sys.executable] + cmd, check=True)
out_ = subprocess.run([sys.executable, "sim_summary.py"], check=True, capture_output=True, text=True)
REVR = json.load(open(f"{REVDIR}/revision_results.json"))
PERM = json.load(open(f"{REVDIR}/permutation_results.json"))
SIMS = json.load(open(f"{REVDIR}/simulation_summary.json"))

### 16.1 Permutation null of fixed configurations (Fig. 4a-c, Table S3)

In [34]:
from IPython.display import display
print("permutations:", PERM["n_perm"])
display(pd.DataFrame({k: v for k, v in PERM.items() if k != "n_perm"}).T.round(3))

permutations: 1000


,null_mean,null_sd,null_q025,null_q975,frac_null_ge_0p75,observed,p_value
cole|naive|fusion,0.696,0.094,0.512,0.869,0.290,0.888,0.019
cole|naive|abs,0.563,0.117,0.344,0.806,0.064,0.950,0.001
cole|naive|mah,0.764,0.072,0.619,0.906,0.638,0.650,0.957
cole|sa|fusion,0.454,0.082,0.312,0.638,0.004,0.694,0.010
cole|sa|abs,0.459,0.088,0.312,0.700,0.013,0.950,0.001
cole|sa|mah,0.459,0.112,0.250,0.669,0.005,0.438,0.582
published|naive|fusion,0.740,0.088,0.562,0.906,0.454,0.875,0.081
published|naive|abs,0.557,0.129,0.319,0.794,0.081,0.812,0.020
published|naive|mah,0.823,0.055,0.719,0.938,0.958,1.000,0.001
published|sa|fusion,0.445,0.083,0.312,0.656,0.001,0.900,0.001


### 16.2 Fully nested pipeline selection, size-matched referencing, P2 labellings, logistic regression (Sect. 3.6; Tables 4, S4, S6)

In [35]:
print("fully nested, pipeline chosen per outer fold:", REVR["nested_pipeline_choice"])
print("inner-fold frequency ranking method:", REVR["inner_selection_method"])
display(pd.read_csv(f"{REVDIR}/nested_pipeline_inner_scores.csv").pivot_table(index="Fold", columns=["Representation", "Branch"], values="InnerBA").round(3))
REVP = json.load(open(f"{REVDIR}/revision_results_v2_outer_ranking.json"))
print("partially nested (outer ranking reused):", REVP["nested_pipeline_choice"], REVP["summary"]["nested_pipeline|selected"]["BA"])
rs = pd.DataFrame(REVR["summary"]).T[["BA", "TP", "N1", "TN", "N0", "AUC"]]
rs["CI"] = [REVR["ci"].get(k) for k in rs.index]
display(rs)

fully nested, pipeline chosen per outer fold: {'P1': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 1.0}, 'P2': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.9142857142857144}, 'P3': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.95}, 'P4': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.9375}, 'P5': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.8660714285714286}, 'P6': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.9166666666666667}}
inner-fold frequency ranking method: {'P1|P2': 'LOPO', 'P1|P3': 'LOPO', 'P1|P4': 'LOSO', 'P1|P5': 'LOSO', 'P1|P6': 'LOPO', 'P2|P1': 'LOPO', 'P2|P3': 'LOPO', 'P2|P4': 'LOPO', 'P2|P5': 'LOPO', 'P2|P6': 'LOPO', 'P3|P1': 'LOPO', 'P3|P2': 'LOPO', 'P3|P4': 'LOPO', 'P3|P5': 'LOPO', 'P3|P6': 'LOPO', 'P4|P1': 'LOSO', 'P4|P2': 'LOPO', 'P4|P3': 'LOPO', 'P4|P5': 'LOSO', 'P4|P6': 'LOPO', 'P5|P1': 'LOSO', 'P5|P2': 'LOPO', 'P5|P3': 'LOPO', 'P5|P4': 'LOSO', 'P5|P6': 'LOPO', 'P6|P1': 'LOPO', 'P6|P2': 'LOPO',

Representation   cole                 freq              
Branch            abs fusion    mah    abs fusion    mah
Fold                                                    
P1              1.000  0.825  0.300  0.625  0.575  0.425
P2              0.914  0.793  0.514  0.650  0.864  0.443
P3              0.950  0.742  0.500  0.708  0.908  0.375
P4              0.938  0.458  0.188  0.842  0.779  0.429
P5              0.866  0.375  0.402  0.795  0.464  0.250
P6              0.917  0.717  0.650  0.767  0.833  0.617

partially nested (outer ranking reused): {'P1': {'representation': 'freq', 'branch': 'abs', 'inner_BA': 1.0}, 'P2': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.9142857142857144}, 'P3': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.95}, 'P4': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.9375}, 'P5': {'representation': 'freq', 'branch': 'abs', 'inner_BA': 0.9375}, 'P6': {'representation': 'cole', 'branch': 'abs', 'inner_BA': 0.9166666666666667}} 0.7125


,BA,TP,N1,TN,N0,AUC,CI
p2_naive_nested|fusion,0.916667,15.0,15.0,10.0,12.0,0.994444,"[0.625, 1.0]"
p2_naive_nested|abs,0.833333,15.0,15.0,8.0,12.0,0.733333,None
p2_naive_nested|mah,0.800000,9.0,15.0,12.0,12.0,0.994444,None
p2_naive_leaky|fusion,0.916667,15.0,15.0,10.0,12.0,1.000000,"[0.8125, 1.0]"
p2_naive_leaky|abs,0.875000,15.0,15.0,9.0,12.0,0.816667,None
p2_naive_leaky|mah,1.000000,15.0,15.0,12.0,12.0,1.000000,None
p2_sa_nested|fusion,0.583333,10.0,15.0,6.0,12.0,0.661111,"[0.1111111111111111, 0.8333333333333333]"
p2_sa_nested|abs,0.600000,8.0,15.0,8.0,12.0,0.611111,None
p2_sa_nested|mah,0.491667,11.0,15.0,3.0,12.0,0.655556,None
p2_sa_leaky|fusion,0.700000,11.0,15.0,8.0,12.0,0.772222,"[0.29166666666666663, 0.9166666666666667]"


### 16.3 Robustness of the fully nested selection, per-patient influence, stability, design effect (Tables 4, S1, S2, S5)

In [36]:
NS = json.load(open(f"{REVDIR}/nested_sensitivity.json"))
display(pd.DataFrame({k: {c: v[c] for c in ("BA", "TP", "N1", "TN", "N0", "n_cole", "n_abs")} | {"CI": v["CI"]} for k, v in NS.items()}).T)
EX = json.load(open(f"{REVDIR}/reviewer_extras.json"))
for k in ("influence", "frequency_stability", "cole_in_window_only", "seed_stability", "published_without_P2"):
    print(k, json.dumps(EX[k], indent=1, default=str)[:1500])
display(pd.read_csv(f"{REVDIR}/per_patient_confusion.csv"))
print("paired sample-wise vs size-matched:", json.dumps(json.load(open(f"{REVDIR}/paired_matched.json")), indent=1))

,BA,TP,N1,TN,N0,n_cole,n_abs,CI
fallback_outer,0.7125,10,16,8,10,4,6,"[0.5, 1.0]"
fallback_generic,0.95,16,16,9,10,6,6,"[0.9166666666666667, 1.0]"
matched,0.75625,13,16,7,10,5,5,"[0.3076923076923077, 0.9226190476190477]"
p2_exchanged,0.608333,7,15,9,12,4,4,"[0.275, 0.7976190476190477]"
p2_excluded,0.914286,13,14,9,10,5,5,"[0.842948717948718, 1.0]"


influence {
 "Fully nested pipeline": {
  "min": 0.9375,
  "max": 1.0,
  "min_patient": "P4",
  "max_patient": "P1"
 },
 "Sample-wise, nested frequencies": {
  "min": 0.7,
  "max": 0.9017857142857143,
  "min_patient": "P1",
  "max_patient": "P5"
 },
 "Sample-wise, published set": {
  "min": 0.875,
  "max": 1.0,
  "min_patient": "P4",
  "max_patient": "P1"
 },
 "Spectrum-wise, nested frequencies": {
  "min": 0.95,
  "max": 1.0,
  "min_patient": "P1",
  "max_patient": "P6"
 }
}
frequency_stability {
 "sample-wise outer": {
  "n_sets": 6,
  "share_below_10k": 0.03333333333333333,
  "share_10_100k": 0.13333333333333333,
  "share_above_100k": 0.8333333333333334,
  "mean_jaccard": 0.13941798941798939,
  "n_distinct": 17,
  "max_count": 3,
  "most_frequent_khz": 184.20699693267164
 },
 "sample-wise inner": {
  "n_sets": 30,
  "share_below_10k": 0.24,
  "share_10_100k": 0.12,
  "share_above_100k": 0.64,
  "mean_jaccard": 0.13471994161649334,
  "n_distinct": 38,
  "max_count": 12,
  "most_frequ

,Config,Patient,TP,FN,TN,FP,BA
0,"Spectrum-wise, nested frequencies",P1,6,0,6,0,1.000000
1,"Spectrum-wise, nested frequencies",P2,2,0,0,0,NaN
2,"Spectrum-wise, nested frequencies",P3,4,0,0,0,NaN
3,"Spectrum-wise, nested frequencies",P4,1,0,2,0,1.000000
4,"Spectrum-wise, nested frequencies",P5,2,0,2,0,1.000000
5,"Spectrum-wise, nested frequencies",P6,0,1,0,0,NaN
6,"Sample-wise, nested frequencies",P1,6,0,5,1,0.916667
7,"Sample-wise, nested frequencies",P2,2,0,0,0,NaN
8,"Sample-wise, nested frequencies",P3,4,0,0,0,NaN
9,"Sample-wise, nested frequencies",P4,1,0,2,0,1.000000


paired sample-wise vs size-matched: {
 "n_valid": 9861,
 "nested": {
  "BA_sa": 0.95,
  "BA_matched": 0.75625,
  "diff": 0.19374999999999998,
  "boot_mean": 0.23559854499976754,
  "ci_low": 0.022727272727272707,
  "ci_high": 0.6923076923076923,
  "ci_sa": [
   0.9166666666666667,
   1.0
  ],
  "ci_matched": [
   0.3076923076923077,
   0.9226190476190477
  ]
 },
 "prespecified": {
  "BA_sa": 0.81875,
  "BA_matched": 0.81875,
  "diff": 0.0,
  "boot_mean": 0.0,
  "ci_low": 0.0,
  "ci_high": 0.0,
  "ci_sa": [
   0.42857142857142855,
   0.9642857142857143
  ],
  "ci_matched": [
   0.42857142857142855,
   0.9642857142857143
  ]
 },
 "coleabs": {
  "BA_sa": 0.95,
  "BA_matched": 0.8875,
  "diff": 0.0625,
  "boot_mean": 0.06885981977484874,
  "ci_low": 0.0,
  "ci_high": 0.25,
  "ci_sa": [
   0.9166666666666667,
   1.0
  ],
  "ci_matched": [
   0.7375,
   1.0
  ]
 }
}


### 16.4 Selection-aware permutation test (Fig. 4d, Sect. 3.5)

In [37]:
SP = json.load(open(f"{REVDIR}/selection_permutation_results.json"))
print(json.dumps(SP, indent=1))

{
 "n_perm": 100,
 "BA_selected": {
  "observed": 0.95,
  "null_mean": 0.47112499999999996,
  "null_sd": 0.09654617948281162,
  "null_q025": 0.31546874999999996,
  "null_q975": 0.6914062499999998,
  "null_max": 0.75,
  "p_value": 0.009900990099009901
 },
 "BA_prespecified": {
  "observed": 0.81875,
  "null_mean": 0.45549999999999996,
  "null_sd": 0.07223589128611199,
  "null_q025": 0.32734375,
  "null_q975": 0.6446874999999996,
  "null_max": 0.675,
  "p_value": 0.009900990099009901
 },
 "null_choice": {
  "mean_folds_cole": 3.15,
  "mean_folds_abs": 2.79,
  "frac_all_cole_abs": 0.0
 }
}


### 16.5 Simulation study (Fig. 5, Tables S7-S9, Note S1)

In [38]:
display(pd.read_csv(f"{REVDIR}/simulation_cell_means.csv").round(3))
print(json.dumps(SIMS["precision"], indent=1))

,delta,icc,m,naive_est,naive_true,sa_est,sa_true,matched_est,matched_true,naive_opt,sa_opt,matched_opt
0,0.0,0.10,1,0.869,0.509,0.869,0.509,0.481,0.503,0.359,0.359,-0.022
1,0.0,0.10,5,0.488,0.514,0.517,0.512,0.522,0.511,-0.026,0.005,0.012
2,0.0,0.10,15,0.537,0.506,0.523,0.488,0.497,0.495,0.031,0.035,0.002
3,0.0,0.10,30,0.581,0.509,0.491,0.505,0.493,0.503,0.071,-0.014,-0.010
4,0.0,0.25,1,0.857,0.501,0.857,0.501,0.525,0.502,0.356,0.356,0.023
5,0.0,0.25,5,0.519,0.516,0.528,0.483,0.504,0.497,0.003,0.045,0.007
6,0.0,0.25,15,0.631,0.508,0.484,0.499,0.514,0.493,0.123,-0.015,0.021
7,0.0,0.25,30,0.650,0.528,0.481,0.484,0.468,0.497,0.122,-0.003,-0.029
8,0.0,0.50,1,0.846,0.510,0.846,0.510,0.481,0.498,0.336,0.336,-0.017
9,0.0,0.50,5,0.649,0.534,0.526,0.484,0.516,0.497,0.115,0.042,0.019


{
 "n6_h2": {
  "lo": 0.6309027777777778,
  "hi": 0.9583333333333331,
  "width": 0.32743055555555534
 },
 "n6_h3": {
  "lo": 0.7076388888888889,
  "hi": 0.9722222222222222,
  "width": 0.2645833333333333
 },
 "n12_h2": {
  "lo": 0.7083333333333333,
  "hi": 0.9411458333333332,
  "width": 0.23281249999999998
 },
 "n12_h3": {
  "lo": 0.7704861111111111,
  "hi": 0.9305555555555556,
  "width": 0.1600694444444445
 },
 "n24_h2": {
  "lo": 0.7273437500000001,
  "hi": 0.8943576388888888,
  "width": 0.16701388888888868
 },
 "n24_h3": {
  "lo": 0.7880208333333333,
  "hi": 0.9272569444444444,
  "width": 0.13923611111111112
 },
 "n48_h2": {
  "lo": 0.7760416666666666,
  "hi": 0.8871527777777778,
  "width": 0.11111111111111116
 },
 "n48_h3": {
  "lo": 0.8141493055555555,
  "hi": 0.8993055555555556,
  "width": 0.08515625000000004
 }
}


### 16.6 Second-revision analyses: 2x2 interaction, patient-level BA, drift versus class, Clopper-Pearson bound, size-matched branches

In [ ]:
X2 = json.load(open(f"{REVDIR}/revision2_extras.json"))
F2 = X2["factorial"]
print("Sect. 3.2  factorial BA:", {k: round(v, 4) for k, v in F2["BA"].items()})
print(f"           interaction {100 * F2['interaction']:.1f} points (95% CI {100 * F2['interaction_CI'][0]:.1f} to {100 * F2['interaction_CI'][1]:.1f}); "
      f"sum of main effects {100 * F2['sum_of_main_effects']:.1f}; combined {100 * F2['combined_effect']:.1f}; valid resamples {F2['valid_resamples']}")
print("\nSect. 3.4/3.6  patient-level balanced accuracy")
display(pd.DataFrame({k: {c: v[c] for c in ("patient_BA", "macro_sens", "macro_spec", "n_patients_all_correct")}
                      for k, v in X2["patient_level"].items()}).T.round(3))
D2 = X2["drift_vs_class"]
print("Sect. 3.1  drift versus class:", {k: D2[k] for k in ("median_abs_rho", "median_abs_drift_lnR0", "share_positive_drift")})
print(f"           Mann-Whitney p |rho| = {D2['mannwhitney_p_abs_rho']:.3f}, p |drift| = {D2['mannwhitney_p_abs_drift']:.3f}; "
      f"contrast sign agreement {D2['sign_agreement']}/{D2['n_patients']}; max change {D2['max_abs_change_in_contrast']:.3f}")
display(pd.DataFrame({"first fifth": D2["contrast_first_fifth"], "last fifth": D2["contrast_last_fifth"]}).round(3))
P2_ = X2["selection_aware_p"]
print(f"Sect. 3.5  selection-aware p = {P2_['p_hat']:.4f} (B = {P2_['B']}); one-sided 95% Clopper-Pearson upper bound {P2_['clopper_pearson_upper_95']:.3f}")
print("Table 3    size-matched BA:", {k: round(v, 4) for k, v in X2["size_matched_BA"].items()})

## 17. Tables 1-4 of the article and Tables S1-S9 of Online Resource 1
`article_tables.py` rebuilds Tables 1-4 and compares every cell with the printed article; `make_supp_tables.py` writes the LaTeX source of Tables S1-S9 (`outputs_ijs/supp_tables.tex`).

In [ ]:
import importlib, article_tables as at
importlib.reload(at)
TABS, TCHK = at.main()
for name, tab in TABS.items():
    print(name); display(tab)
print(f"Tables 1-4: {(TCHK.Status == 'PASS').sum()} / {len(TCHK)} cells agree with the article")
if (TCHK.Status == "FAIL").any():
    display(TCHK[TCHK.Status == "FAIL"])
subprocess.run([sys.executable, "make_supp_tables.py"], check=True)
print(open("outputs_ijs/supp_tables.tex").read()[:1500], "...")

## 18. Figures 1-5

In [ ]:
import ijs_revision_figs as rf
from IPython.display import Image as IPImage
importlib.reload(rf)
subprocess.run([sys.executable, "make_fig12.py"], check=True)   # Figs. 1 and 2
subprocess.run([sys.executable, "make_fig3.py"], check=True)    # Fig. 3
rf.permutation_figure(); rf.simulation_figure()                 # Figs. 4 and 5
for f in ("Fig1_Referencing_Designs", "Fig2_Sample_Structure", "Fig3_Selected_Frequencies", "Fig4_Permutation_Null", "Fig5_Simulation"):
    print(f); display(IPImage(f"outputs_ijs/figs/{f}.png", width=900))

## 19. Consistency check against the article
Each entry compares a reported value with the value computed above; `PASS` means agreement within the printed rounding. The list also contains values reported in earlier versions and in the response to reviewers, so it is a superset of the values printed in the current article; Tables 1-4 are additionally checked cell by cell in Section 17.

In [41]:
# ---------------------------------------------------------------- consistency check
S_ = SUM
def ba(k): return S_[k]["BA"]
def cnt(k, which): return S_[k]["TP"] if which == "T" else S_[k]["TN"]
def d_ci(a, b):
    key = f"{a} - {b}"
    if key in DIFF:
        return DIFF[key][1], DIFF[key][2]
    lo, hi = DIFF[f"{b} - {a}"][1], DIFF[f"{b} - {a}"][2]
    return -hi, -lo
dd = pd.read_csv(f"{OUT}/mahalanobis_within_between.csv").groupby(["Patient", "kind"]).d.median()
sk = RES["selected_khz"]
C = []
def add(item, printed, computed, dec):
    C.append((item, printed, computed, dec))

# structure
add("spectra", 767, RES["structure"]["n_spectra"], 0); add("tissue samples", 29, RES["structure"]["n_samples"], 0)
add("healthy samples", 13, RES["structure"]["n_healthy_samples"], 0); add("tumor samples", 16, RES["structure"]["n_tumor_samples"], 0)
add("spectra per sample min", 6, RES["structure"]["spectra_per_sample_min"], 0); add("spectra per sample max", 45, RES["structure"]["spectra_per_sample_max"], 0)
add("spectra per sample median", 15, RES["structure"]["spectra_per_sample_median"], 0)
add("evaluable samples", 26, RES["evaluable"]["n"], 0); add("evaluable healthy", 10, RES["evaluable"]["healthy"], 0)
for p, hs, hn, ts, tn in (("P1", 6, 105, 6, 110), ("P2", 1, 36, 2, 90), ("P3", 1, 45, 4, 180), ("P4", 2, 90, 1, 45), ("P5", 2, 16, 2, 20), ("P6", 1, 16, 1, 14)):
    r_ = T1.set_index("Patient").loc[p]
    add(f"T1 {p} healthy samples", hs, r_.HealthySamples, 0); add(f"T1 {p} healthy spectra", hn, r_.HealthySpectra, 0)
    add(f"T1 {p} tumor samples", ts, r_.TumorSamples, 0); add(f"T1 {p} tumor spectra", tn, r_.TumorSpectra, 0)
# variance decomposition
for k_, v in (("healthy|ln R0", .975), ("healthy|ln Rinf", .573), ("healthy|log10 fc", .967), ("healthy|alpha", .905),
              ("tumor|ln R0", .992), ("tumor|ln Rinf", .901), ("tumor|log10 fc", .984), ("tumor|alpha", .971)):
    add(f"ICC {k_}", v, RES["icc"][k_], 3)
add("mean spectra per sample", 26.4, RES["design_effect"]["m_bar"], 1)
add("design effect", 25.8, RES["design_effect"]["deff_healthy_lnR0"], 1); add("effective n", 29.7, RES["design_effect"]["n_eff"], 1)
add("ICC ln|Z| 1 kHz", 0.950, RES["icc_freq"]["first"], 3); add("ICC ln|Z| 1 MHz", 0.601, RES["icc_freq"]["last"], 3)
add("ICC ln|Z| monotone decreasing", 1, float(RES["icc_freq"]["monotone"]), 0)
# Cole fit
add("Cole fit error median (%)", 0.74, RES["cole_fit"]["rmse_median"], 2); add("Cole fit error p95 (%)", 2.03, RES["cole_fit"]["rmse_p95"], 2)
add("Cole fit error max (%)", 6.81, RES["cole_fit"]["rmse_max"], 2); add("SE log10 fc median", 0.016, RES["cole_ident"]["se_log10fc_median"], 3)
add("fc outside window (n)", 59, RES["cole_ident"]["outside_n"], 0); add("fc outside window (%)", 7.7, RES["cole_ident"]["outside_pct"], 1)
add("fc outside P4 healthy", 40, RES["cole_ident"]["outside_by"].get("P4-H", 0), 0); add("fc outside P1 healthy", 16, RES["cole_ident"]["outside_by"].get("P1-H", 0), 0)
add("fc outside P6 healthy", 3, RES["cole_ident"]["outside_by"].get("P6-H", 0), 0)
# distances
add("within P1", 1.40, dd[("P1", "within")], 2); add("within P4", 0.95, dd[("P4", "within")], 2); add("within P5", 1.51, dd[("P5", "within")], 2)
add("between P1", 0.85, dd[("P1", "between")], 2); add("between P5", 2.27, dd[("P5", "between")], 2); add("between P4", 48.7, dd[("P4", "between")], 1)
add("tumor P4", 2.56, dd[("P4", "tumor")], 2); add("P4 between/tumor ratio", 19, dd[("P4", "between")] / dd[("P4", "tumor")], 0)
# drift
add("drift median |rho|", 0.96, RES["drift"]["median_abs_rho"], 2); add("drift |rho|>0.8 (%)", 62, 100 * RES["drift"]["frac_abs_rho_gt_08"], 0)
add("drift median |d ln R0|", 0.057, RES["drift"]["median_abs_drift_lnR0"], 3); add("contrast median |ln R0 ratio|", 0.955, RES["drift"]["median_abs_contrast_lnR0"], 3)
add("contrast fold", 2.6, float(np.exp(RES["drift"]["median_abs_contrast_lnR0"])), 1)
# Table 2 factorial
for k_, tp, tn, b, lo, hi, auc, sp in (("naive_leaky|fusion", 16, 10, 1.000, 1.000, 1.000, 1.000, .984),
                                      ("naive_nested|fusion", 15, 10, .969, .857, 1.000, .981, .965),
                                      ("sa_leaky|fusion", 15, 8, .869, .667, .964, .875, .930),
                                      ("sa_nested|fusion", 15, 7, .819, .429, .964, .812, .912)):
    add(f"T2 {k_} tumor correct", tp, cnt(k_, "T"), 0); add(f"T2 {k_} healthy correct", tn, cnt(k_, "H"), 0)
    add(f"T2 {k_} BA", b, ba(k_), 3); add(f"T2 {k_} CI low", lo, CI[k_][0], 3); add(f"T2 {k_} CI high", hi, CI[k_][1], 3)
    add(f"T2 {k_} AUC", auc, S_[k_]["AUC"], 3); add(f"T2 {k_} spectrum BA", sp, S_[k_]["SpectrumBA"], 3)
# Table 3 effects
for name, a, b, v, lo, hi in (("ref (nested)", "naive_nested|fusion", "sa_nested|fusion", .150, .000, .500),
                              ("ref (leaky)", "naive_leaky|fusion", "sa_leaky|fusion", .131, .036, .333),
                              ("sel (spectrum-wise)", "naive_leaky|fusion", "naive_nested|fusion", .031, .000, .143),
                              ("sel (sample-wise)", "sa_leaky|fusion", "sa_nested|fusion", .050, .000, .250),
                              ("combined", "naive_leaky|fusion", "sa_nested|fusion", .181, .036, .571),
                              ("ref Mahalanobis", "naive_nested|mah", "sa_nested|mah", .625, .500, .794),
                              ("ref absolute", "naive_nested|abs", "sa_nested|abs", .206, .000, .349)):
    l_, h_ = d_ci(a, b)
    add(f"T3 {name}", v, ba(a) - ba(b), 3); add(f"T3 {name} CI low", lo, l_, 3); add(f"T3 {name} CI high", hi, h_, 3)
add("spectrum-sample gap (pp)", 9.4, 100 * (S_["sa_nested|fusion"]["SpectrumBA"] - ba("sa_nested|fusion")), 1)
add("bootstrap discarded", 139, 10000 - RES["boot_valid"], 0)
# frequencies
add("naive P6 fold min (kHz)", 121, sk["naive_nested"]["P6"][0], 0); add("naive P6 fold max (kHz)", 212, sk["naive_nested"]["P6"][-1], 0)
add("naive leaky min (kHz)", 11, sk["naive_leaky"][0], 0); add("naive leaky max (kHz)", 26, sk["naive_leaky"][-1], 0)
add("naive P1 fold two low freqs < 3 kHz", 2, sum(f_ < 3 for f_ in sk["naive_nested"]["P1"]), 0)
add("naive folds with >=3 in 10-34 kHz (P1-P5)", 5, sum(sum(10 <= f_ <= 34 for f_ in sk["naive_nested"][p]) >= 3 for p in ("P1", "P2", "P3", "P4", "P5")), 0)
add("SA folds all >= 139 kHz", 5, sum(min(sk["sa_nested"][p]) >= 138.9 for p in ("P1", "P2", "P3", "P4", "P5", "P6")), 0)
add("SA P5 fold min (kHz)", 8.3, sk["sa_nested"]["P5"][0], 1); add("SA P5 fold max (kHz)", 19.3, sk["sa_nested"]["P5"][-1], 1)
add("SA leaky min (kHz)", 160, sk["sa_leaky"][0], 0); add("SA leaky max (kHz)", 1000, sk["sa_leaky"][-1], 0)
# Table 5 representations
for k_, v in (("naive_nested|abs", .888), ("naive_nested|mah", .969), ("naive_nested|fusion", .969),
              ("sa_nested|abs", .681), ("sa_nested|mah", .344), ("sa_nested|fusion", .819), ("sa_nested|innerw", .681),
              ("naive_leaky|abs", .825), ("naive_leaky|mah", 1.000), ("naive_leaky|fusion", 1.000),
              ("sa_leaky|abs", .781), ("sa_leaky|mah", .344), ("sa_leaky|fusion", .869),
              ("sa_published|abs", .813), ("sa_published|mah", .563), ("sa_published|fusion", .900),
              ("sa_generic|abs", .813), ("sa_generic|mah", .506), ("sa_generic|fusion", .763),
              ("naive_cole|abs", .950), ("naive_cole|mah", .650), ("naive_cole|fusion", .888),
              ("sa_cole|abs", .950), ("sa_cole|mah", .438), ("sa_cole|fusion", .694), ("sa_cole|innerw", .856)):
    add(f"T5 {k_}", v, ba(k_), 3)
add("SA nested mah healthy correct", 0, cnt("sa_nested|mah", "H"), 0)
add("SA nested mah CI low", .150, CI["sa_nested|mah"][0], 3); add("SA nested mah CI high", .472, CI["sa_nested|mah"][1], 3)
add("published CI low", .833, CI["sa_published|fusion"][0], 3); add("published CI high", 1.000, CI["sa_published|fusion"][1], 3)
add("published tumor correct", 16, cnt("sa_published|fusion", "T"), 0); add("published healthy correct", 8, cnt("sa_published|fusion", "H"), 0)
add("generic CI low", .586, CI["sa_generic|fusion"][0], 3); add("generic CI high", 1.000, CI["sa_generic|fusion"][1], 3)
l_, h_ = d_ci("sa_published|fusion", "sa_nested|fusion")
add("published - nested", .081, ba("sa_published|fusion") - ba("sa_nested|fusion"), 3); add("published - nested CI low", -.071, l_, 3); add("published - nested CI high", .571, h_, 3)
add("Cole abs SA CI low", .917, CI["sa_cole|abs"][0], 3); add("Cole abs SA CI high", 1.000, CI["sa_cole|abs"][1], 3)
add("Cole abs tumor correct", 16, cnt("sa_cole|abs", "T"), 0); add("Cole abs healthy correct", 9, cnt("sa_cole|abs", "H"), 0)
add("Cole inner CI low", .700, CI["sa_cole|innerw"][0], 3); add("Cole inner CI high", .962, CI["sa_cole|innerw"][1], 3)
add("Cole weights min", 0.6, min(RES["inner_weights"]["sa_cole"].values()), 1); add("Cole weights max", 1.0, max(RES["inner_weights"]["sa_cole"].values()), 1)
add("freq weights min", 0.5, min(RES["inner_weights"]["sa_nested"].values()), 1); add("freq weights max", 0.7, max(RES["inner_weights"]["sa_nested"].values()), 1)
add("corrected correct samples", 22, cnt("sa_nested|fusion", "T") + cnt("sa_nested|fusion", "H"), 0)
add("misclassified n min", 6, RES["misclassified"]["n_min"], 0); add("misclassified n max", 14, RES["misclassified"]["n_max"], 0)
add("correct mean n", 28.6, RES["misclassified"]["mean_n_correct"], 1)
add("P1 tumor prob min", .508, RES["misclassified"]["p1_tumor_min"], 3); add("P1 tumor prob max", .508, RES["misclassified"]["p1_tumor_max"], 3)
add("P1 wrong healthy prob", .507, RES["misclassified"]["p1_healthy_wrong"], 3)
# Table 6 sensitivity
for k_, tp, tn, b in (("sa_nested|fusion|cb", 15, 8, .869), ("sa_cole|abs|cb", 13, 9, .856), ("sa_cole|fusion|cb", 10, 9, .763),
                      ("noP2_naive_nested|fusion", 13, 10, .964), ("noP2_sa_nested|fusion", 13, 8, .864), ("noP2_sa_cole|abs", 13, 9, .914),
                      ("sa_coleR|abs", 16, 9, .950), ("sa_coleR|mah", 1, 3, .181), ("sa_coleR|fusion", 11, 5, .594)):
    add(f"T6 {k_} tumor", tp, cnt(k_, "T"), 0); add(f"T6 {k_} healthy", tn, cnt(k_, "H"), 0); add(f"T6 {k_} BA", b, ba(k_), 3)
add("noP2 referencing effect (pp)", 10.0, 100 * (ba("noP2_naive_nested|fusion") - ba("noP2_sa_nested|fusion")), 1)

out = []
for item, printed, computed, dec in C:
    ok_ = abs(float(computed) - float(printed)) <= 0.5 * 10 ** (-dec) + 1e-9
    out.append(dict(Item=item, Manuscript=printed, Computed=round(float(computed), dec + 2), Status="PASS" if ok_ else "FAIL"))
CHECK = pd.DataFrame(out)
CHECK.to_csv(f"{OUT}/consistency_check.csv", index=False)
pd.set_option("display.max_rows", 400)
print(CHECK[CHECK.Status == "FAIL"].to_string() if (CHECK.Status == "FAIL").any() else "no failures")
print(f"\n{(CHECK.Status == 'PASS').sum()} / {len(CHECK)} values agree with the manuscript.")
_m = fits.groupby(["Patient", "Label"]).R0.median().unstack()
_r = _m[1] / _m[0]
C2 = [("P2 tumor/healthy R0 ratio", 10.4, _r["P2"], 1),
      ("other patients with lower tumor R0", 4, int((_r.drop("P2") < 1).sum()), 0)]
for item, printed, computed, dec in C2:
    ok_ = abs(float(computed) - float(printed)) <= 0.5 * 10 ** (-dec) + 1e-9
    CHECK.loc[len(CHECK)] = dict(Item=item, Manuscript=printed, Computed=round(float(computed), dec + 2), Status="PASS" if ok_ else "FAIL")
CHECK.to_csv(f"{OUT}/consistency_check.csv", index=False)
print(f"after P2 checks: {(CHECK.Status == 'PASS').sum()} / {len(CHECK)} values agree with the manuscript.")

no failures

214 / 214 values agree with the manuscript.
after P2 checks: 216 / 216 values agree with the manuscript.


In [42]:
# values of the simulation study printed in the manuscript: (item, printed value, path in simulation_summary.json, decimals)
SIM_PRINTED = [
    ("sim naive opt min (null, m>=5)", -0.026, ("null", "naive_opt_min"), 3),
    ("sim naive opt max (null, m>=5)", 0.437, ("null", "naive_opt_max"), 3),
    ("sim naive opt ICC0.1 m5 (null)", -0.026, ("null", "naive_opt_icc01_m5"), 3),
    ("sim naive opt ICC0.97 m30 (null)", 0.437, ("null", "naive_opt_icc097_m30"), 3),
    ("sim naive opt max abstract", 0.44, ("null", "naive_opt_max"), 2),
    ("sim naive opt min (effect)", 0.027, ("effect", "naive_opt_min"), 3),
    ("sim naive opt max (effect)", 0.273, ("effect", "naive_opt_max"), 3),
    ("sim naive est ICC0.97 m15 (null)", 0.935, ("null", "naive_est_icc097_m15"), 3),
    ("sim naive est ICC0.97 m15 (effect)", 0.987, ("effect", "naive_est_icc097_m15"), 3),
    ("sim naive true ICC0.97 m15 (effect)", 0.742, ("effect", "naive_true_icc097_m15"), 3),
    ("sim sa true ICC0.97 m15 (effect)", 0.828, ("effect", "sa_true_icc097_m15"), 3),
    ("sim sa opt min (effect, m>=5)", -0.087, ("effect", "sa_opt_min"), 3),
    ("sim sa opt max (effect, m>=5)", -0.008, ("effect", "sa_opt_max"), 3),
    ("sim sa opt max (null, m>=5)", 0.170, ("null", "sa_opt_max"), 3),
    ("sim sa opt m1 min (effect)", 0.122, ("effect", "sa_opt_m1_min"), 3),
    ("sim sa opt m1 max (null)", 0.359, ("null", "sa_opt_m1_max"), 3),
    ("sim matched opt min (null)", -0.049, ("null", "matched_opt_min"), 3),
    ("sim matched opt max (effect)", 0.066, ("effect", "matched_opt_max"), 3),
    ("sim xgb naive opt null ICC0.97", 0.439, ("xgb", "naive_opt_null_icc097"), 3),
    ("sim xgb naive opt effect ICC0.97", 0.276, ("xgb", "naive_opt_effect_icc097"), 3),
    ("sim xgb matched opt min", -0.036, ("xgb", "matched_opt_min"), 3),
    ("sim xgb matched opt max", 0.040, ("xgb", "matched_opt_max"), 3),
    ("sim precision n6 h3 lo", 0.708, ("precision", "n6_h3", "lo"), 3),
    ("sim precision n6 h3 hi", 0.972, ("precision", "n6_h3", "hi"), 3),
    ("sim precision n6 h3 width", 0.265, ("precision", "n6_h3", "width"), 3),
    ("sim precision n6 h3 width (about 0.26)", 0.26, ("precision", "n6_h3", "width"), 2),
    ("sim precision n6 h2 width", 0.327, ("precision", "n6_h2", "width"), 3),
    ("sim precision n24 h3 width", 0.139, ("precision", "n24_h3", "width"), 3),
    ("sim precision n24 h3 width (about 0.14)", 0.14, ("precision", "n24_h3", "width"), 2),
    ("sim precision n48 h3 width", 0.085, ("precision", "n48_h3", "width"), 3),
    ("sim precision mean true BA", 0.831, ("precision_true_mean",), 3),
]

# ---------------------------------------------------------------- consistency check: revision analyses
REVR = json.load(open("outputs_ijs/revision/revision_results.json"))
PERM = json.load(open("outputs_ijs/revision/permutation_results.json"))
SIMS = json.load(open("outputs_ijs/revision/simulation_summary.json"))
RS = REVR["summary"]
C3 = []
def add3(item, printed, computed, dec):
    C3.append((item, printed, computed, dec))
def rdiff(group, a, b):
    d = REVR["diff"][group]
    if f"{a} - {b}" in d:
        return d[f"{a} - {b}"][1], d[f"{a} - {b}"][2]
    lo, hi = d[f"{b} - {a}"][1], d[f"{b} - {a}"][2]
    return -hi, -lo

# Table 7 permutation null
for k_, mean_, lo_, hi_, obs_, p_ in (("cole|naive|fusion", .696, .512, .869, .887, .019), ("cole|sa|fusion", .454, .312, .637, .694, .010),
                                      ("cole|naive|abs", .563, .344, .806, .950, .001), ("cole|sa|abs", .459, .312, .700, .950, .001),
                                      ("cole|naive|mah", .764, .619, .906, .650, .957), ("cole|sa|mah", .459, .250, .669, .438, .582),
                                      ("published|naive|fusion", .740, .562, .906, .875, .081), ("published|sa|fusion", .445, .312, .656, .900, .001),
                                      ("published|naive|mah", .823, .719, .938, 1.000, .001), ("published|sa|mah", .424, .250, .625, .563, .088)):
    v_ = PERM[k_]
    add3(f"T7 {k_} null mean", mean_, v_["null_mean"], 3); add3(f"T7 {k_} q025", lo_, v_["null_q025"], 3)
    add3(f"T7 {k_} q975", hi_, v_["null_q975"], 3); add3(f"T7 {k_} observed", obs_, v_["observed"], 3)
    add3(f"T7 {k_} p", p_, v_["p_value"], 3)
add3("permutations", 1000, PERM["n_perm"], 0)
add3("published naive mah null >= 0.75 (%)", 95.8, 100 * PERM["published|naive|mah"]["frac_null_ge_0p75"], 1)
add3("SA null mean min", .424, min(PERM[k]["null_mean"] for k in PERM if "|sa|" in k), 3)
add3("SA null mean max", .484, max(PERM[k]["null_mean"] for k in PERM if "|sa|" in k), 3)
add3("SA null q975 min", .625, min(PERM[k]["null_q975"] for k in PERM if "|sa|" in k), 3)
add3("SA null q975 max", .750, max(PERM[k]["null_q975"] for k in PERM if "|sa|" in k), 3)
# fully nested pipeline selection
ch = REVR["nested_pipeline_choice"]
add3("folds choosing absolute branch", 6, sum(v["branch"] == "abs" for v in ch.values()), 0)
add3("folds choosing Cole", 6, sum(v["representation"] == "cole" for v in ch.values()), 0)
add3("inner BA of chosen pipeline min", .866, min(v["inner_BA"] for v in ch.values()), 3)
add3("inner BA of chosen pipeline max", 1.000, max(v["inner_BA"] for v in ch.values()), 3)
_inn = pd.read_csv("outputs_ijs/revision/nested_pipeline_inner_scores.csv")
_bestf = _inn[_inn.Representation == "freq"].groupby("Fold").InnerBA.max()
_bestc = _inn[(_inn.Representation == "cole") & (_inn.Branch == "abs")].set_index("Fold").InnerBA
add3("folds where Cole abs beats best frequency candidate", 6, int((_bestc > _bestf).sum()), 0)
add3("inner folds with LOSO fallback", 6, sum(v == "LOSO" for v in REVR["inner_selection_method"].values()), 0)
add3("inner folds total", 30, len(REVR["inner_selection_method"]), 0)
s_ = RS["nested_pipeline|selected"]
add3("nested pipeline BA", .950, s_["BA"], 3); add3("nested pipeline tumor", 16, s_["TP"], 0); add3("nested pipeline healthy", 9, s_["TN"], 0)
add3("nested pipeline CI low", .917, REVR["ci"]["nested_pipeline|selected"][0], 3)
add3("nested pipeline CI high", 1.000, REVR["ci"]["nested_pipeline|selected"][1], 3)
add3("nested pipeline - pre-specified", .131, s_["BA"] - ba("sa_nested|fusion"), 3)
lo_, hi_ = rdiff("main_extra", "nested_pipeline|selected", "sa_nested|fusion")
add3("nested pipeline - pre-specified CI low", 0.000, lo_, 3); add3("nested pipeline - pre-specified CI high", .571, hi_, 3)
add3("nested pipeline equals Cole abs", 1, float(abs(s_["BA"] - ba("sa_cole|abs")) < 1e-9), 0)
# incomplete nesting: outer-fold frequency ranking reused in the inner folds
REVP = json.load(open("outputs_ijs/revision/revision_results_v2_outer_ranking.json"))
_innp = pd.read_csv("outputs_ijs/revision/nested_pipeline_inner_scores_v2_outer_ranking.csv")
_bestfp = _innp[_innp.Representation == "freq"].groupby("Fold").InnerBA.max()
add3("partial: max rise of best frequency candidate", .375, (_bestfp - _bestf).max(), 3)
add3("partial: P1 best frequency inner BA", 1.000, _bestfp["P1"], 3); add3("full: P1 best frequency inner BA", .625, _bestf["P1"], 3)
chp = REVP["nested_pipeline_choice"]
add3("partial: P1 chose frequency abs", 1, float(chp["P1"]["representation"] == "freq" and chp["P1"]["branch"] == "abs"), 0)
add3("partial: P5 chose frequency abs", 1, float(chp["P5"]["representation"] == "freq" and chp["P5"]["branch"] == "abs"), 0)
sp_ = REVP["summary"]["nested_pipeline|selected"]
add3("partial pipeline BA", .713, sp_["BA"], 3); add3("partial pipeline tumor", 10, sp_["TP"], 0); add3("partial pipeline healthy", 8, sp_["TN"], 0)
# size-matched referencing
for k_, v in (("matched_nested|fusion", .819), ("matched_cole|abs", .887), ("matched_published|fusion", .950),
              ("matched_nested|mah", .344), ("matched_cole|mah", .306)):
    add3(f"matched {k_}", v, RS[k_]["BA"], 3)
add3("matched nested correct samples", 22, RS["matched_nested|fusion"]["TP"] + RS["matched_nested|fusion"]["TN"], 0)
# P2 labels exchanged
add3("P2 swap evaluable healthy", 12, REVR["p2swap_evaluable"]["healthy"], 0); add3("P2 swap evaluable tumor", 15, REVR["p2swap_evaluable"]["tumor"], 0)
for k_, tp, tn, b in (("p2_naive_nested|fusion", 15, 10, .917), ("p2_sa_nested|fusion", 10, 6, .583),
                      ("p2_sa_cole|abs", 14, 9, .842), ("p2_sa_published|fusion", 13, 10, .850)):
    add3(f"T6 {k_} tumor", tp, RS[k_]["TP"], 0); add3(f"T6 {k_} healthy", tn, RS[k_]["TN"], 0); add3(f"T6 {k_} BA", b, RS[k_]["BA"], 3)
add3("P2 swap referencing effect", .333, RS["p2_naive_nested|fusion"]["BA"] - RS["p2_sa_nested|fusion"]["BA"], 3)
lo_, hi_ = rdiff("p2swap", "p2_naive_nested|fusion", "p2_sa_nested|fusion")
add3("P2 swap effect CI low", .113, lo_, 3); add3("P2 swap effect CI high", .760, hi_, 3)
# logistic regression
for k_, tp, tn, b in (("lr_naive_nested|fusion", 10, 10, .812), ("lr_naive_leaky|fusion", 13, 9, .856),
                      ("lr_sa_nested|fusion", 11, 8, .744), ("lr_sa_leaky|fusion", 11, 9, .794),
                      ("lr_naive_cole|abs", 15, 9, .919), ("lr_sa_cole|abs", 16, 7, .850), ("lr_sa_nested|mah", 16, 0, .500)):
    add3(f"T6 {k_} tumor", tp, RS[k_]["TP"], 0); add3(f"T6 {k_} healthy", tn, RS[k_]["TN"], 0); add3(f"T6 {k_} BA", b, RS[k_]["BA"], 3)
add3("LR referencing effect nested", .069, RS["lr_naive_nested|fusion"]["BA"] - RS["lr_sa_nested|fusion"]["BA"], 3)
add3("LR referencing effect leaky", .063, RS["lr_naive_leaky|fusion"]["BA"] - RS["lr_sa_leaky|fusion"]["BA"], 3)
add3("LR referencing effect Cole fusion", .138, RS["lr_naive_cole|fusion"]["BA"] - RS["lr_sa_cole|fusion"]["BA"], 3)
add3("LR Cole Mahalanobis SA", .438, RS["lr_sa_cole|mah"]["BA"], 3)
add3("LR Cole Mahalanobis SA healthy correct", 0, RS["lr_sa_cole|mah"]["TN"], 0)
# simulation
SIM_PRINTED = globals().get("SIM_PRINTED", [])
for item, printed, path, dec in SIM_PRINTED:
    v_ = SIMS
    for k in path:
        v_ = v_[k]
    add3(item, printed, v_, dec)

for item, printed, computed, dec in C3:
    ok_ = abs(float(computed) - float(printed)) <= 0.5 * 10 ** (-dec) + 1e-6
    CHECK.loc[len(CHECK)] = dict(Item=item, Manuscript=printed, Computed=round(float(computed), dec + 2), Status="PASS" if ok_ else "FAIL")
CHECK.to_csv(f"{OUT}/consistency_check.csv", index=False)
print(CHECK[CHECK.Status == "FAIL"].to_string() if (CHECK.Status == "FAIL").any() else "no failures")
print(f"after revision checks: {(CHECK.Status == 'PASS').sum()} / {len(CHECK)} values agree with the manuscript.")

no failures
after revision checks: 376 / 376 values agree with the manuscript.


In [43]:
SEL_PRINTED = [
    ("selection-aware: permutations", 100, ["n_perm"], 0),
    ("selection-aware: observed", .950, ["BA_selected", "observed"], 3),
    ("selection-aware: null mean", .471, ["BA_selected", "null_mean"], 3),
    ("selection-aware: null 2.5%", .315, ["BA_selected", "null_q025"], 3),
    ("selection-aware: null 97.5%", .691, ["BA_selected", "null_q975"], 3),
    ("selection-aware: p", .0099, ["BA_selected", "p_value"], 4),
    ("selection-aware: observed above all permutations", 1, ["_above_all"], 0),
    ("pre-specified (same runs): observed", .819, ["BA_prespecified", "observed"], 3),
    ("pre-specified (same runs): null mean", .455, ["BA_prespecified", "null_mean"], 3),
    ("pre-specified (same runs): p", .0099, ["BA_prespecified", "p_value"], 4),
]

SE_PRINTED = [
    ("sim ext: max naive optimism across departures", .422, lambda d: max(d[k]["naive_opt_max"] for k in ("correlated", "heteroscedastic", "heterogeneous")), 3),
    ("sim ext: min size-matched optimism across departures", -.027, lambda d: min(d[k]["matched_opt_min"] for k in ("correlated", "heteroscedastic", "heterogeneous")), 3),
    ("sim ext: max size-matched optimism across departures", .029, lambda d: max(d[k]["matched_opt_max"] for k in ("correlated", "heteroscedastic", "heterogeneous")), 3),
    ("sim ext: base reproduces naive null estimate", .935, lambda d: d["base"]["naive_est_null_icc097"], 3),
]

# ---------------------------------------------------------------- consistency check: analyses added in review
NS = json.load(open("outputs_ijs/revision/nested_sensitivity.json"))
EX = json.load(open("outputs_ijs/revision/reviewer_extras.json"))
SP = json.load(open("outputs_ijs/revision/selection_permutation_results.json"))
SP["_above_all"] = float(SP["BA_selected"]["null_max"] < SP["BA_selected"]["observed"])
SE = json.load(open("outputs_ijs/revision/sim_ext_summary.json"))
C4 = []
def add4(item, printed, computed, dec):
    C4.append((item, printed, computed, dec))

# design effect ranges
de_ = EX["design_effect"]
add4("N_eff min", 29.2, de_["N_eff_min"], 1); add4("N_eff max", 49.2, de_["N_eff_max"], 1)
add4("N_eff Kish min", 21.2, de_["N_eff_kish_min"], 1); add4("N_eff Kish max", 35.9, de_["N_eff_kish_max"], 1)
add4("Discussion: 21 to 49 observations (min)", 21, de_["N_eff_kish_min"], 0); add4("Discussion: 21 to 49 (max)", 49, de_["N_eff_max"], 0)
# frequency stability
fs_ = EX["frequency_stability"]
add4("share of outer sample-wise selections above 100 kHz (%)", 83, 100 * fs_["sample-wise outer"]["share_above_100k"], 0)
add4("Jaccard sample-wise outer", .14, fs_["sample-wise outer"]["mean_jaccard"], 2)
add4("Jaccard sample-wise inner", .13, fs_["sample-wise inner"]["mean_jaccard"], 2)
add4("Jaccard spectrum-wise outer", .25, fs_["spectrum-wise outer"]["mean_jaccard"], 2)
# nested sensitivity
def nsv(k, tp, n1, tn, n0, ba):
    v = NS[k]
    add4(f"{k} tumor", tp, v["TP"], 0); add4(f"{k} N1", n1, v["N1"], 0)
    add4(f"{k} healthy", tn, v["TN"], 0); add4(f"{k} N0", n0, v["N0"], 0); add4(f"{k} BA", ba, v["BA"], 3)
nsv("fallback_generic", 16, 16, 9, 10, .950); nsv("fallback_outer", 10, 16, 8, 10, .713)
nsv("matched", 13, 16, 7, 10, .756); nsv("p2_excluded", 13, 14, 9, 10, .914); nsv("p2_exchanged", 7, 15, 9, 12, .608)
add4("fallback generic: Cole abs in all folds", 6, NS["fallback_generic"]["n_cole"] * (NS["fallback_generic"]["n_abs"] == 6), 0)
fo = NS["fallback_outer"]["choice"]
add4("fallback outer: frequency abs in P1 and P5", 2, sum(fo[p]["representation"] == "freq" and fo[p]["branch"] == "abs" for p in ("P1", "P5")), 0)
add4("fallback outer equals all-fold reuse (choices)", 1, float(all((fo[p]["representation"], fo[p]["branch"]) == (REVP["nested_pipeline_choice"][p]["representation"], REVP["nested_pipeline_choice"][p]["branch"]) for p in fo)), 0)
add4("fallback outer equals all-fold reuse (BA)", .713, REVP["summary"]["nested_pipeline|selected"]["BA"], 3)
add4("matched CI low", .308, NS["matched"]["CI"][0], 3); add4("matched CI high", .923, NS["matched"]["CI"][1], 3)
add4("matched: folds with Cole abs", 4, sum(v["representation"] == "cole" and v["branch"] == "abs" for v in NS["matched"]["choice"].values()), 0)
_stm = pd.read_csv("outputs_ijs/revision/sampletable_nested_matched.csv")
_err = _stm[(_stm.p >= 0.5).astype(int) != _stm.y]
add4("matched: both P2 tumor samples misclassified", 2, int(((_err.Patient == "P2") & (_err.y == 1)).sum()), 0)
add4("P2 excluded: Cole abs in all five folds", 5, sum(v["representation"] == "cole" and v["branch"] == "abs" for v in NS["p2_excluded"]["choice"].values()), 0)
add4("P2 exchanged: distinct pipelines", 4, len({(v["representation"], v["branch"]) for v in NS["p2_exchanged"]["choice"].values()}), 0)
# robustness table entries not checked elsewhere
add4("T7 published without P2", .900, EX["published_without_P2"]["BA"], 3)
add4("corrected range min", .583, min(RS["p2_sa_nested|fusion"]["BA"], NS["p2_exchanged"]["BA"]), 3)
add4("corrected range max", .950, max(NS["fallback_generic"]["BA"], RS["nested_pipeline|selected"]["BA"]), 3)
add4("conclusion: down to 0.61", .61, NS["p2_exchanged"]["BA"], 2)
# influence and stability
inf_ = EX["influence"]
add4("influence fully nested min", .938, inf_["Fully nested pipeline"]["min"], 3); add4("influence fully nested max", 1.000, inf_["Fully nested pipeline"]["max"], 3)
add4("influence pre-specified min", .700, inf_["Sample-wise, nested frequencies"]["min"], 3)
add4("influence pre-specified min is P1", 1, float(inf_["Sample-wise, nested frequencies"]["min_patient"] == "P1"), 0)
add4("influence pre-specified max", .902, inf_["Sample-wise, nested frequencies"]["max"], 3)
add4("influence pre-specified max is P5", 1, float(inf_["Sample-wise, nested frequencies"]["max_patient"] == "P5"), 0)
ss_ = EX["seed_stability"]
add4("seed Cole abs min", .919, ss_["Cole absolute (fully nested choice)"]["BA_min"], 3)
add4("seed Cole abs max", .950, ss_["Cole absolute (fully nested choice)"]["BA_max"], 3)
add4("seed Cole abs samples changing class", 1, ss_["Cole absolute (fully nested choice)"]["n_samples_changing_class"], 0)
add4("seed pre-specified samples changing class", 0, ss_["Nested frequencies, equal fusion"]["n_samples_changing_class"], 0)
add4("seed pre-specified BA range", 0.0, ss_["Nested frequencies, equal fusion"]["BA_max"] - ss_["Nested frequencies, equal fusion"]["BA_min"], 3)
iw_ = EX["cole_in_window_only"]
add4("in-window Cole abs BA", .950, iw_["BA"], 3); add4("in-window tumor", 16, iw_["TP"], 0); add4("in-window healthy", 9, iw_["TN"], 0)
add4("out-of-window spectra", 59, EX["cole_out_of_window"]["n"], 0)
add4("misclassified sample out-of-window spectra", 14, iw_["misclassified_main_out_of_window"], 0)
add4("misclassified sample is healthy P1", 1, float(len(iw_["misclassified_main"]) == 1 and iw_["misclassified_main"][0].startswith("P1:hbw")), 0)
# paired bootstrap: sample-wise vs size-matched referencing
PM = json.load(open("outputs_ijs/revision/paired_matched.json"))
add4("paired: nested sa - matched difference", .194, PM["nested"]["diff"], 3)
add4("paired: nested difference CI low", .023, PM["nested"]["ci_low"], 3)
add4("paired: nested difference CI high", .692, PM["nested"]["ci_high"], 3)
add4("paired: Cole abs sa - matched difference", .063, PM["coleabs"]["diff"], 3)
add4("paired: Cole abs difference CI low", 0.000, PM["coleabs"]["ci_low"], 3)
add4("paired: Cole abs difference CI high", .250, PM["coleabs"]["ci_high"], 3)
add4("paired: pre-specified unchanged", 0.0, PM["prespecified"]["diff"], 3)
add4("paired: valid bootstrap draws", 9861, PM["n_valid"], 0)
# selection-aware permutation
SEL_PRINTED = globals().get("SEL_PRINTED", [])
for item, printed, keys, dec in SEL_PRINTED:
    v_ = SP
    for k in keys:
        v_ = v_[k]
    add4(item, printed, v_, dec)
# simulation scenarios
SE_PRINTED = globals().get("SE_PRINTED", [])
for item, printed, fn, dec in SE_PRINTED:
    add4(item, printed, fn(SE), dec)

for item, printed, computed, dec in C4:
    ok_ = abs(float(computed) - float(printed)) <= 0.5 * 10 ** (-dec) + 1e-6
    CHECK.loc[len(CHECK)] = dict(Item=item, Manuscript=printed, Computed=round(float(computed), dec + 2), Status="PASS" if ok_ else "FAIL")
CHECK.to_csv(f"{OUT}/consistency_check.csv", index=False)
print(CHECK[CHECK.Status == "FAIL"].to_string() if (CHECK.Status == "FAIL").any() else "no failures")
print(f"after review checks: {(CHECK.Status == 'PASS').sum()} / {len(CHECK)} values agree with the manuscript.")

no failures
after review checks: 464 / 464 values agree with the manuscript.


In [ ]:
# values added in the second revision, checked against revision2_extras.json (printed to 3 decimals)
X2 = json.load(open(f"{REVDIR}/revision2_extras.json"))
R2_PRINTED = [("interaction (points)", -1.9, 100 * X2["factorial"]["interaction"], 0.05),
              ("interaction CI low", -21.7, 100 * X2["factorial"]["interaction_CI"][0], 0.05),
              ("interaction CI high", 10.7, 100 * X2["factorial"]["interaction_CI"][1], 0.05),
              ("sum of main effects", 20.0, 100 * X2["factorial"]["sum_of_main_effects"], 0.05),
              ("patient BA pre-specified", 0.722, X2["patient_level"]["pre-specified (sample-wise, nested)"]["patient_BA"], 5e-4),
              ("patient BA fully nested", 0.972, X2["patient_level"]["fully nested pipeline"]["patient_BA"], 5e-4),
              ("patient BA fully nested, size-matched", 0.639, X2["patient_level"]["fully nested, size-matched"]["patient_BA"], 5e-4),
              ("median |rho| healthy", 0.79, X2["drift_vs_class"]["median_abs_rho"]["healthy"], 5e-3),
              ("median |rho| tumor", 0.97, X2["drift_vs_class"]["median_abs_rho"]["tumor"], 5e-3),
              ("Mann-Whitney p |rho|", 0.23, X2["drift_vs_class"]["mannwhitney_p_abs_rho"], 5e-3),
              ("Mann-Whitney p |drift|", 0.52, X2["drift_vs_class"]["mannwhitney_p_abs_drift"], 5e-3),
              ("max change in contrast", 0.19, X2["drift_vs_class"]["max_abs_change_in_contrast"], 5e-3),
              ("Clopper-Pearson upper bound", 0.030, X2["selection_aware_p"]["clopper_pearson_upper_95"], 5e-4),
              ("size-matched published abs", 0.844, X2["size_matched_BA"]["published_abs"], 5e-4),
              ("size-matched published mah", 0.719, X2["size_matched_BA"]["published_mah"], 5e-4),
              ("size-matched Cole fusion", 0.706, X2["size_matched_BA"]["cole_fusion"], 5e-4)]
bad = [(n, p_, v) for n, p_, v, tol in R2_PRINTED if abs(p_ - v) > tol + 1e-9]
print(f"second-revision values: {len(R2_PRINTED) - len(bad)}/{len(R2_PRINTED)} agree", bad)


In [ ]:
for item, printed, computed, tol in R2_PRINTED:      # second-revision values (Section 16.6)
    CHECK.loc[len(CHECK)] = dict(Item=item, Manuscript=printed, Computed=round(float(computed), 5),
                                 Status="PASS" if abs(float(computed) - float(printed)) <= tol + 1e-9 else "FAIL")
CHECK.to_csv(f"{OUT}/consistency_check.csv", index=False)
n_all = len(CHECK) + len(TCHK)
n_ok = (CHECK.Status == "PASS").sum() + (TCHK.Status == "PASS").sum()
print(f"OVERALL: {n_ok} / {n_all} checked values agree (reported values {len(CHECK)}, table cells {len(TCHK)})")
if n_ok < n_all:
    display(CHECK[CHECK.Status == "FAIL"]); display(TCHK[TCHK.Status == "FAIL"])


## 20. Export

In [44]:
import zipfile
with zipfile.ZipFile("ijs_results_bundle.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, names in os.walk(OUT):
        for n in names:
            zf.write(os.path.join(root, n))
    for n in ("eis_pipeline.py", "eis_groups.py", "cole_analysis.py", "ijs_figures.py", "rev_common.py", "ijs_permutation.py",
              "ijs_revision.py", "ijs_simulation.py", "sim_summary.py", "ijs_revision_figs.py", "nested_core.py",
              "ijs_nested_sensitivity.py", "ijs_selection_permutation.py", "ijs_reviewer_extras.py", "ijs_simulation_ext.py",
              "ijs_paired_matched.py", "ijs_revision2_extras.py", "article_tables.py",
              "make_supp_tables.py", "make_fig12.py", "make_fig3.py"):
        zf.write(n)
print("written ijs_results_bundle.zip")
try:
    from google.colab import files
    files.download("ijs_results_bundle.zip")
except Exception:
    pass

written ijs_results_bundle.zip
